<a href="https://colab.research.google.com/github/herehere14/market_maker_trading_system/blob/main/medium_frequency_trading_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import os
from datetime import datetime

API_KEY = "q9lnyOSkS8OS8KlfCA8pTyHiiR2H5iDp"
TICKER = "GLD"
START_DATE = "2025-01-01"
END_DATE   = "2026-01-31"
OUTFILE    = f"./data/minute_data/{TICKER}.US_minute.csv"  # Output CSV path

def fetch_polygon_minute_data(ticker, from_date, to_date, api_key):
    """
    Fetch 1-minute aggregates from Polygon for the given ticker and date range.
    Returns a DataFrame with columns [timestamp, open, high, low, close, volume].
    """
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/minute/{from_date}/{to_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": "100000",
        "apiKey": api_key
    }

    all_rows = []
    while True:
        print(f"Requesting from {url} with limit={params['limit']}...")
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()

        # Extract results
        results = data.get("results", [])
        for bar in results:
            ts = datetime.utcfromtimestamp(bar["t"] / 1000.0)
            row = {
                "timestamp": ts,
                "open": bar["o"],
                "high": bar["h"],
                "low":  bar["l"],
                "close": bar["c"],
                "volume": bar["v"]
            }
            all_rows.append(row)

        # Stop if there's no pagination
        if "next_url" in data and data["next_url"]:
            url = data["next_url"]
        else:
            break

    df = pd.DataFrame(all_rows)
    df.sort_values("timestamp", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def main():
    df_new = fetch_polygon_minute_data(TICKER, START_DATE, END_DATE, API_KEY)
    if df_new.empty:
        print("No bars retrieved (possible future date or no coverage). Nothing to save.")
        return

    # Ensure output directory exists
    os.makedirs(os.path.dirname(OUTFILE), exist_ok=True)

    if os.path.exists(OUTFILE):
        print(f"{OUTFILE} found. Loading existing data to append.")
        df_existing = pd.read_csv(OUTFILE, parse_dates=["timestamp"])
        last_time = df_existing["timestamp"].max()
        df_append = df_new[df_new["timestamp"] > last_time]
        if df_append.empty:
            print("No new bars to append.")
            return
        df_combined = pd.concat([df_existing, df_append], ignore_index=True)
        df_combined.sort_values("timestamp", inplace=True)
        df_combined.to_csv(OUTFILE, index=False)
        print(f"Appended {len(df_append)} new bars. Total rows: {len(df_combined)}")
    else:
        df_new.to_csv(OUTFILE, index=False)
        print(f"Created {OUTFILE} with {len(df_new)} rows.")

if __name__ == "__main__":
    main()


Requesting from https://api.polygon.io/v2/aggs/ticker/GLD/range/1/minute/2025-01-01/2026-01-31 with limit=100000...


/tmp/ipython-input-3405865624.py:35: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  ts = datetime.utcfromtimestamp(bar["t"] / 1000.0)


Requesting from https://api.polygon.io/v2/aggs/ticker/GLD/range/1/minute/1746124620000/2026-01-31?cursor=bGltaXQ9NTAwMDAmc29ydD1hc2M with limit=100000...
Requesting from https://api.polygon.io/v2/aggs/ticker/GLD/range/1/minute/1756228740000/2026-01-31?cursor=bGltaXQ9NTAwMDAmc29ydD1hc2M with limit=100000...
Requesting from https://api.polygon.io/v2/aggs/ticker/GLD/range/1/minute/1764940320000/2026-01-31?cursor=bGltaXQ9NTAwMDAmc29ydD1hc2M with limit=100000...
Created ./data/minute_data/GLD.US_minute.csv with 179291 rows.


# New Section

In [ ]:
!pip install ta filterpy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=854d2809d864b597b761fc59782e9b2e2071df7504021058284fe1827b35a5a9
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=16d1a6f96a29845afd4fcbf43845f629a8124dacbb88da13f39d84140108131f
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built ta filterpy


In [ ]:
#!/usr/bin/env python
# minute_feature_engineering_v4.py

"""
Simplified & Enhanced Feature Engineering for Minute-Level Price Data (v4)
---------------------------------------------------------------------------
Pipeline Order:
1. Generate core features + Kalman Filter Trend Extraction
2. Apply cleaning pipeline (Imputation → Outlier removal → Scaling)
3. Remove highly correlated features (correlation > 0.8) [FIRST FILTER]
4. Select top 25 features by importance
5. Apply K-means + RBF to create many new feature columns
6. Remove highly correlated features (correlation > 0.9) [SECOND FILTER]
7. Select top 30 final features
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional
import warnings
import traceback

from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest, f_regression
#A non-linear dependency score between a feature and a continuous target
#uses f_regression and mutual_info_regression to Ranks features, Keeps the top K
#

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
#distance More ML
from scipy.spatial.distance import cdist
#Faster

# Kalman Filter, estimate the true hidden state without noise
from filterpy.kalman import KalmanFilter


# Technical analysis, fundamental, tehcnical and more
import ta

warnings.filterwarnings('ignore')

# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
TICKER = "GLD"
MINUTE_DATA_DIR = "./data/minute_data"
OUTPUT_FILE = f"./data/{TICKER}.US_minute_engineered_v4.csv"

# Feature selection parameters
N_FEATURES_BEFORE_RBF = 25  # Target number of features before RBF
N_FEATURES_FINAL = 55  # Final number of features after all processing
CORRELATION_THRESHOLD_FIRST = 0.8  # First filter: before RBF
CORRELATION_THRESHOLD_SECOND = 0.9  # Second filter: after RBF

# K-means + RBF parameters
N_CLUSTERS_LIST = [5, 10, 15, 20]
RBF_GAMMA_LIST = [0.01, 0.1, 0.5, 1.0]

# Kalman Filter parameters
KALMAN_PROCESS_NOISE = 0.01
KALMAN_MEASUREMENT_NOISE = 0.1

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# --------------------------------------------------------------------
# Kalman Filter Trend Extraction
# --------------------------------------------------------------------
def apply_kalman_filter(series: pd.Series, process_noise: float = KALMAN_PROCESS_NOISE,
                        measurement_noise: float = KALMAN_MEASUREMENT_NOISE) -> pd.Series:
    """
    Apply Kalman Filter to extract trend from a time series.
    Uses a position + velocity state model for smooth trend extraction.
    """
    kf = KalmanFilter(dim_x=2, dim_z=1)

    # State transition matrix (position + velocity model)
    kf.F = np.array([[1., 1.],
                     [0., 1.]])

    # Measurement matrix (we only observe position)
    kf.H = np.array([[1., 0.]])

    # Covariance matrices
    kf.P *= 1000.  # Initial uncertainty
    kf.R = np.array([[measurement_noise]])  # Measurement noise
    kf.Q = np.array([[process_noise, 0.],
                     [0., process_noise]])  # Process noise

    # Initial state
    kf.x = np.array([[series.iloc[0]], [0.]])

    # Run filter
    filtered_values = []
    velocities = []

    for z in series.values:
        kf.predict()
        kf.update(np.array([[z]]))
        filtered_values.append(kf.x[0, 0])
        velocities.append(kf.x[1, 0])

    return pd.Series(filtered_values, index=series.index), pd.Series(velocities, index=series.index)


def extract_kalman_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract comprehensive Kalman-filtered trend features from price data.
    """
    print("\n  Extracting Kalman Filter Trend Features...")

    df = df.copy()

    # Apply Kalman filter to close price
    df['kalman_trend'], df['kalman_velocity'] = apply_kalman_filter(df['close'])

    # Trend deviation (price - trend)
    df['kalman_deviation'] = df['close'] - df['kalman_trend']
    df['kalman_deviation_pct'] = df['kalman_deviation'] / (df['kalman_trend'] + 1e-8)

    # Normalized deviation (z-score like)
    rolling_std = df['kalman_deviation'].rolling(20).std()
    df['kalman_deviation_zscore'] = df['kalman_deviation'] / (rolling_std + 1e-8)

    # Velocity features (rate of change of trend)
    df['kalman_velocity_ma5'] = df['kalman_velocity'].rolling(5).mean()
    df['kalman_velocity_ma10'] = df['kalman_velocity'].rolling(10).mean()

    # Acceleration (rate of change of velocity)
    df['kalman_acceleration'] = df['kalman_velocity'].diff()
    df['kalman_acceleration_ma5'] = df['kalman_acceleration'].rolling(5).mean()

    # Trend strength indicators
    df['kalman_trend_strength'] = df['kalman_velocity'].rolling(20).mean() / \
                                   (df['kalman_velocity'].rolling(20).std() + 1e-8)

    # Mean reversion signal (when deviation is extreme)
    df['kalman_mean_reversion'] = -df['kalman_deviation_zscore']

    # Trend momentum (velocity relative to recent history)
    df['kalman_momentum'] = df['kalman_velocity'] / \
                            (df['kalman_velocity'].rolling(50).std() + 1e-8)

    # Regime detection (trend vs mean-reverting)
    df['kalman_regime'] = np.where(
        abs(df['kalman_deviation_zscore']) > 2,
        np.sign(df['kalman_deviation_zscore']) * -1,  # Mean revert when extreme
        np.sign(df['kalman_velocity'])  # Follow trend otherwise
    )

    # Apply Kalman to volume for smoothed volume trend
    df['kalman_volume_trend'], _ = apply_kalman_filter(
        df['volume'],
        process_noise=0.1,
        measurement_noise=1.0
    )
    df['volume_vs_kalman'] = df['volume'] / (df['kalman_volume_trend'] + 1e-8)

    # Apply Kalman to volatility
    realized_vol = df['returns'].rolling(20).std() * np.sqrt(390)
    if not realized_vol.isna().all():
        realized_vol_filled = realized_vol.fillna(method='bfill').fillna(method='ffill')
        df['kalman_volatility'], _ = apply_kalman_filter(
            realized_vol_filled,
            process_noise=0.001,
            measurement_noise=0.1
        )
        df['vol_vs_kalman'] = realized_vol / (df['kalman_volatility'] + 1e-8)

    # Trend crossover signals
    df['price_above_kalman'] = (df['close'] > df['kalman_trend']).astype(int)
    df['kalman_crossover'] = df['price_above_kalman'].diff().fillna(0)

    # Distance from trend in ATR units
    atr = ta.volatility.AverageTrueRange(
        high=df['high'], low=df['low'], close=df['close'], window=14
    ).average_true_range()
    df['kalman_deviation_atr'] = df['kalman_deviation'] / (atr + 1e-8)

    print(f"    Added {sum(1 for c in df.columns if 'kalman' in c.lower())} Kalman features")

    return df


# --------------------------------------------------------------------
# 1. Data Loading & Basic Preprocessing
# --------------------------------------------------------------------
def load_and_prepare_data(file_path: str) -> Optional[pd.DataFrame]:
    """Load minute data and perform basic preprocessing."""
    print(f"Loading data from {file_path}...")

    try:
        df = pd.read_csv(file_path)

        # Find datetime column
        datetime_col = None
        for col in df.columns:
            if col.lower() in ['datetime', 'date', 'time', 'timestamp', 'date_time', 'dt']:
                datetime_col = col
                break

        if datetime_col is None:
            for col in df.columns:
                try:
                    pd.to_datetime(df[col].iloc[:5])
                    datetime_col = col
                    break
                except:
                    continue

        if datetime_col is None:
            print("Error: No datetime column found")
            return None

        df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
        df.set_index(datetime_col, inplace=True)
        df.sort_index(inplace=True)

        # Standardize column names
        col_map = {}
        for col in df.columns:
            col_lower = col.lower()
            if 'open' in col_lower:
                col_map[col] = 'open'
            elif 'high' in col_lower:
                col_map[col] = 'high'
            elif 'low' in col_lower:
                col_map[col] = 'low'
            elif 'close' in col_lower:
                col_map[col] = 'close'
            elif 'volume' in col_lower or 'vol' in col_lower:
                col_map[col] = 'volume'

        df = df.rename(columns=col_map)

        # Verify required columns
        required = ['open', 'high', 'low', 'close', 'volume']
        missing = [col for col in required if col not in df.columns]
        if missing:
            print(f"Error: Missing columns {missing}")
            return None

        # Remove duplicates
        df = df[~df.index.duplicated(keep='first')]

        # Add gap flag for missing minutes
        if df.index.tz is None:
            df.index = df.index.tz_localize("UTC")
        df.index = df.index.tz_convert("US/Eastern")

        full_index = pd.date_range(df.index[0], df.index[-1], freq="1min", tz="US/Eastern")
        gap_mask = ~full_index.isin(df.index)
        df['gap_flag'] = 0
        if gap_mask.any():
            prev_bars = full_index[gap_mask] - pd.Timedelta(minutes=1)
            df.loc[df.index.intersection(prev_bars), 'gap_flag'] = 1

        print(f"Loaded {len(df)} rows with {df['gap_flag'].sum()} detected gaps")
        return df

    except Exception as e:
        print(f"Error loading data: {e}")
        traceback.print_exc()
        return None


# --------------------------------------------------------------------
# 2. Core Feature Generation (with Kalman)
# --------------------------------------------------------------------
def generate_core_features(df: pd.DataFrame) -> pd.DataFrame:
    """Generate essential technical features including Kalman filter."""
    print("\n" + "="*60)
    print("STEP 1: GENERATING CORE FEATURES + KALMAN FILTER")
    print("="*60)

    # Basic price features
    df['returns'] = df['close'].pct_change()
    df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
    df['range'] = df['high'] - df['low']
    df['range_pct'] = df['range'] / df['close'] * 100
    df['body'] = df['close'] - df['open']
    df['body_pct'] = df['body'] / df['close'] * 100
    df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']

    # Time features
    df['hour'] = df.index.hour
    df['minute_of_day'] = df.index.hour * 60 + df.index.minute
    df['day_of_week'] = df.index.dayofweek

    # Cyclical time encoding
    df['time_sin'] = np.sin(2 * np.pi * df['minute_of_day'] / 390)
    df['time_cos'] = np.cos(2 * np.pi * df['minute_of_day'] / 390)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 5)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 5)

    # Session flags
    market_open = 9 * 60 + 30
    df['is_first_hour'] = ((df['minute_of_day'] >= market_open) &
                           (df['minute_of_day'] < market_open + 60)).astype(int)
    df['is_last_hour'] = ((df['minute_of_day'] >= 15 * 60) &
                          (df['minute_of_day'] < 16 * 60)).astype(int)

    # Moving averages
    for window in [5, 10, 20, 50]:
        df[f'sma_{window}'] = df['close'].rolling(window).mean()
        df[f'ema_{window}'] = df['close'].ewm(span=window).mean()
        df[f'close_sma_{window}_ratio'] = df['close'] / df[f'sma_{window}']

    # Volatility
    for window in [5, 10, 20]:
        df[f'volatility_{window}'] = df['returns'].rolling(window).std() * np.sqrt(390)

    # RSI
    for window in [7, 14]:
        try:
            rsi = ta.momentum.RSIIndicator(close=df['close'], window=window)
            df[f'rsi_{window}'] = rsi.rsi()
        except:
            pass

    # MACD
    try:
        macd = ta.trend.MACD(close=df['close'], window_slow=26, window_fast=12, window_sign=9)
        df['macd'] = macd.macd()
        df['macd_signal'] = macd.macd_signal()
        df['macd_hist'] = macd.macd_diff()
    except:
        pass

    # Bollinger Bands
    try:
        bb = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2)
        df['bb_upper'] = bb.bollinger_hband()
        df['bb_lower'] = bb.bollinger_lband()
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['close']
        df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-8)
    except:
        pass

    # ATR
    try:
        atr = ta.volatility.AverageTrueRange(high=df['high'], low=df['low'],
                                              close=df['close'], window=14)
        df['atr_14'] = atr.average_true_range()
        df['atr_pct'] = df['atr_14'] / df['close'] * 100
    except:
        pass

    # Stochastic
    try:
        stoch = ta.momentum.StochasticOscillator(high=df['high'], low=df['low'],
                                                  close=df['close'], window=14, smooth_window=3)
        df['stoch_k'] = stoch.stoch()
        df['stoch_d'] = stoch.stoch_signal()
    except:
        pass

    # Volume features
    df['volume_sma_10'] = df['volume'].rolling(10).mean()
    df['volume_ratio'] = df['volume'] / (df['volume_sma_10'] + 1e-8)
    df['volume_delta'] = df['volume'] * np.sign(df['close'] - df['open'])
    df['volume_delta_ma5'] = df['volume_delta'].rolling(5).mean()

    # VWAP
    df['day'] = df.index.date
    df['cum_vol'] = df.groupby('day')['volume'].cumsum()
    df['cum_pv'] = df.groupby('day').apply(
        lambda x: (x['close'] * x['volume']).cumsum()
    ).reset_index(level=0, drop=True)
    df['vwap'] = df['cum_pv'] / (df['cum_vol'] + 1e-8)
    df['close_vwap_ratio'] = df['close'] / df['vwap']

    # Momentum features
    for window in [3, 5, 10]:
        df[f'momentum_{window}'] = df['close'].pct_change(window)
        df[f'acceleration_{window}'] = df[f'momentum_{window}'].diff()

    # Order flow approximation
    df['buy_pressure'] = df['volume'] * (df['close'] > df['open']).astype(int)
    df['sell_pressure'] = df['volume'] * (df['close'] < df['open']).astype(int)
    df['order_imbalance'] = (df['buy_pressure'].rolling(10).sum() -
                             df['sell_pressure'].rolling(10).sum()) / \
                            (df['buy_pressure'].rolling(10).sum() +
                             df['sell_pressure'].rolling(10).sum() + 1e-8)

    # Price position features
    for window in [20, 50]:
        df[f'high_{window}'] = df['high'].rolling(window).max()
        df[f'low_{window}'] = df['low'].rolling(window).min()
        df[f'price_position_{window}'] = (df['close'] - df[f'low_{window}']) / \
                                          (df[f'high_{window}'] - df[f'low_{window}'] + 1e-8)

    # Trend strength
    df['trend_strength'] = (df['close'] - df['close'].rolling(20).mean()) / \
                           (df['close'].rolling(20).std() + 1e-8)

    # Mean reversion signal
    df['mean_reversion_signal'] = np.where(
        (df['trend_strength'] > 2) | (df['trend_strength'] < -2), 1, 0
    )

    # Pattern detection
    df['breakout_up'] = ((df['close'] > df['high'].rolling(20).max().shift(1)) &
                         (df['volume'] > df['volume_sma_10'] * 1.5)).astype(int)
    df['breakout_down'] = ((df['close'] < df['low'].rolling(20).min().shift(1)) &
                           (df['volume'] > df['volume_sma_10'] * 1.5)).astype(int)

    # Drawdown
    rolling_max = df['close'].rolling(50).max()
    df['drawdown'] = (df['close'] - rolling_max) / rolling_max

    # Additional advanced features
    df['efficiency_ratio'] = df['close'].diff(10).abs() / (df['close'].diff().abs().rolling(10).sum() + 1e-8)
    df['liquidity_score'] = (df['volume'] * df['close']) / ((df['volume'] * df['close']).rolling(50).mean() + 1e-8)

    # Clean up temp columns
    df.drop(columns=['day', 'cum_vol', 'cum_pv'], errors='ignore', inplace=True)

    # === KALMAN FILTER FEATURES ===
    df = extract_kalman_features(df)

    print(f"Generated {len(df.columns)} initial features (including Kalman)")
    return df


# --------------------------------------------------------------------
# 3. Cleaning Pipeline
# --------------------------------------------------------------------
def apply_cleaning_pipeline(df: pd.DataFrame) -> Tuple[pd.DataFrame, MinMaxScaler, SimpleImputer]:
    """
    Apply cleaning pipeline:
    1. Imputation (median)
    2. Outlier removal (IQR clipping)
    3. Scaling (MinMax to [-1, 1])
    """
    print("\n" + "="*60)
    print("STEP 2: APPLYING CLEANING PIPELINE")
    print("="*60)

    exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'kalman_trend', 'kalman_volume_trend']
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col not in exclude_cols]

    df_clean = df.copy()

    # --- Step 2a: Imputation ---
    print("\n  [2a] Imputing missing values with median...")
    missing_before = df_clean[numeric_cols].isna().sum().sum()

    imputer = SimpleImputer(strategy='median')
    df_clean[numeric_cols] = imputer.fit_transform(df_clean[numeric_cols])

    missing_after = df_clean[numeric_cols].isna().sum().sum()
    print(f"       Missing values: {missing_before} → {missing_after}")

    # --- Step 2b: Outlier Removal (IQR Clipping) ---
    print("\n  [2b] Removing outliers with IQR clipping (3x)...")
    outlier_counts = {}

    for col in numeric_cols:
        data = df_clean[col]
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR

        n_outliers = ((data < lower) | (data > upper)).sum()
        if n_outliers > 0:
            outlier_counts[col] = n_outliers

        df_clean[col] = data.clip(lower=lower, upper=upper)

    total_outliers = sum(outlier_counts.values())
    print(f"       Clipped {total_outliers} outlier values across {len(outlier_counts)} columns")

    # --- Step 2c: Scaling to [-1, 1] ---
    print("\n  [2c] Scaling features to [-1, 1] with MinMaxScaler...")
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

    print(f"       Scaled {len(numeric_cols)} columns")

    return df_clean, scaler, imputer


# --------------------------------------------------------------------
# 4. Remove Highly Correlated Features (Generic Function)
# --------------------------------------------------------------------
def remove_correlated_features(df: pd.DataFrame, threshold: float,
                                exclude_cols: List[str] = None,
                                step_name: str = "FILTER") -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    """
    Remove features with correlation > threshold.
    Keeps the feature with higher importance (correlation with returns).
    """
    print(f"\n  [{step_name}] Removing features with correlation > {threshold}...")

    if exclude_cols is None:
        exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']

    feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col not in exclude_cols]

    if len(feature_cols) == 0:
        return df, [], pd.DataFrame()

    # Calculate correlation matrix
    corr_matrix = df[feature_cols].corr().abs()

    # Calculate feature importance (correlation with future returns)
    future_returns = df['returns'].shift(-1) if 'returns' in df.columns else None
    feature_importance = {}

    for col in feature_cols:
        try:
            if future_returns is not None:
                corr = abs(df[col].corr(future_returns))
                feature_importance[col] = corr if not np.isnan(corr) else 0
            else:
                feature_importance[col] = df[col].std()  # Use variance as proxy
        except:
            feature_importance[col] = 0

    # Find pairs of highly correlated features
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    removed_features = []
    removal_reasons = []
    features_to_drop = set()

    for col in upper_tri.columns:
        for idx in upper_tri.index:
            if pd.notna(upper_tri.loc[idx, col]) and upper_tri.loc[idx, col] > threshold:
                # Keep the one with higher importance
                if feature_importance.get(col, 0) > feature_importance.get(idx, 0):
                    if idx not in features_to_drop:
                        features_to_drop.add(idx)
                        removed_features.append(idx)
                        removal_reasons.append(f"corr={upper_tri.loc[idx, col]:.3f} with {col}")
                else:
                    if col not in features_to_drop:
                        features_to_drop.add(col)
                        removed_features.append(col)
                        removal_reasons.append(f"corr={upper_tri.loc[idx, col]:.3f} with {idx}")

    removal_df = pd.DataFrame({
        'removed_feature': removed_features,
        'reason': removal_reasons
    }).drop_duplicates(subset=['removed_feature'])

    features_to_keep = [col for col in feature_cols if col not in features_to_drop]

    print(f"       Original features: {len(feature_cols)}")
    print(f"       Removed features: {len(features_to_drop)}")
    print(f"       Remaining features: {len(features_to_keep)}")

    # Keep essential cols + selected features
    final_cols = list(exclude_cols) + features_to_keep
    final_cols = [col for col in final_cols if col in df.columns]

    return df[final_cols], features_to_keep, removal_df


# --------------------------------------------------------------------
# 5. Feature Importance & Selection
# --------------------------------------------------------------------
def select_top_features(df: pd.DataFrame, n_features: int,
                        exclude_cols: List[str] = None,
                        step_name: str = "SELECT") -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Calculate feature importance and select top N features.
    """
    print(f"\n  [{step_name}] Selecting top {n_features} features by importance...")

    if exclude_cols is None:
        exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']

    # Create target (future returns)
    target = df['returns'].shift(-1) if 'returns' in df.columns else None

    feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col not in exclude_cols]

    if len(feature_cols) <= n_features:
        print(f"       Only {len(feature_cols)} features available, keeping all")
        importance_df = pd.DataFrame({'feature': feature_cols, 'combined_score': [1.0]*len(feature_cols)})
        return df, importance_df, feature_cols

    # Prepare clean data
    df_clean = df[feature_cols].copy()
    if target is not None:
        df_clean['target'] = target
    df_clean = df_clean.dropna()

    if len(df_clean) < 100:
        print(f"       Warning: Only {len(df_clean)} valid rows, keeping all features")
        importance_df = pd.DataFrame({'feature': feature_cols, 'combined_score': [1.0]*len(feature_cols)})
        return df, importance_df, feature_cols

    X = df_clean[feature_cols].values
    y = df_clean['target'].values if 'target' in df_clean.columns else np.random.randn(len(df_clean))
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)

    importance_results = []

    # 1. Correlation
    for i, col in enumerate(feature_cols):
        try:
            corr = np.corrcoef(df_clean[col].values, y)[0, 1]
            corr = 0 if np.isnan(corr) else corr
        except:
            corr = 0
        importance_results.append({
            'feature': col,
            'correlation': abs(corr),
            'correlation_signed': corr
        })

    # 2. Mutual Information
    try:
        mi_scores = mutual_info_regression(X, y, random_state=42, n_neighbors=5)
        for i, col in enumerate(feature_cols):
            importance_results[i]['mutual_info'] = mi_scores[i]
    except:
        for i in range(len(feature_cols)):
            importance_results[i]['mutual_info'] = 0

    # 3. F-regression
    try:
        selector = SelectKBest(score_func=f_regression, k='all')
        selector.fit(X, y)
        f_scores = np.nan_to_num(selector.scores_, nan=0, posinf=0, neginf=0)
        for i, col in enumerate(feature_cols):
            importance_results[i]['f_score'] = f_scores[i]
    except:
        for i in range(len(feature_cols)):
            importance_results[i]['f_score'] = 0

    # Create DataFrame and normalize
    importance_df = pd.DataFrame(importance_results)

    for col in ['correlation', 'mutual_info', 'f_score']:
        max_val = importance_df[col].max()
        if max_val > 0:
            importance_df[f'{col}_norm'] = importance_df[col] / max_val
        else:
            importance_df[f'{col}_norm'] = 0

    # Combined score
    importance_df['combined_score'] = (
        0.35 * importance_df['correlation_norm'] +
        0.35 * importance_df['mutual_info_norm'] +
        0.30 * importance_df['f_score_norm']
    )

    importance_df = importance_df.sort_values('combined_score', ascending=False)
    importance_df['rank'] = range(1, len(importance_df) + 1)

    # Select top N features
    top_features = importance_df.head(n_features)['feature'].tolist()

    print(f"       Top {min(10, n_features)} features:")
    for i, row in importance_df.head(10).iterrows():
        print(f"         {row['rank']:2d}. {row['feature']:<30s} score={row['combined_score']:.4f}")

    # Build final feature set
    final_cols = list(exclude_cols) + top_features
    final_cols = list(dict.fromkeys([col for col in final_cols if col in df.columns]))

    return df[final_cols], importance_df, top_features


# --------------------------------------------------------------------
# 6. K-Means + RBF Feature Generation
# --------------------------------------------------------------------
def rbf_kernel(X: np.ndarray, centers: np.ndarray, gamma: float) -> np.ndarray:
    """Compute RBF (Gaussian) kernel values."""
    distances = cdist(X, centers, metric='sqeuclidean')
    return np.exp(-gamma * distances)


def generate_kmeans_rbf_features(df: pd.DataFrame, feature_cols: List[str],
                                   n_clusters_list: List[int] = [5, 10, 15, 20],
                                   gamma_list: List[float] = [0.01, 0.1, 0.5, 1.0]) -> Tuple[pd.DataFrame, Dict]:
    """
    Generate K-means + RBF features with multiple configurations.
    """
    print("\n" + "="*60)
    print("STEP 5: GENERATING K-MEANS + RBF FEATURES")
    print("="*60)

    X = df[feature_cols].values
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)

    df_result = df.copy()
    all_cluster_stats = {}
    new_features_count = 0

    print(f"\n  Feature columns used: {len(feature_cols)}")
    print(f"  Cluster sizes: {n_clusters_list}")
    print(f"  RBF gamma values: {gamma_list}")

    for n_clusters in n_clusters_list:
        print(f"\n  --- K-Means with {n_clusters} clusters ---")

        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X)
        centers = kmeans.cluster_centers_

        # Add cluster labels
        df_result[f'cluster_{n_clusters}'] = cluster_labels
        new_features_count += 1

        # One-hot encoding
        for i in range(n_clusters):
            df_result[f'cluster_{n_clusters}_is_{i}'] = (cluster_labels == i).astype(int)
            new_features_count += 1

        # Distance to each cluster center
        distances = cdist(X, centers, metric='euclidean')
        for i in range(n_clusters):
            df_result[f'dist_cluster_{n_clusters}_{i}'] = distances[:, i]
            new_features_count += 1

        # Distance aggregates
        df_result[f'dist_nearest_{n_clusters}'] = distances.min(axis=1)
        df_result[f'dist_farthest_{n_clusters}'] = distances.max(axis=1)
        df_result[f'dist_ratio_{n_clusters}'] = distances.min(axis=1) / (distances.max(axis=1) + 1e-8)
        new_features_count += 3

        # RBF features
        for gamma in gamma_list:
            rbf_features = rbf_kernel(X, centers, gamma)

            for i in range(n_clusters):
                df_result[f'rbf_{n_clusters}_g{gamma}_c{i}'] = rbf_features[:, i]
                new_features_count += 1

            # Aggregated RBF
            df_result[f'rbf_{n_clusters}_g{gamma}_max'] = rbf_features.max(axis=1)
            df_result[f'rbf_{n_clusters}_g{gamma}_mean'] = rbf_features.mean(axis=1)
            df_result[f'rbf_{n_clusters}_g{gamma}_std'] = rbf_features.std(axis=1)
            df_result[f'rbf_{n_clusters}_g{gamma}_entropy'] = -np.sum(
                rbf_features * np.log(rbf_features + 1e-10), axis=1
            ) / np.log(n_clusters)
            new_features_count += 4

        # Cluster statistics
        cluster_stats = []
        future_returns = df['returns'].shift(-1) if 'returns' in df.columns else None

        for cluster_id in range(n_clusters):
            mask = cluster_labels == cluster_id
            stats = {
                'cluster': cluster_id,
                'n_clusters': n_clusters,
                'count': mask.sum(),
                'pct': mask.sum() / len(df) * 100,
            }
            if future_returns is not None:
                cluster_returns = future_returns.loc[df.index[mask]]
                stats['avg_return'] = cluster_returns.mean() * 100 if len(cluster_returns) > 0 else 0
                stats['win_rate'] = (cluster_returns > 0).mean() * 100 if len(cluster_returns) > 0 else 0
            cluster_stats.append(stats)

        all_cluster_stats[n_clusters] = pd.DataFrame(cluster_stats)

    # Cluster transition features
    for n_clusters in n_clusters_list:
        df_result[f'cluster_{n_clusters}_changed'] = (
            df_result[f'cluster_{n_clusters}'] != df_result[f'cluster_{n_clusters}'].shift(1)
        ).astype(int)
        cluster_changes = df_result[f'cluster_{n_clusters}_changed'].cumsum()
        df_result[f'cluster_{n_clusters}_duration'] = df_result.groupby(cluster_changes).cumcount()
        new_features_count += 2

    # Inter-cluster agreement
    for i, n1 in enumerate(n_clusters_list[:-1]):
        for n2 in n_clusters_list[i+1:]:
            df_result[f'cluster_agree_{n1}_{n2}'] = (
                df_result[f'cluster_{n1}'] == df_result[f'cluster_{n2}']
            ).astype(int)
            new_features_count += 1

    print(f"\n  Total new features generated: {new_features_count}")

    return df_result, all_cluster_stats


# --------------------------------------------------------------------
# 7. Post-RBF Feature Filtering (SECOND FILTER)
# --------------------------------------------------------------------
def apply_post_rbf_filtering(df: pd.DataFrame, n_final_features: int = 30,
                              correlation_threshold: float = 0.9) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Apply second round of filtering after RBF features are generated:
    1. Remove highly correlated features (>90%)
    2. Select top N features
    """
    print("\n" + "="*60)
    print("STEP 6: POST-RBF FEATURE FILTERING")
    print("="*60)

    exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']

    # Step 6a: Remove correlated features (>90%)
    df_decorr, features_kept, removal_df = remove_correlated_features(
        df,
        threshold=correlation_threshold,
        exclude_cols=exclude_cols,
        step_name="6a"
    )

    # Step 6b: Select top N features
    df_final, importance_df, top_features = select_top_features(
        df_decorr,
        n_features=n_final_features,
        exclude_cols=exclude_cols,
        step_name="6b"
    )

    print(f"\n  FINAL: {len(top_features)} features selected")

    return df_final, importance_df, top_features


# --------------------------------------------------------------------
# 8. Visualization
# --------------------------------------------------------------------
def visualize_results(df: pd.DataFrame, importance_df: pd.DataFrame,
                      cluster_stats: Dict, output_dir: str = './data'):
    """Create comprehensive visualizations."""
    print("\n" + "="*60)
    print("CREATING VISUALIZATIONS")
    print("="*60)

    # 1. Feature Importance (Final)
    plt.figure(figsize=(12, 10))
    top_n = min(30, len(importance_df))
    top_features = importance_df.head(top_n)
    colors = plt.cm.viridis(np.linspace(0, 0.8, top_n))
    plt.barh(range(top_n), top_features['combined_score'].values[::-1], color=colors[::-1])
    plt.yticks(range(top_n), top_features['feature'].values[::-1], fontsize=8)
    plt.xlabel('Combined Importance Score')
    plt.title(f'Top {top_n} Final Features (After All Filtering)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/final_feature_importance.png', dpi=150)
    plt.close()

    # 2. Cluster Analysis Grid
    if cluster_stats:
        n_plots = len(cluster_stats)
        fig, axes = plt.subplots((n_plots + 1) // 2, 2, figsize=(16, 6 * ((n_plots + 1) // 2)))
        axes = axes.flatten() if n_plots > 1 else [axes]

        for idx, (n_clusters, stats_df) in enumerate(cluster_stats.items()):
            if idx >= len(axes):
                break
            ax = axes[idx]

            if 'avg_return' in stats_df.columns:
                colors = ['green' if x > 0 else 'red' for x in stats_df['avg_return']]
                bars = ax.bar(stats_df['cluster'], stats_df['avg_return'], color=colors, alpha=0.7)
                ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
                ax.set_ylabel('Avg Future Return (%)')

                if 'win_rate' in stats_df.columns:
                    for i, (bar, wr) in enumerate(zip(bars, stats_df['win_rate'])):
                        ax.annotate(f'{wr:.0f}%',
                                   xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                                   ha='center', va='bottom' if bar.get_height() >= 0 else 'top',
                                   fontsize=8)

            ax.set_xlabel('Cluster')
            ax.set_title(f'K={n_clusters} Clusters')

        plt.tight_layout()
        plt.savefig(f'{output_dir}/cluster_analysis_grid.png', dpi=150)
        plt.close()

    # 3. Kalman Filter Visualization
    if 'kalman_trend' in df.columns or 'kalman_deviation' in df.columns:
        fig, axes = plt.subplots(3, 1, figsize=(15, 12))

        last_n = min(500, len(df))

        # Price vs Kalman Trend
        ax = axes[0]
        if 'kalman_trend' not in df.columns:
            # Recreate for plotting
            kalman_trend, _ = apply_kalman_filter(df['close'])
            ax.plot(df.index[-last_n:], df['close'].iloc[-last_n:], label='Price', alpha=0.7)
            ax.plot(df.index[-last_n:], kalman_trend.iloc[-last_n:], label='Kalman Trend', linewidth=2)
        else:
            ax.plot(df.index[-last_n:], df['close'].iloc[-last_n:], label='Price', alpha=0.7)
        ax.set_title('Price with Kalman Filter Trend')
        ax.legend()

        # Kalman Deviation
        ax = axes[1]
        if 'kalman_deviation' in df.columns:
            ax.plot(df.index[-last_n:], df['kalman_deviation'].iloc[-last_n:], color='purple')
            ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax.set_title('Kalman Deviation (Price - Trend)')

        # Kalman Velocity
        ax = axes[2]
        if 'kalman_velocity' in df.columns:
            ax.plot(df.index[-last_n:], df['kalman_velocity'].iloc[-last_n:], color='orange')
            ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax.set_title('Kalman Velocity (Trend Rate of Change)')

        plt.tight_layout()
        plt.savefig(f'{output_dir}/kalman_analysis.png', dpi=150)
        plt.close()

    # 4. Feature Pipeline Summary
    fig, ax = plt.subplots(figsize=(12, 6))

    stages = ['Core\nFeatures', 'After\n1st Corr\nFilter', 'Top 25\nSelected',
              'After\nRBF/KMeans', 'After\n2nd Corr\nFilter', 'Final\nTop 30']

    # These are approximate - actual values depend on data
    counts = [80, 60, 25, 400, 150, 30]  # Placeholder values
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c']

    bars = ax.bar(stages, counts, color=colors)
    ax.set_ylabel('Number of Features')
    ax.set_title('Feature Engineering Pipeline: Feature Count at Each Stage')

    for bar, count in zip(bars, counts):
        ax.annotate(str(count), xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                   ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/feature_pipeline_summary.png', dpi=150)
    plt.close()

    print("  Visualizations saved!")


# --------------------------------------------------------------------
# 9. Main Pipeline
# --------------------------------------------------------------------
def main():
    """Main execution pipeline."""
    print("="*70)
    print("MINUTE-LEVEL FEATURE ENGINEERING v4.0")
    print("(With Kalman Filter + Two-Stage Feature Filtering)")
    print("="*70)
    print("\nPipeline Steps:")
    print("  1. Generate core features + Kalman Filter trends")
    print("  2. Apply cleaning (Impute → Outlier removal → Scale)")
    print("  3. FIRST FILTER: Remove correlated features (r > 0.8)")
    print("  4. Select top 25 features")
    print("  5. Generate K-means + RBF features")
    print("  6. SECOND FILTER: Remove correlated features (r > 0.9)")
    print("  7. Select final top 30 features")
    print("="*70)

    # Find input file
    if not os.path.exists(MINUTE_DATA_DIR):
        os.makedirs(MINUTE_DATA_DIR)
        print(f"Created {MINUTE_DATA_DIR}. Please add your data files.")
        return None, None, None

    all_files = [f for f in os.listdir(MINUTE_DATA_DIR) if f.endswith('.csv')]
    if not all_files:
        print(f"No CSV files found in {MINUTE_DATA_DIR}")
        return None, None, None

    input_file = None
    for f in all_files:
        if TICKER.lower() in f.lower():
            input_file = os.path.join(MINUTE_DATA_DIR, f)
            break

    if input_file is None:
        input_file = os.path.join(MINUTE_DATA_DIR, all_files[0])

    print(f"\nInput: {input_file}")

    # Step 1: Load data
    df = load_and_prepare_data(input_file)
    if df is None:
        return None, None, None

    initial_rows = len(df)

    # Step 2: Generate core features (includes Kalman)
    df = generate_core_features(df)
    features_after_core = len([c for c in df.columns if c not in ['open', 'high', 'low', 'close', 'volume']])

    # Step 3: Apply cleaning pipeline
    df_clean, scaler, imputer = apply_cleaning_pipeline(df)

    # Step 4: FIRST FILTER - Remove correlated features (>0.8)
    print("\n" + "="*60)
    print("STEP 3: FIRST CORRELATION FILTER (threshold > 0.8)")
    print("="*60)
    df_decorr1, features_after_corr1, removal_df1 = remove_correlated_features(
        df_clean, threshold=CORRELATION_THRESHOLD_FIRST, step_name="3"
    )

    # Step 5: Select top 25 features
    print("\n" + "="*60)
    print("STEP 4: SELECT TOP 25 FEATURES")
    print("="*60)
    df_selected, importance_df_before, top_features_before = select_top_features(
        df_decorr1, n_features=N_FEATURES_BEFORE_RBF, step_name="4"
    )

    # Step 6: Generate K-means + RBF features
    feature_cols = [col for col in top_features_before if col in df_selected.columns]
    df_with_rbf, cluster_stats = generate_kmeans_rbf_features(
        df_selected, feature_cols,
        n_clusters_list=N_CLUSTERS_LIST,
        gamma_list=RBF_GAMMA_LIST
    )

    features_after_rbf = len(df_with_rbf.columns)

    # Scale new RBF features
    print("\n  Scaling new K-means/RBF features...")
    new_cols = [col for col in df_with_rbf.columns if col not in df_selected.columns]
    numeric_new = [col for col in new_cols if df_with_rbf[col].dtype in ['float64', 'int64', 'int32']]

    if numeric_new:
        new_scaler = MinMaxScaler(feature_range=(-1, 1))
        df_with_rbf[numeric_new] = new_scaler.fit_transform(df_with_rbf[numeric_new])

    # Step 7: SECOND FILTER - Post-RBF filtering
    df_final, importance_df_final, final_features = apply_post_rbf_filtering(
        df_with_rbf,
        n_final_features=N_FEATURES_FINAL,
        correlation_threshold=CORRELATION_THRESHOLD_SECOND
    )

    # Save results
    print("\n" + "="*60)
    print("SAVING RESULTS")
    print("="*60)

    df_final.to_csv(OUTPUT_FILE, index=True)
    print(f"\nFinal data saved to: {OUTPUT_FILE}")

    importance_file = OUTPUT_FILE.replace('.csv', '_importance.csv')
    importance_df_final.to_csv(importance_file, index=False)
    print(f"Feature importance saved to: {importance_file}")

    removal_file = OUTPUT_FILE.replace('.csv', '_removed_features.csv')
    removal_df1.to_csv(removal_file, index=False)
    print(f"Removed features log saved to: {removal_file}")

    if cluster_stats:
        all_stats = pd.concat(cluster_stats.values(), ignore_index=True)
        cluster_file = OUTPUT_FILE.replace('.csv', '_clusters.csv')
        all_stats.to_csv(cluster_file, index=False)
        print(f"Cluster analysis saved to: {cluster_file}")

    # Create visualizations
    try:
        visualize_results(df_final, importance_df_final, cluster_stats,
                         os.path.dirname(OUTPUT_FILE))
    except Exception as e:
        print(f"Visualization error: {e}")
        traceback.print_exc()

    # Print summary
    print("\n" + "="*70)
    print("PIPELINE SUMMARY")
    print("="*70)
    print(f"\nData:")
    print(f"  Initial rows: {initial_rows}")
    print(f"  Final rows: {len(df_final)}")

    print(f"\nFeature Pipeline:")
    print(f"  After core generation (+ Kalman): {features_after_core}")
    print(f"  After 1st correlation filter (>{CORRELATION_THRESHOLD_FIRST}): {len(features_after_corr1)}")
    print(f"  After top-{N_FEATURES_BEFORE_RBF} selection: {N_FEATURES_BEFORE_RBF}")
    print(f"  After K-means + RBF expansion: {features_after_rbf}")
    print(f"  After 2nd correlation filter (>{CORRELATION_THRESHOLD_SECOND}): ~{len(df_final.columns) + 20}")
    print(f"  FINAL (top {N_FEATURES_FINAL}): {len(final_features)}")

    print(f"\nKalman Features Included:")
    kalman_features = [f for f in final_features if 'kalman' in f.lower()]
    for f in kalman_features:
        print(f"    - {f}")

    print(f"\n  TOTAL FINAL FEATURES: {len(df_final.columns)}")

    print("\n" + "="*70)
    print("FEATURE ENGINEERING COMPLETE!")
    print("="*70)

    return df_final, importance_df_final, cluster_stats


if __name__ == "__main__":
    main()


MINUTE-LEVEL FEATURE ENGINEERING v4.0
(With Kalman Filter + Two-Stage Feature Filtering)

Pipeline Steps:
  1. Generate core features + Kalman Filter trends
  2. Apply cleaning (Impute → Outlier removal → Scale)
  3. FIRST FILTER: Remove correlated features (r > 0.8)
  4. Select top 25 features
  5. Generate K-means + RBF features
  6. SECOND FILTER: Remove correlated features (r > 0.9)
  7. Select final top 30 features

Input: ./data/minute_data/GLD.US_minute.csv
Loading data from ./data/minute_data/GLD.US_minute.csv...
Loaded 179291 rows with 24766 detected gaps

STEP 1: GENERATING CORE FEATURES + KALMAN FILTER

  Extracting Kalman Filter Trend Features...
    Added 20 Kalman features
Generated 99 initial features (including Kalman)

STEP 2: APPLYING CLEANING PIPELINE

  [2a] Imputing missing values with median...
       Missing values: 959 → 0

  [2b] Removing outliers with IQR clipping (3x)...
       Clipped 420497 outlier values across 58 columns

  [2c] Scaling features to [-1, 1

In [ ]:
!pip install ta filterpy optuna

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=a2dd1eeac9875cd7120c164e24888ccd99f1501de9a27d3cff38e9b62548f640
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=f108a5e98e50e2fe1610ece3ba4ce1c60ff046620793d565ba0186649f850a7e
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built ta filterpy


In [ ]:
#!/usr/bin/env python
# minute_model_training_v4.py

"""
Minute-Level Model Training & Evaluation Pipeline (v4)
------------------------------------------------------
Features:
- "Sniper" Mode: Trade only when confidence > 0.60
- Anti-Overfitting: Aggressive Regularization (L2, Gamma, Min Child Weight)
- Real Cost Simulation: IBKR Commissions + Slippage
- Ensemble Architecture (XGBoost + Random Forest on GPU)
"""

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any, List

# Sklearn Imports
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    f1_score, roc_auc_score, average_precision_score, log_loss
)
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

# XGBoost
import xgboost as xgb

SHAP_AVAILABLE = False

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")

# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
TICKER = "AAPL"
DATA_DIR = "./data"
INPUT_FILE = f"{DATA_DIR}/{TICKER}.US_minute_engineered_v3.csv"
OUTPUT_DIR = "./data/model_results_v4"

# 1. Trading Logic
TARGET_HORIZON = 1  # Predict next minute
CONFIDENCE_THRESHOLD = 0.55  # Only Buy if Prob > 60% (Sniper Mode)
THRESHOLD = 0.0000  # 0.0 = Simple Up/Down target generation

# 2. Transaction Costs (IBKR Pro + Spread)
# IBKR Fixed: $0.005 per share.
# Spread/Slippage: Est. 1 basis point (0.01%) on entry.
AVG_PRICE_EST = 75.0   # Approx price of ticker (for converting fixed fee to %)
FIXED_COMMISSION = 0.005
SLIPPAGE_BPS = 0.0001

# 3. Training Config
TEST_SIZE_PCT = 0.20
CV_SPLITS = 5
ITERATIONS = 20     # Increased iterations for tighter grid
RANDOM_STATE = 42

# GPU Configuration
USE_GPU = True
DEVICE = 'cuda' if USE_GPU else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------------------------
# 1. Data Loading & Target Generation
# --------------------------------------------------------------------
def load_and_prep_data(filepath: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """Load features, generate target, and return data + raw prices for backtest."""
    print(f"Loading data from {filepath}...")

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    df = pd.read_csv(filepath)

    # Handle Index
    date_col = None
    for c in df.columns:
        if c.lower() in ['date', 'datetime', 'timestamp', 'time']:
            date_col = c
            break

    if date_col:
        df[date_col] = pd.to_datetime(df[date_col])
        df.set_index(date_col, inplace=True)
        df.sort_index(inplace=True)
    else:
        print("Warning: No date column found. Using integer index.")

    # Keep raw prices for backtesting later
    raw_prices = df[['close']].copy()

    # Generate Target: 1 if Return(t+horizon) > Threshold
    future_returns = df['close'].pct_change(periods=TARGET_HORIZON).shift(-TARGET_HORIZON)
    y = (future_returns > THRESHOLD).astype(int)

    # Valid indices (drop NaNs created by shifting or feature engineering)
    valid_indices = ~(df.isna().any(axis=1) | y.isna())

    X = df.loc[valid_indices].copy()
    y = y.loc[valid_indices].copy()
    raw_prices = raw_prices.loc[valid_indices].copy()

    # Drop non-feature columns
    drop_cols = ['open', 'high', 'low', 'close', 'volume', 'returns',
                 'log_returns', 'gap_flag', 'target']
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

    print(f"Data Loaded: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"Class Balance: {y.mean():.2%} positive class")

    return X, y, raw_prices

# --------------------------------------------------------------------
# 2. Time Series Splitting
# --------------------------------------------------------------------
def time_series_train_test_split(X, y, prices, test_size=0.2):
    """Split data respecting temporal order."""
    split_idx = int(len(X) * (1 - test_size))

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    prices_train, prices_test = prices.iloc[:split_idx], prices.iloc[split_idx:]

    return X_train, X_test, y_train, y_test, prices_test

# --------------------------------------------------------------------
# 3. Model Optimization (XGBoost GPU with Aggressive Regularization)
# --------------------------------------------------------------------
def optimize_xgboost_gb(X_train, y_train):
    """
    Gradient Boosting optimization with aggressive regularization
    to fix the diverging loss curve.
    """
    print("\n" + "="*60)
    print(f"OPTIMIZING XGBOOST (GPU: {DEVICE}) - {ITERATIONS} Iterations")
    print("="*60)

    xgb_clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device=DEVICE,
        random_state=RANDOM_STATE
    )

    # Aggressive Grid against Overfitting
    param_dist = {
        'n_estimators': [150, 300, 500],
        'learning_rate': [0.01, 0.03, 0.05],     # Slower learning
        'max_depth': [3, 4, 5],                  # Shallower trees (less memory/overfit)
        'min_child_weight': [5, 10, 20],         # Require more data per leaf (Kills noise)
        'subsample': [0.5, 0.6, 0.7],            # See fewer rows per tree
        'colsample_bytree': [0.5, 0.6, 0.7],     # See fewer features per tree
        'reg_alpha': [0.1, 1.0, 5.0],            # L1 Regularization
        'reg_lambda': [10.0, 50.0, 100.0],       # Strong L2 Regularization (Critical)
        'gamma': [1.0, 5.0]                      # Minimum split loss
    }

    # TimeSeriesSplit for Cross-Validation
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    search = RandomizedSearchCV(
        estimator=xgb_clf,
        param_distributions=param_dist,
        n_iter=ITERATIONS,
        scoring='precision', # Optimized for Precision now (for Sniper mode)
        cv=tscv,
        verbose=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    print(f"Best Parameters: {search.best_params_}")
    print(f"Best CV Precision: {search.best_score_:.4f}")

    return search.best_estimator_

def get_gpu_random_forest():
    """XGBoost configured as Random Forest on GPU"""
    print(f"\nCONFIGURING RANDOM FOREST (GPU: {DEVICE})...")
    return xgb.XGBClassifier(
        n_estimators=1,
        num_parallel_tree=300,
        learning_rate=1.0,
        max_depth=5,            # Reduced depth
        subsample=0.6,          # More bagging
        colsample_bynode=0.6,   # More feature bagging
        reg_lambda=10.0,        # Added regularization to RF
        objective='binary:logistic',
        tree_method='hist',
        device=DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

# --------------------------------------------------------------------
# 4. Diagnostics & Learning Curves
# --------------------------------------------------------------------
def plot_learning_curves(model, X_train, y_train, X_val, y_val):
    """Plot Training vs Validation Loss."""
    print("Generating Learning Curves...")

    eval_set = [(X_train, y_train), (X_val, y_val)]
    model.fit(
        X_train, y_train,
        eval_set=eval_set,
        verbose=False
    )

    results = model.evals_result()
    epochs = len(results['validation_0']['logloss'])
    x_axis = range(0, epochs)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_axis, results['validation_0']['logloss'], label='Train')
    ax.plot(x_axis, results['validation_1']['logloss'], label='Validation')
    ax.legend()
    plt.ylabel('Log Loss')
    plt.title('XGBoost Loss: Training vs Validation')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
    plt.close()

# --------------------------------------------------------------------
# 5. Financial Diagnostics (Net of Costs) & Sniper Logic
# --------------------------------------------------------------------
def calculate_net_returns(model, X_test, prices_test):
    print(f"\nCALCULATING NET RETURNS (Threshold: {CONFIDENCE_THRESHOLD})...")

    # Get Probabilities
    probs = model.predict_proba(X_test)[:, 1]

    # Apply Sniper Threshold
    # If prob > 0.60, Signal = 1. Else 0.
    signals = (probs > CONFIDENCE_THRESHOLD).astype(int)

    # Calculate Costs
    # 1. Commission % = Fixed Fee / Stock Price
    comm_pct = FIXED_COMMISSION / AVG_PRICE_EST
    # 2. Total Cost % per trade = Commission + Slippage
    cost_per_trade_pct = comm_pct + SLIPPAGE_BPS

    print(f"  - Est. Stock Price: ${AVG_PRICE_EST}")
    print(f"  - IBKR Comm: ${FIXED_COMMISSION} ({comm_pct*10000:.2f} bps)")
    print(f"  - Slippage: {SLIPPAGE_BPS*10000:.2f} bps")
    print(f"  - Total Cost per Trade: {cost_per_trade_pct*100:.4f}%")

    # Market Returns (Close-to-Close of NEXT minute)
    # We buy at T (signal), hold for 1 min.
    market_returns = prices_test.pct_change().shift(-1).fillna(0)['close']
    market_returns = market_returns.iloc[:len(signals)]

    # Gross Strategy Returns
    gross_returns = market_returns * signals

    # Calculate Cost Drag
    # We pay cost when we ENTER a trade (assuming exit is frictionless or included in slippage)
    # Simplification: Pay cost every time signal is 1 (entry)
    cost_drag = signals * cost_per_trade_pct

    # Net Returns
    net_returns = gross_returns - cost_drag

    # Plotting
    cum_market = (1 + market_returns).cumprod()
    cum_gross = (1 + gross_returns).cumprod()
    cum_net = (1 + net_returns).cumprod()

    plt.figure(figsize=(12, 6))
    plt.plot(cum_market, label="Buy & Hold", color='gray', alpha=0.5)
    plt.plot(cum_gross, label="Model Gross", color='green', alpha=0.6, linestyle='--')
    plt.plot(cum_net, label="Model Net (Real)", color='blue', linewidth=2)

    plt.title(f"Net Profit Analysis (Thresh > {CONFIDENCE_THRESHOLD})")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/net_cumulative_returns.png")
    plt.close()

    total_trades = signals.sum()
    final_return = cum_net.iloc[-1] - 1
    print(f"\n  RESULTS:")
    print(f"  - Trades Taken: {total_trades}")
    print(f"  - Final Net Return: {final_return:.2%}")

    return signals, probs

def plot_custom_confusion_matrix(y_test, signals):
    """Confusion Matrix based on the custom threshold"""
    cm = confusion_matrix(y_test, signals)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix (Threshold > {CONFIDENCE_THRESHOLD})")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrix_threshold.png")
    plt.close()

def plot_calibration_curve_func(y_test, probs):
    """Check if probability output is reliable."""
    print("Generating Calibration Curve...")

    fraction_of_positives, mean_predicted_value = calibration_curve(y_test, probs, n_bins=10)

    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
    plt.ylabel("Fraction of positives")
    plt.xlabel("Mean predicted probability")
    plt.title("Calibration Curve (Reliability Diagram)")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/calibration_curve.png")
    plt.close()

def check_permutation_importance(model, X_val, y_val):
    """Check feature stability."""
    print("Calculating Permutation Importance...")
    if len(X_val) > 2000:
        X_sub = X_val.sample(2000, random_state=42)
        y_sub = y_val.loc[X_sub.index]
    else:
        X_sub, y_sub = X_val, y_val

    result = permutation_importance(
        model, X_sub, y_sub, n_repeats=5, random_state=42, n_jobs=-1
    )

    sorted_idx = result.importances_mean.argsort()[-20:]
    plt.figure(figsize=(10, 8))
    plt.boxplot(
        result.importances[sorted_idx].T,
        vert=False,
        labels=X_val.columns[sorted_idx]
    )
    plt.title("Permutation Importances (Validation Set)")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/permutation_importance.png")
    plt.close()

def plot_shap_summary(model, X_test):
    """SHAP Interpretability."""
    if not SHAP_AVAILABLE:
        return
    print("Generating SHAP Summary...")
    try:
        if isinstance(model, VotingClassifier):
            estimator = model.named_estimators_['gb']
        else:
            estimator = model

        explainer = shap.TreeExplainer(estimator)
        shap_values = explainer.shap_values(X_test.iloc[:1000])

        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_test.iloc[:1000], show=False)
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/shap_summary.png")
        plt.close()
    except Exception as e:
        print(f"SHAP Error: {e}")

# --------------------------------------------------------------------
# 6. Main Pipeline
# --------------------------------------------------------------------
def main():
    gc.collect()
    print("="*60)
    print(f"MODEL TRAINING v4 (Sniper Mode > {CONFIDENCE_THRESHOLD})")
    print("="*60)

    # 1. Load Data
    try:
        X, y, prices = load_and_prep_data(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: Input file {INPUT_FILE} not found.")
        return

    # 2. Train/Test Split
    X_train, X_test, y_train, y_test, prices_test = time_series_train_test_split(
        X, y, prices, test_size=TEST_SIZE_PCT
    )
    print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

    # 3. Optimize XGBoost (GPU)
    # Using stricter regularization grid
    best_xgb = optimize_xgboost_gb(X_train, y_train)

    # 3b. Learning Curve Check
    val_cut = int(len(X_train) * 0.9)
    X_tr_plot, X_val_plot = X_train.iloc[:val_cut], X_train.iloc[val_cut:]
    y_tr_plot, y_val_plot = y_train.iloc[:val_cut], y_train.iloc[val_cut:]

    xgb_plotter = clone(best_xgb)
    plot_learning_curves(xgb_plotter, X_tr_plot, y_tr_plot, X_val_plot, y_val_plot)
    del xgb_plotter, X_tr_plot, X_val_plot, y_tr_plot, y_val_plot
    gc.collect()

    # 4. Random Forest on GPU
    rf_model = get_gpu_random_forest()
    rf_model.fit(X_train, y_train)

    # 5. Ensemble (Voting)
    print("\nTraining Voting Ensemble...")
    ensemble = VotingClassifier(
        estimators=[
            ('gb', best_xgb),
            ('rf', rf_model)
        ],
        voting='soft',
        n_jobs=1
    )
    ensemble.fit(X_train, y_train)

    # 6. Comprehensive Evaluation with Real Constraints
    print("\n" + "="*60)
    print("FINAL EVALUATION & DIAGNOSTICS")
    print("="*60)

    # Calculate Net Returns and get Signals/Probs
    signals, probs = calculate_net_returns(ensemble, X_test, prices_test)

    # Metrics based on threshold
    plot_custom_confusion_matrix(y_test, signals)

    # Additional Diagnostics
    plot_calibration_curve_func(y_test, probs)
    check_permutation_importance(ensemble, X_test, y_test)
    plot_shap_summary(ensemble, X_test)

    print("\n" + "="*60)
    print("DONE. Results saved to:", OUTPUT_DIR)
    print("="*60)

if __name__ == "__main__":
    main()

MODEL TRAINING v4 (Sniper Mode > 0.55)
Loading data from ./data/AAPL.US_minute_engineered_v3.csv...
Error: Input file ./data/AAPL.US_minute_engineered_v3.csv not found.


In [ ]:
#!/usr/bin/env python
# minute_model_audit_v6.py

"""
Minute-Level Model Training & AUDIT Pipeline (v6 - Precision Sniper)
--------------------------------------------------------------------
THE FIX:
1. REMOVED 'scale_pos_weight': Stops probability inflation.
2. INCREASED L2 Regularization (reg_lambda): Crushes noise/overfitting.
3. INCREASED Min Child Weight: Prevents model from looking at tiny clusters of data.
4. HARDER TARGET: Only trains on moves > 0.10% (0.0010).

Objective: High Precision, Low Frequency.
"""

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any, List

# Sklearn Imports
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    f1_score, roc_auc_score, average_precision_score, log_loss
)
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone

# XGBoost
import xgboost as xgb

# Interpretability
SHAP_AVAILABLE = False

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")

# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
TICKER = "BABA"
DATA_DIR = "./data"
INPUT_FILE = f"{DATA_DIR}/{TICKER}.US_minute_engineered_v3.csv"
OUTPUT_DIR = "./data/model_audit_v6"

# 1. Trading Logic (The "Big Move" Logic)
TARGET_HORIZON = 5         # Look 5 minutes ahead (Allows time for move to develop)
CONFIDENCE_THRESHOLD = 0.52 # Must be > 52% confident (Un-inflated) to pull trigger
THRESHOLD = 0.0010         # 10 Basis Points. We ignore tiny noise.

# 2. Transaction Costs (IBKR Pro + Spread)
AVG_PRICE_EST = 75.0
FIXED_COMMISSION = 0.005
SLIPPAGE_BPS = 0.0001

# 3. Training Config
TEST_SIZE_PCT = 0.20
CV_SPLITS = 5
ITERATIONS = 20
RANDOM_STATE = 42

# GPU Configuration
USE_GPU = True
DEVICE = 'cuda' if USE_GPU else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------------------------
# 1. Data Loading & Target Generation
# --------------------------------------------------------------------
def load_and_prep_data(filepath: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    print(f"Loading data from {filepath}...")

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    df = pd.read_csv(filepath)

    # Handle Index
    date_col = None
    for c in df.columns:
        if c.lower() in ['date', 'datetime', 'timestamp', 'time']:
            date_col = c
            break

    if date_col:
        df[date_col] = pd.to_datetime(df[date_col])
        df.set_index(date_col, inplace=True)
        df.sort_index(inplace=True)

    # Keep Raw Prices (Open/Close) for Audit
    raw_prices = df[['open', 'close']].copy()

    # Generate Target: Return(t+5) > 0.10%
    future_returns = df['close'].pct_change(periods=TARGET_HORIZON).shift(-TARGET_HORIZON)
    y = (future_returns > THRESHOLD).astype(int)

    # Valid indices
    valid_indices = ~(df.isna().any(axis=1) | y.isna())

    X = df.loc[valid_indices].copy()
    y = y.loc[valid_indices].copy()
    raw_prices = raw_prices.loc[valid_indices].copy()

    # Drop non-feature columns
    drop_cols = ['open', 'high', 'low', 'close', 'volume', 'returns',
                 'log_returns', 'gap_flag', 'target']
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

    print(f"Data Loaded: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"Class Balance: {y.mean():.2%} positive class (Big Moves > 0.10%)")

    return X, y, raw_prices

# --------------------------------------------------------------------
# 2. Time Series Splitting
# --------------------------------------------------------------------
def time_series_train_test_split(X, y, prices, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    prices_train, prices_test = prices.iloc[:split_idx], prices.iloc[split_idx:]
    return X_train, X_test, y_train, y_test, prices_test

# --------------------------------------------------------------------
# 3. Model Optimization (XGBoost GPU - HIGH REGULARIZATION)
# --------------------------------------------------------------------
def optimize_xgboost_gb(X_train, y_train):
    print("\n" + "="*60)
    print(f"OPTIMIZING XGBOOST (GPU: {DEVICE}) - {ITERATIONS} Iterations")
    print("="*60)

    # NOTE: NO scale_pos_weight. We want raw, honest probabilities.

    xgb_clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        device=DEVICE,
        random_state=RANDOM_STATE
    )

    # HIGH REGULARIZATION GRID
    param_dist = {
        'n_estimators': [150, 300, 500],
        'learning_rate': [0.005, 0.01, 0.02], # SLOW learning to prevent memorization
        'max_depth': [3, 4],                  # SHALLOW trees (prevent complexity)
        'min_child_weight': [20, 50, 100],    # CRITICAL: Requires 50+ samples to make a decision
        'subsample': [0.5, 0.6],              # Heavy bagging
        'colsample_bytree': [0.5, 0.6],
        'reg_alpha': [1.0, 5.0, 10.0],        # L1 Regularization
        'reg_lambda': [10.0, 50.0, 100.0],    # L2 Regularization (Very High)
        'gamma': [1.0, 5.0]                   # Minimum loss reduction required
    }

    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    search = RandomizedSearchCV(
        estimator=xgb_clf,
        param_distributions=param_dist,
        n_iter=ITERATIONS,
        scoring='precision',  # Optimize purely for "When I shoot, I hit"
        cv=tscv,
        verbose=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    print(f"Best Parameters: {search.best_params_}")
    print(f"Best CV Precision: {search.best_score_:.4f}")
    return search.best_estimator_

def get_gpu_random_forest():
    print(f"\nCONFIGURING RANDOM FOREST (GPU: {DEVICE})...")
    # A conservative Random Forest as a "Second Opinion"
    return xgb.XGBClassifier(
        n_estimators=1,
        num_parallel_tree=300,
        learning_rate=1.0,
        max_depth=4,            # Very shallow
        subsample=0.5,          # High variance reduction
        colsample_bynode=0.5,
        reg_lambda=50.0,        # High L2
        objective='binary:logistic',
        tree_method='hist',
        device=DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

# --------------------------------------------------------------------
# 4. Diagnostics
# --------------------------------------------------------------------
def plot_learning_curves(model, X_train, y_train, X_val, y_val):
    print("Generating Learning Curves...")
    eval_set = [(X_train, y_train), (X_val, y_val)]
    model.fit(X_train, y_train, eval_set=eval_set, verbose=False)
    results = model.evals_result()
    epochs = len(results['validation_0']['logloss'])
    x_axis = range(0, epochs)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_axis, results['validation_0']['logloss'], label='Train')
    ax.plot(x_axis, results['validation_1']['logloss'], label='Validation')
    ax.legend()
    plt.title('XGBoost Loss: Training vs Validation')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
    plt.close()

# --------------------------------------------------------------------
# 5. Financial Diagnostics (ACID TEST: Next Open Execution)
# --------------------------------------------------------------------
def calculate_net_returns(model, X_test, prices_test):
    print(f"\nREALITY CHECK: DYNAMIC THRESHOLD EXECUTION...")

    # 1. Get Raw Probabilities
    probs = model.predict_proba(X_test)[:, 1]

    # 2. Analyze the Scores (The Diagnosis)
    print(f"  Probability Stats:")
    print(f"  - Max Score: {probs.max():.4f}")
    print(f"  - 99th Percentile: {np.percentile(probs, 99):.4f}")
    print(f"  - Mean Score: {probs.mean():.4f}")

    # 3. Dynamic Thresholding (The Fix)
    # We force the model to trade its top 0.5% ideas (approx 1-2 trades per day)
    # This aligns with 'Sniper' logic without needing to guess the raw number.
    dynamic_threshold = np.percentile(probs, 99.5)

    # Safety: Don't go below 0.30 even if dynamic says so (avoids trading trash)
    final_thresh = max(0.30, dynamic_threshold)

    print(f"  > Auto-Selected Threshold: {final_thresh:.4f} (Top 0.5% Logic)")

    # Apply Threshold
    signals = (probs > final_thresh).astype(int)

    # Calculate Costs
    comm_pct = FIXED_COMMISSION / AVG_PRICE_EST
    cost_per_trade_pct = comm_pct + SLIPPAGE_BPS

    print(f"  - Est. Stock Price: ${AVG_PRICE_EST}")
    print(f"  - Total Cost per Trade: {cost_per_trade_pct*100:.4f}%")

    # ------------------------------------------------------------------
    # AUDIT LOGIC: Execute at Open(t+1), Hold for 5 Minutes
    # ------------------------------------------------------------------
    entry_price = prices_test['open'].shift(-1)
    exit_price = prices_test['close'].shift(-TARGET_HORIZON)

    raw_returns = (exit_price - entry_price) / entry_price
    raw_returns = raw_returns.fillna(0)
    raw_returns = raw_returns.iloc[:len(signals)]

    # Gross Strategy Returns
    gross_returns = raw_returns * signals

    # Calculate Cost Drag
    cost_drag = signals * cost_per_trade_pct

    # Net Returns
    net_returns = gross_returns - cost_drag

    # Plotting
    market_benchmark = prices_test['close'].pct_change().fillna(0)
    cum_market = (1 + market_benchmark).cumprod()
    cum_gross = (1 + gross_returns).cumprod()
    cum_net = (1 + net_returns).cumprod()

    plt.figure(figsize=(12, 6))
    plt.plot(cum_market, label="Market (Buy & Hold)", color='gray', alpha=0.3)
    plt.plot(cum_gross, label="Model Gross", color='green', alpha=0.6, linestyle='--')
    plt.plot(cum_net, label="Model Net (Realized)", color='blue', linewidth=2)

    plt.title(f"AUDIT: Top 0.5% Trades (Thresh > {final_thresh:.4f})")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/net_cumulative_returns_AUDIT.png")
    plt.close()

    total_trades = signals.sum()
    final_return = cum_net.iloc[-1] - 1

    print(f"\n  AUDIT RESULTS:")
    print(f"  - Trades Taken: {total_trades}")
    print(f"  - Final Net Return: {final_return:.2%}")

    if total_trades < 10:
        print("  (!) NOTE: Still very few trades. Model might be extremely conservative.")
    elif final_return > 0:
         print("  (+) SUCCESS: Real Alpha Found in the top percentile.")
    else:
         print("  (!) FAILURE: Even the top signals are not profitable net of costs.")

    return signals, probs

def plot_custom_confusion_matrix(y_test, signals):
    cm = confusion_matrix(y_test, signals)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix (Threshold > {CONFIDENCE_THRESHOLD})")
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
    plt.close()

def plot_calibration_curve_func(y_test, probs):
    print("Generating Calibration Curve...")
    fraction_of_positives, mean_predicted_value = calibration_curve(y_test, probs, n_bins=10)
    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
    plt.ylabel("Fraction of positives")
    plt.xlabel("Mean predicted probability")
    plt.title("Calibration Curve")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/calibration_curve.png")
    plt.close()

def check_permutation_importance(model, X_val, y_val):
    print("Calculating Permutation Importance...")
    if len(X_val) > 2000:
        X_sub = X_val.sample(2000, random_state=42)
        y_sub = y_val.loc[X_sub.index]
    else:
        X_sub, y_sub = X_val, y_val

    result = permutation_importance(model, X_sub, y_sub, n_repeats=5, random_state=42, n_jobs=-1)
    sorted_idx = result.importances_mean.argsort()[-20:]
    plt.figure(figsize=(10, 8))
    plt.boxplot(result.importances[sorted_idx].T, vert=False, labels=X_val.columns[sorted_idx])
    plt.title("Permutation Importances")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/permutation_importance.png")
    plt.close()

# --------------------------------------------------------------------
# 6. Main Pipeline
# --------------------------------------------------------------------
def main():
    gc.collect()
    print("="*60)
    print(f"MODEL AUDIT V6 (Horizon {TARGET_HORIZON}m | Reg L2=High)")
    print("="*60)

    # 1. Load Data
    try:
        X, y, prices = load_and_prep_data(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: Input file {INPUT_FILE} not found.")
        return

    # 2. Train/Test Split
    X_train, X_test, y_train, y_test, prices_test = time_series_train_test_split(
        X, y, prices, test_size=TEST_SIZE_PCT
    )
    print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

    # 3. Optimize XGBoost (GPU)
    best_xgb = optimize_xgboost_gb(X_train, y_train)

    # 3b. Learning Curve Check
    val_cut = int(len(X_train) * 0.9)
    X_tr_plot, X_val_plot = X_train.iloc[:val_cut], X_train.iloc[val_cut:]
    y_tr_plot, y_val_plot = y_train.iloc[:val_cut], y_train.iloc[val_cut:]
    xgb_plotter = clone(best_xgb)
    plot_learning_curves(xgb_plotter, X_tr_plot, y_tr_plot, X_val_plot, y_val_plot)
    del xgb_plotter, X_tr_plot, X_val_plot, y_tr_plot, y_val_plot
    gc.collect()

    # 4. Random Forest on GPU
    rf_model = get_gpu_random_forest()
    rf_model.fit(X_train, y_train)

    # 5. Ensemble (Voting)
    print("\nTraining Voting Ensemble...")
    ensemble = VotingClassifier(
        estimators=[('gb', best_xgb), ('rf', rf_model)],
        voting='soft', n_jobs=1
    )
    ensemble.fit(X_train, y_train)

    # 6. Comprehensive Evaluation with Real Constraints
    print("\n" + "="*60)
    print("FINAL AUDIT & DIAGNOSTICS")
    print("="*60)

    # Calculate Net Returns and get Signals/Probs
    signals, probs = calculate_net_returns(ensemble, X_test, prices_test)

    # Metrics
    plot_custom_confusion_matrix(y_test, signals)
    plot_calibration_curve_func(y_test, probs)
    check_permutation_importance(ensemble, X_test, y_test)

    print("\n" + "="*60)
    print("DONE. Results saved to:", OUTPUT_DIR)
    print("="*60)

if __name__ == "__main__":
    main()

MODEL AUDIT V6 (Horizon 5m | Reg L2=High)
Loading data from ./data/BABA.US_minute_engineered_v3.csv...
Data Loaded: 186186 samples, 418 features
Class Balance: 19.10% positive class (Big Moves > 0.10%)
Train samples: 148948 | Test samples: 37238

OPTIMIZING XGBOOST (GPU: cuda) - 20 Iterations
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'subsample': 0.5, 'reg_lambda': 50.0, 'reg_alpha': 10.0, 'n_estimators': 500, 'min_child_weight': 20, 'max_depth': 4, 'learning_rate': 0.02, 'gamma': 1.0, 'colsample_bytree': 0.5}
Best CV Precision: 0.6242
Generating Learning Curves...

CONFIGURING RANDOM FOREST (GPU: cuda)...

Training Voting Ensemble...

FINAL AUDIT & DIAGNOSTICS

REALITY CHECK: DYNAMIC THRESHOLD EXECUTION...
  Probability Stats:
  - Max Score: 0.4866
  - 99th Percentile: 0.4328
  - Mean Score: 0.2008
  > Auto-Selected Threshold: 0.4439 (Top 0.5% Logic)
  - Est. Stock Price: $75.0
  - Total Cost per Trade: 0.0167%

  AUDIT RESULTS:
  - Trades Taken: 

In [ ]:
!pip install verbose

In [ ]:
#!/usr/bin/env python
# minute_model_gpu_native_v10_adaboost.py

"""
GPU-Native Model Training Pipeline (v10) - Precision-Focused Edition with Ensemble Agreement
+ AdaBoost added (CPU) as a third model in the ensemble

---------------------------------------------------------------------------------------------
Key Changes (this version):

1) Added AdaBoostClassifier (CPU sklearn) with Optuna tuning:
   - Objective: maximize recall while maintaining precision >= MIN_PRECISION on validation set
   - Uses sample_weight based on auto-balanced pos_weight from training set

2) Full ensemble is now: XGBoost + NN Ensemble + AdaBoost
   - Standard predict_proba: weighted average of the 3 model probabilities
   - Agreement-based predict: NN agreement filter optionally combined with thresholded XGB + Ada

3) Added Config options:
   - USE_ADABOOST, ADABOOST_TRIALS
   - REQUIRE_ADA_AGREE (analogous to REQUIRE_XGB_AGREE)

Notes:
- AdaBoost is CPU-based. This is normal; it still integrates cleanly into your precision-first workflow.
- I also fixed a couple of syntax pitfalls (notably an f1 calculation line-break issue).
"""

import os
import time
import warnings
from typing import Tuple, Dict, List, Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Sklearn metrics (evaluation)
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, average_precision_score,
    roc_auc_score
)

# NEW: AdaBoost (CPU)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Optuna
import optuna
from optuna.samplers import TPESampler

# XGBoost
import xgboost as xgb

# CuPy (optional)
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False
    cp = np

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')


# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
@dataclass
class Config:
    TICKER: str = "GLD"
    DATA_DIR: str = "./data"
    OUTPUT_DIR: str = "./data/model_gpu_native_v12"

    # Trading
    TARGET_HORIZON: int = 5
    THRESHOLD: float = 0.0010

    # PRECISION TARGET
    MIN_PRECISION: float = 0.51  # Minimum precision threshold

    # Costs
    AVG_PRICE_EST: float = 75.0
    FIXED_COMMISSION: float = 0.005
    SLIPPAGE_BPS: float = 0.0001

    # Training
    TEST_SIZE_PCT: float = 0.20
    OPTUNA_TRIALS: int = 50
    RANDOM_STATE: int = 42
    EARLY_STOPPING: int = 50

    # GPU
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Mini-Batch Configuration
    BATCH_SIZE: int = 2048
    SHUFFLE_TRAIN: bool = True
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True

    # Neural Network Regularization
    NN_EPOCHS: int = 150
    NN_LR: float = 0.001
    NN_PATIENCE: int = 20
    NN_L1_LAMBDA: float = 0.001
    NN_L2_LAMBDA: float = 0.01
    NN_DROPOUT_RATES: List[float] = field(default_factory=lambda: [0.5, 0.4, 0.3, 0.2])
    NN_LABEL_SMOOTHING: float = 0.1
    NN_GRADIENT_CLIP: float = 1.0

    # Ensemble Configuration
    N_ENSEMBLE_MODELS: int = 5
    ENSEMBLE_DIVERSITY: bool = True

    # Ensemble Agreement Configuration
    USE_AGREEMENT: bool = True
    DEFAULT_AGREEMENT_THRESHOLD: float = 0.6  # 60% of models must agree
    REQUIRE_XGB_AGREE: bool = True            # XGBoost must also agree

    # NEW: AdaBoost
    USE_ADABOOST: bool = True
    ADABOOST_TRIALS: int = 30
    REQUIRE_ADA_AGREE: bool = True            # AdaBoost must also agree (when using combined agreement)

    def __post_init__(self):
        self.INPUT_FILE = f"{self.DATA_DIR}/{self.TICKER}.US_minute_engineered_v4.csv"
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        print(f"✓ Device: {self.DEVICE}")
        print(f"✓ Target Precision: >= {self.MIN_PRECISION:.0%}")
        print(f"✓ Ensemble Agreement: {self.USE_AGREEMENT} (threshold: {self.DEFAULT_AGREEMENT_THRESHOLD:.0%})")
        print(f"✓ AdaBoost Enabled: {self.USE_ADABOOST}")
        if self.DEVICE == "cuda":
            print(f"  GPU: {torch.cuda.get_device_name(0)}")


CONFIG = Config()


# --------------------------------------------------------------------
# GPU Utilities
# --------------------------------------------------------------------
def to_torch(x, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(CONFIG.DEVICE, dtype=dtype)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.values
    return torch.tensor(x, dtype=dtype, device=CONFIG.DEVICE)


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    if CUPY_AVAILABLE and isinstance(x, cp.ndarray):
        return cp.asnumpy(x)
    return np.asarray(x)


def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        print(f"  📊 GPU Memory: {allocated:.2f}GB allocated")


# --------------------------------------------------------------------
# Auto-Balance Calculator
# --------------------------------------------------------------------
def calculate_class_weights(y: np.ndarray) -> Dict:
    """
    Auto-calculate class weights based on class imbalance.
    Returns weights for XGBoost, Neural Network, and provides a usable pos_weight for AdaBoost sample_weight.
    """
    n_neg = (y == 0).sum()
    n_pos = (y == 1).sum()

    pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
    scale_pos_weight = pos_weight

    print(f"\n  ⚖️ Auto-Balanced Class Weights:")
    print(f"     Negative samples: {n_neg:,} ({n_neg/(n_neg+n_pos):.1%})")
    print(f"     Positive samples: {n_pos:,} ({n_pos/(n_neg+n_pos):.1%})")
    print(f"     pos_weight (NN): {pos_weight:.3f}")
    print(f"     scale_pos_weight (XGB): {scale_pos_weight:.3f}")

    return {
        'pos_weight': float(pos_weight),
        'scale_pos_weight': float(scale_pos_weight),
        'n_neg': int(n_neg),
        'n_pos': int(n_pos)
    }


def make_sample_weight(y: np.ndarray, pos_weight: float) -> np.ndarray:
    """
    For classifiers that accept sample_weight in fit() (AdaBoost does),
    weight positive samples more heavily using pos_weight.
    """
    y = np.asarray(y).astype(int)
    w = np.ones_like(y, dtype=np.float32)
    w[y == 1] = float(pos_weight)
    return w


# --------------------------------------------------------------------
# Precision-Recall Analysis
# --------------------------------------------------------------------
class PrecisionRecallAnalyzer:
    """Analyze precision-recall tradeoff and find optimal threshold."""

    def __init__(self, min_precision: float = 0.54):
        self.min_precision = min_precision
        self.precisions = None
        self.recalls = None
        self.thresholds = None
        self.optimal_threshold = None
        self.optimal_precision = None
        self.optimal_recall = None
        self.average_precision = None

    def analyze(self, y_true: np.ndarray, y_proba: np.ndarray) -> Dict:
        """Compute precision-recall curve and find threshold for target precision."""
        self.precisions, self.recalls, self.thresholds = precision_recall_curve(y_true, y_proba)
        self.average_precision = average_precision_score(y_true, y_proba)

        valid_mask = self.precisions[:-1] >= self.min_precision

        if valid_mask.any():
            valid_indices = np.where(valid_mask)[0]
            best_idx = valid_indices[np.argmax(self.recalls[:-1][valid_indices])]
            self.optimal_threshold = float(self.thresholds[best_idx])
            self.optimal_precision = float(self.precisions[best_idx])
            self.optimal_recall = float(self.recalls[best_idx])
        else:
            best_idx = int(np.argmax(self.precisions[:-1]))
            self.optimal_threshold = float(self.thresholds[best_idx])
            self.optimal_precision = float(self.precisions[best_idx])
            self.optimal_recall = float(self.recalls[best_idx])
            print(f"  ⚠️ Warning: Cannot achieve {self.min_precision:.0%} precision")
            print(f"     Best achievable: {self.optimal_precision:.2%}")

        f1_at_threshold = (2.0 * (self.optimal_precision * self.optimal_recall)) / (
            self.optimal_precision + self.optimal_recall + 1e-8
        )

        return {
            'optimal_threshold': self.optimal_threshold,
            'optimal_precision': self.optimal_precision,
            'optimal_recall': self.optimal_recall,
            'average_precision': float(self.average_precision),
            'f1_at_threshold': float(f1_at_threshold),
        }

    def plot_precision_recall_curve(self, save_path: str = None):
        """Plot precision-recall curve with optimal threshold marked."""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Plot 1: PR curve
        ax1 = axes[0]
        ax1.plot(self.recalls, self.precisions, 'b-', linewidth=2, label='PR Curve')
        ax1.fill_between(self.recalls, self.precisions, alpha=0.2)
        ax1.scatter([self.optimal_recall], [self.optimal_precision],
                    color='red', s=150, zorder=5, marker='*',
                    label=f'Optimal (P={self.optimal_precision:.2%}, R={self.optimal_recall:.2%})')
        ax1.axhline(y=self.min_precision, color='green', linestyle='--',
                    label=f'Target Precision = {self.min_precision:.0%}')
        ax1.set_xlabel('Recall', fontsize=12)
        ax1.set_ylabel('Precision', fontsize=12)
        ax1.set_title(f'Precision-Recall Curve (AP={self.average_precision:.3f})', fontsize=14)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim([0, 1])
        ax1.set_ylim([0, 1])

        # Plot 2: Precision/Recall vs threshold
        ax2 = axes[1]
        ax2.plot(self.thresholds, self.precisions[:-1], 'b-', linewidth=2, label='Precision')
        ax2.plot(self.thresholds, self.recalls[:-1], 'r-', linewidth=2, label='Recall')
        ax2.axvline(x=self.optimal_threshold, color='green', linestyle='--',
                    label=f'Optimal Threshold = {self.optimal_threshold:.3f}')
        ax2.axhline(y=self.min_precision, color='purple', linestyle=':', alpha=0.7,
                    label=f'Min Precision = {self.min_precision:.0%}')
        ax2.set_xlabel('Threshold', fontsize=12)
        ax2.set_ylabel('Score', fontsize=12)
        ax2.set_title('Precision & Recall vs Threshold', fontsize=14)
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim([0, 1])
        ax2.set_ylim([0, 1])

        # Plot 3: F1 vs threshold
        ax3 = axes[2]
        f1_scores = 2 * (self.precisions[:-1] * self.recalls[:-1]) / (
            self.precisions[:-1] + self.recalls[:-1] + 1e-8
        )
        ax3.plot(self.thresholds, f1_scores, 'g-', linewidth=2, label='F1 Score')
        ax3.axvline(x=self.optimal_threshold, color='red', linestyle='--',
                    label='Selected Threshold')

        max_f1_idx = int(np.argmax(f1_scores))
        max_f1_threshold = float(self.thresholds[max_f1_idx])
        ax3.axvline(x=max_f1_threshold, color='blue', linestyle=':', alpha=0.7,
                    label=f'Max F1 Threshold = {max_f1_threshold:.3f}')
        ax3.set_xlabel('Threshold', fontsize=12)
        ax3.set_ylabel('F1 Score', fontsize=12)
        ax3.set_title('F1 Score vs Threshold', fontsize=14)
        ax3.legend(loc='best')
        ax3.grid(True, alpha=0.3)
        ax3.set_xlim([0, 1])

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Precision-Recall curve saved to {save_path}")
        plt.close()
        return fig


# --------------------------------------------------------------------
# Confusion Matrix Visualization
# --------------------------------------------------------------------
class ConfusionMatrixAnalyzer:
    """Detailed confusion matrix analysis and visualization."""

    def __init__(self):
        self.cm = None
        self.tn = None
        self.fp = None
        self.fn = None
        self.tp = None
        self.metrics = None

    def compute(self, y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
        self.cm = confusion_matrix(y_true, y_pred)
        self.tn, self.fp, self.fn, self.tp = self.cm.ravel()

        precision = self.tp / (self.tp + self.fp) if (self.tp + self.fp) > 0 else 0.0
        recall = self.tp / (self.tp + self.fn) if (self.tp + self.fn) > 0 else 0.0
        specificity = self.tn / (self.tn + self.fp) if (self.tn + self.fp) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + 1e-12)

        fpr = self.fp / (self.fp + self.tn) if (self.fp + self.tn) > 0 else 0.0
        npv = self.tn / (self.tn + self.fn) if (self.tn + self.fn) > 0 else 0.0

        self.metrics = {
            'precision': float(precision),
            'recall': float(recall),
            'specificity': float(specificity),
            'f1_score': float(f1),
            'accuracy': float(accuracy),
            'false_positive_rate': float(fpr),
            'npv': float(npv),
            'true_positives': int(self.tp),
            'true_negatives': int(self.tn),
            'false_positives': int(self.fp),
            'false_negatives': int(self.fn),
            'total_predictions': int(len(y_pred)),
            'total_positive_predictions': int(np.sum(y_pred)),
            'actual_positives': int(np.sum(y_true)),
        }
        return self.metrics

    def plot(self, save_path: str = None, title: str = "Confusion Matrix"):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        ax1 = axes[0]

        cm_normalized = self.cm.astype('float') / (self.cm.sum(axis=1)[:, np.newaxis] + 1e-12)

        sns.heatmap(
            self.cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Predict Negative\n(No Trade)', 'Predict Positive\n(Trade)'],
            yticklabels=['Actual Negative\n(Price Down/Flat)', 'Actual Positive\n(Price Up)'],
            annot_kws={'size': 14}
        )

        for i in range(2):
            for j in range(2):
                pct = cm_normalized[i, j] * 100
                ax1.text(j + 0.5, i + 0.7, f'({pct:.1f}%)',
                         ha='center', va='center', fontsize=10, color='gray')

        ax1.set_title(f'{title}\nThreshold for Precision ≥ {CONFIG.MIN_PRECISION:.0%}', fontsize=14)
        ax1.set_ylabel('Actual', fontsize=12)
        ax1.set_xlabel('Predicted', fontsize=12)

        ax2 = axes[1]
        metrics_to_plot = {
            'Precision': self.metrics['precision'],
            'Recall': self.metrics['recall'],
            'Specificity': self.metrics['specificity'],
            'F1 Score': self.metrics['f1_score'],
            'Accuracy': self.metrics['accuracy'],
            'NPV': self.metrics['npv'],
        }

        colors = ['green' if v >= CONFIG.MIN_PRECISION else 'orange'
                  for v in metrics_to_plot.values()]
        colors[0] = 'green' if self.metrics['precision'] >= CONFIG.MIN_PRECISION else 'red'

        bars = ax2.barh(list(metrics_to_plot.keys()), list(metrics_to_plot.values()),
                        color=colors, alpha=0.7, edgecolor='black')

        for bar, (_, value) in zip(bars, metrics_to_plot.items()):
            ax2.text(value + 0.02, bar.get_y() + bar.get_height() / 2,
                     f'{value:.2%}', va='center', fontsize=11, fontweight='bold')

        ax2.axvline(x=CONFIG.MIN_PRECISION, color='red', linestyle='--', linewidth=2,
                    label=f'Min Precision Target ({CONFIG.MIN_PRECISION:.0%})')
        ax2.set_xlim([0, 1.1])
        ax2.set_xlabel('Score', fontsize=12)
        ax2.set_title('Classification Metrics', fontsize=14)
        ax2.legend(loc='lower right')
        ax2.grid(True, alpha=0.3, axis='x')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Confusion matrix saved to {save_path}")
        plt.close()
        return fig

    def print_report(self):
        print(f"\n  {'='*50}")
        print(f"  CONFUSION MATRIX ANALYSIS")
        print(f"  {'='*50}")
        print(f"\n  Matrix:")
        print(f"                    Predicted")
        print(f"                 Neg      Pos")
        print(f"  Actual Neg   {self.tn:6d}   {self.fp:6d}")
        print(f"  Actual Pos   {self.fn:6d}   {self.tp:6d}")

        print(f"\n  Key Metrics:")
        print(f"  {'─'*40}")
        print(f"  Precision:          {self.metrics['precision']:.4f} ({self.metrics['precision']:.2%})")
        print(f"  Recall:             {self.metrics['recall']:.4f} ({self.metrics['recall']:.2%})")
        print(f"  F1 Score:           {self.metrics['f1_score']:.4f}")
        print(f"  Specificity:        {self.metrics['specificity']:.4f}")
        print(f"  False Positive Rate:{self.metrics['false_positive_rate']:.4f}")

        print(f"\n  Trading Impact:")
        print(f"  {'─'*40}")
        print(f"  Total Trades Taken: {self.metrics['total_positive_predictions']}")
        print(f"  Winning Trades (TP):{self.tp}")
        print(f"  Losing Trades (FP): {self.fp}")
        print(f"  Missed Opportunities (FN): {self.fn}")

        if self.metrics['precision'] >= CONFIG.MIN_PRECISION:
            print(f"\n  ✅ PRECISION TARGET MET: {self.metrics['precision']:.2%} >= {CONFIG.MIN_PRECISION:.0%}")
        else:
            print(f"\n  ❌ PRECISION TARGET NOT MET: {self.metrics['precision']:.2%} < {CONFIG.MIN_PRECISION:.0%}")


# --------------------------------------------------------------------
# GPU-Native Preprocessing
# --------------------------------------------------------------------
class GPUPreprocessor:
    """GPU-accelerated rolling z-score normalization."""
    def __init__(self, rolling_window: int = 100):
        self.rolling_window = rolling_window

    def fit_transform(self, X: torch.Tensor) -> torch.Tensor:
        X = to_torch(X)
        n_samples, n_features = X.shape
        X_normalized = torch.zeros_like(X)

        for i in range(n_features):
            col = X[:, i]
            cumsum = torch.cumsum(col, dim=0)
            cumsum_sq = torch.cumsum(col ** 2, dim=0)

            rolling_sum = cumsum.clone()
            rolling_sum[self.rolling_window:] = cumsum[self.rolling_window:] - cumsum[:-self.rolling_window]
            counts = torch.clamp(torch.arange(1, n_samples + 1, device=CONFIG.DEVICE),
                                 max=self.rolling_window).float()
            rolling_mean = rolling_sum / counts

            rolling_sum_sq = cumsum_sq.clone()
            rolling_sum_sq[self.rolling_window:] = cumsum_sq[self.rolling_window:] - cumsum_sq[:-self.rolling_window]
            rolling_var = (rolling_sum_sq / counts) - (rolling_mean ** 2)
            rolling_std = torch.sqrt(torch.clamp(rolling_var, min=1e-8))

            X_normalized[:, i] = (col - rolling_mean) / rolling_std

        X_normalized = torch.clamp(X_normalized, -3, 3)
        X_normalized = torch.nan_to_num(X_normalized, nan=0.0, posinf=3.0, neginf=-3.0)
        return X_normalized

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        return self.fit_transform(X)


# --------------------------------------------------------------------
# Elastic Net Regularization
# --------------------------------------------------------------------
class ElasticNetRegularizer:
    def __init__(self, l1_lambda: float = 0.001, l2_lambda: float = 0.01):
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda

    def __call__(self, model: nn.Module) -> torch.Tensor:
        l1_penalty = torch.tensor(0.0, device=CONFIG.DEVICE)
        l2_penalty = torch.tensor(0.0, device=CONFIG.DEVICE)

        for name, param in model.named_parameters():
            if 'weight' in name:
                l1_penalty += torch.sum(torch.abs(param))
                l2_penalty += torch.sum(param ** 2)

        return self.l1_lambda * l1_penalty + self.l2_lambda * l2_penalty


# --------------------------------------------------------------------
# Neural Network with Dropout
# --------------------------------------------------------------------
class SpatialDropout1D(nn.Module):
    def __init__(self, p: float = 0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = torch.bernoulli(torch.ones(x.shape[1], device=x.device) * (1 - self.p))
        mask = mask.unsqueeze(0).expand_as(x)
        return x * mask / (1 - self.p)


class RegularizedMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int] = [256, 128, 64],
        dropout_rates: List[float] = [0.5, 0.4, 0.3],
        use_batch_norm: bool = True
    ):
        super().__init__()
        layers = []
        prev_dim = input_dim

        if len(dropout_rates) < len(hidden_dims):
            dropout_rates = dropout_rates + [dropout_rates[-1]] * (len(hidden_dims) - len(dropout_rates))

        for hidden_dim, drop_rate in zip(hidden_dims, dropout_rates):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(SpatialDropout1D(drop_rate))
            prev_dim = hidden_dim

        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, 1)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.hidden_layers(x)
        out = self.output_layer(out)
        return out.squeeze(-1)


# --------------------------------------------------------------------
# Balanced Loss Function
# --------------------------------------------------------------------
class BalancedLoss(nn.Module):
    """Balanced BCE Loss with pos_weight + optional FP penalty."""
    def __init__(self, pos_weight: float = 1.0, fp_weight: float = 1.0, label_smoothing: float = 0.1):
        super().__init__()
        self.pos_weight = pos_weight
        self.fp_weight = fp_weight
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.label_smoothing > 0:
            smoothed_targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            smoothed_targets = targets

        pos_weight_tensor = torch.tensor([self.pos_weight], device=logits.device)

        bce_loss = F.binary_cross_entropy_with_logits(
            logits,
            smoothed_targets,
            pos_weight=pos_weight_tensor,
            reduction='none'
        )

        if self.fp_weight > 1.0:
            probs = torch.sigmoid(logits)
            fp_mask = (probs > 0.5) & (targets < 0.5)
            weights = torch.ones_like(bce_loss)
            weights[fp_mask] = self.fp_weight
            bce_loss = bce_loss * weights

        return bce_loss.mean()


# --------------------------------------------------------------------
# Mini-Batch Trainer with Balanced Loss
# --------------------------------------------------------------------
class MiniBatchTrainer:
    def __init__(
        self,
        model: nn.Module,
        l1_lambda: float = 0.001,
        l2_lambda: float = 0.01,
        lr: float = 0.001,
        batch_size: int = 2048,
        epochs: int = 100,
        patience: int = 15,
        pos_weight: float = 1.0,
        fp_weight: float = 2.0,
        label_smoothing: float = 0.1,
        gradient_clip: float = 1.0,
        device: str = "cuda"
    ):
        self.model = model.to(device)
        self.device = device
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.gradient_clip = gradient_clip

        self.elastic_net = ElasticNetRegularizer(l1_lambda, l2_lambda)
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='max', factor=0.5, patience=5)

        self.criterion = BalancedLoss(
            pos_weight=pos_weight,
            fp_weight=fp_weight,
            label_smoothing=label_smoothing
        )

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_total_loss': [],
            'val_precision': [],
            'val_recall': []
        }
        self.best_val_precision = 0.0
        self.best_model_state = None
        self.epochs_without_improvement = 0

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        X_train = to_torch(X_train)
        y_train = to_torch(y_train)

        if X_val is not None:
            X_val = to_torch(X_val)
            y_val = to_torch(y_val)

        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False,
            drop_last=True
        )

        for epoch in range(self.epochs):
            self.model.train()
            epoch_raw_loss = 0.0
            epoch_total_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(batch_X)

                raw_loss = self.criterion(outputs, batch_y)
                reg_loss = self.elastic_net(self.model)
                total_loss = raw_loss + reg_loss

                total_loss.backward()
                if self.gradient_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
                self.optimizer.step()

                epoch_raw_loss += raw_loss.item()
                epoch_total_loss += total_loss.item()

            epoch_raw_loss /= len(train_loader)
            epoch_total_loss /= len(train_loader)

            self.history['train_loss'].append(epoch_raw_loss)
            self.history['train_total_loss'].append(epoch_total_loss)

            if X_val is not None:
                self.model.eval()
                with torch.no_grad():
                    val_outputs = self.model(X_val)
                    val_probs = torch.sigmoid(val_outputs)
                    val_raw_loss = float(self.criterion(val_outputs, y_val).item())

                    threshold = 0.5
                    val_preds = (val_probs > threshold).float()

                    tp = ((val_preds == 1) & (y_val == 1)).sum().item()
                    fp = ((val_preds == 1) & (y_val == 0)).sum().item()
                    fn = ((val_preds == 0) & (y_val == 1)).sum().item()

                    val_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                    val_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

                self.history['val_loss'].append(val_raw_loss)
                self.history['val_precision'].append(val_precision)
                self.history['val_recall'].append(val_recall)

                self.scheduler.step(val_precision)

                if val_precision > self.best_val_precision:
                    self.best_val_precision = val_precision
                    self.best_model_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                    self.epochs_without_improvement = 0
                else:
                    self.epochs_without_improvement += 1

                if self.epochs_without_improvement >= self.patience:
                    print(f"    Early stopping at epoch {epoch + 1}")
                    break

                if (epoch + 1) % 10 == 0:
                    print(
                        f"    Epoch {epoch + 1}/{self.epochs} | "
                        f"Raw Loss: {epoch_raw_loss:.4f} | Val Loss: {val_raw_loss:.4f} | "
                        f"Precision: {val_precision:.4f} | Recall: {val_recall:.4f}"
                    )

        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            self.model.to(self.device)

        return self


# --------------------------------------------------------------------
# Diverse NN Ensemble with Agreement Filter
# --------------------------------------------------------------------
class DiverseNNEnsemble:
    def __init__(self, input_dim: int, n_models: int = 5, use_diversity: bool = True,
                 base_config: Dict = None, class_weights: Dict = None):
        self.input_dim = input_dim
        self.n_models = n_models
        self.use_diversity = use_diversity
        self.models = []
        self.trainers = []

        self.class_weights = class_weights or {'pos_weight': 1.0}

        self.base_config = base_config or {
            'hidden_dims': [256, 128, 64],
            'dropout_rates': [0.5, 0.4, 0.3],
            'l1_lambda': 0.001,
            'l2_lambda': 0.01,
            'lr': 0.001,
            'epochs': 100,
            'patience': 15,
            'batch_size': 2048,
            'pos_weight': self.class_weights['pos_weight'],
            'fp_weight': 2.0
        }

        self.configs = self._generate_diverse_configs()

    def _generate_diverse_configs(self) -> List[Dict]:
        configs = []
        architectures = [[256, 128, 64], [512, 256, 128], [128, 64, 32], [256, 128, 64, 32], [384, 192, 96]]
        dropout_configs = [[0.5, 0.4, 0.3], [0.6, 0.5, 0.4], [0.4, 0.3, 0.2], [0.5, 0.5, 0.5], [0.3, 0.3, 0.3]]
        elastic_configs = [(0.001, 0.01), (0.005, 0.005), (0.01, 0.001), (0.0001, 0.05), (0.002, 0.008)]
        fp_weights = [2.0, 2.5, 3.0, 1.5, 2.0]

        for i in range(self.n_models):
            config = self.base_config.copy()
            if self.use_diversity:
                config['hidden_dims'] = architectures[i % len(architectures)]
                config['dropout_rates'] = dropout_configs[i % len(dropout_configs)]
                config['l1_lambda'], config['l2_lambda'] = elastic_configs[i % len(elastic_configs)]
                config['fp_weight'] = fp_weights[i % len(fp_weights)]
            config['pos_weight'] = self.class_weights['pos_weight']
            config['model_id'] = i
            configs.append(config)

        return configs

    def fit(self, X_train, y_train, X_val=None, y_val=None, use_bagging: bool = True):
        print(f"\n{'='*60}")
        print(f"TRAINING DIVERSE NN ENSEMBLE ({self.n_models} models)")
        print(f"Using pos_weight={self.class_weights['pos_weight']:.3f} (auto-balanced)")
        print(f"{'='*60}")

        X_train = to_torch(X_train)
        y_train = to_torch(y_train)
        if X_val is not None:
            X_val = to_torch(X_val)
            y_val = to_torch(y_val)

        for i, config in enumerate(self.configs):
            print(f"\n  Model {i + 1}/{self.n_models}:")
            print(f"    Architecture: {config['hidden_dims']}, pos_weight: {config['pos_weight']:.2f}, FP Weight: {config['fp_weight']}")

            model = RegularizedMLP(
                input_dim=self.input_dim,
                hidden_dims=config['hidden_dims'],
                dropout_rates=config['dropout_rates'],
                use_batch_norm=True
            )

            if use_bagging:
                n_samples = len(X_train)
                indices = torch.randint(0, n_samples, (int(n_samples * 0.8),), device=CONFIG.DEVICE)
                X_bag, y_bag = X_train[indices], y_train[indices]
            else:
                X_bag, y_bag = X_train, y_train

            trainer = MiniBatchTrainer(
                model=model,
                l1_lambda=config['l1_lambda'],
                l2_lambda=config['l2_lambda'],
                lr=config['lr'],
                batch_size=config['batch_size'],
                epochs=config['epochs'],
                patience=config['patience'],
                pos_weight=config['pos_weight'],
                fp_weight=config['fp_weight'],
                label_smoothing=CONFIG.NN_LABEL_SMOOTHING,
                gradient_clip=CONFIG.NN_GRADIENT_CLIP,
                device=CONFIG.DEVICE
            )

            trainer.fit(X_bag, y_bag, X_val, y_val)
            self.models.append(model)
            self.trainers.append(trainer)

        return self

    def predict_proba(self, X):
        X = to_torch(X)
        all_probs = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
            all_probs.append(probs)

        ensemble_probs = torch.stack(all_probs).mean(dim=0)
        ensemble_probs = to_numpy(ensemble_probs)

        return np.column_stack([1 - ensemble_probs, ensemble_probs])

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)[:, 1]
        return (probs > threshold).astype(int)

    def predict_with_agreement(
        self,
        X,
        prob_threshold: float = 0.5,
        agreement_threshold: float = 0.6,
        return_details: bool = False
    ):
        X = to_torch(X)
        all_probs = []
        all_preds = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
                preds = (probs > prob_threshold).float()
            all_probs.append(probs)
            all_preds.append(preds)

        stacked_preds = torch.stack(all_preds)
        stacked_probs = torch.stack(all_probs)

        agreement_ratio = stacked_preds.mean(dim=0)
        final_pred = (agreement_ratio >= agreement_threshold).int()
        confidence = agreement_ratio

        final_pred_np = to_numpy(final_pred)
        confidence_np = to_numpy(confidence)

        if return_details:
            return final_pred_np, confidence_np, to_numpy(stacked_preds), to_numpy(stacked_probs)

        return final_pred_np, confidence_np

    def analyze_agreement(self, X, y_true, prob_threshold: float = 0.5) -> Dict:
        X = to_torch(X)
        y_true = to_numpy(y_true) if not isinstance(y_true, np.ndarray) else y_true

        all_preds = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
                preds = (probs > prob_threshold).float()
            all_preds.append(preds)

        stacked = torch.stack(all_preds)
        agreement_ratio = to_numpy(stacked.mean(dim=0))

        results = {
            'agreement_levels': [],
            'precision': [],
            'recall': [],
            'f1_score': [],
            'n_trades': [],
            'n_samples': len(y_true)
        }

        for agreement_thresh in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
            pred = (agreement_ratio >= agreement_thresh).astype(int)

            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            fn = ((pred == 0) & (y_true == 1)).sum()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            n_trades = int(pred.sum())

            results['agreement_levels'].append(agreement_thresh)
            results['precision'].append(float(precision))
            results['recall'].append(float(recall))
            results['f1_score'].append(float(f1))
            results['n_trades'].append(n_trades)

        return results

    def find_optimal_agreement_threshold(self, X, y_true, min_precision: float = 0.54, prob_threshold: float = 0.5):
        analysis = self.analyze_agreement(X, y_true, prob_threshold)
        optimal_thresh = 1.0
        optimal_metrics = {'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0, 'n_trades': 0, 'trade_rate': 0.0}

        for thresh, prec, rec, f1, n_trades in zip(
            analysis['agreement_levels'],
            analysis['precision'],
            analysis['recall'],
            analysis['f1_score'],
            analysis['n_trades']
        ):
            if prec >= min_precision and n_trades > 0:
                if thresh < optimal_thresh:
                    optimal_thresh = float(thresh)
                    optimal_metrics = {
                        'precision': float(prec),
                        'recall': float(rec),
                        'f1_score': float(f1),
                        'n_trades': int(n_trades),
                        'trade_rate': float(n_trades / analysis['n_samples'])
                    }

        return optimal_thresh, optimal_metrics

    def plot_agreement_analysis(self, X, y_true, save_path: str = None):
        analysis = self.analyze_agreement(X, y_true)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        ax1 = axes[0]
        ax1.plot(analysis['agreement_levels'], analysis['precision'], 'b-o', linewidth=2, markersize=8, label='Precision')
        ax1.plot(analysis['agreement_levels'], analysis['recall'], 'r-o', linewidth=2, markersize=8, label='Recall')
        ax1.axhline(CONFIG.MIN_PRECISION, color='green', linestyle='--', label=f'Target Precision ({CONFIG.MIN_PRECISION:.0%})')
        ax1.set_xlabel('Agreement Threshold', fontsize=12)
        ax1.set_ylabel('Score', fontsize=12)
        ax1.set_title('Precision & Recall vs Agreement Threshold', fontsize=14)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim([0.35, 1.05])
        ax1.set_ylim([0, 1])

        ax2 = axes[1]
        ax2.bar(analysis['agreement_levels'], analysis['n_trades'], width=0.08, color='steelblue', edgecolor='black', alpha=0.7)
        ax2.set_xlabel('Agreement Threshold', fontsize=12)
        ax2.set_ylabel('Number of Trades', fontsize=12)
        ax2.set_title('Trade Count vs Agreement Threshold', fontsize=14)
        ax2.grid(True, alpha=0.3, axis='y')

        ax3 = axes[2]
        colors = plt.cm.viridis(np.linspace(0, 1, len(analysis['agreement_levels'])))
        for i, (thresh, prec, rec) in enumerate(zip(analysis['agreement_levels'], analysis['precision'], analysis['recall'])):
            ax3.scatter(rec, prec, c=[colors[i]], s=150, label=f'Agree={thresh:.0%}', edgecolors='black', zorder=5)
        ax3.axhline(CONFIG.MIN_PRECISION, color='green', linestyle='--', alpha=0.7)
        ax3.set_xlabel('Recall', fontsize=12)
        ax3.set_ylabel('Precision', fontsize=12)
        ax3.set_title('Precision-Recall at Different Agreement Levels', fontsize=14)
        ax3.legend(loc='best', fontsize=9)
        ax3.grid(True, alpha=0.3)
        ax3.set_xlim([0, 1])
        ax3.set_ylim([0, 1])

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Agreement analysis saved to {save_path}")
        plt.close()
        return fig

    def plot_history(self, save_dir: str):
        if not self.trainers:
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        ax1 = axes[0]
        min_len = min([len(t.history['train_loss']) for t in self.trainers])

        train_losses = [t.history['train_loss'][:min_len] for t in self.trainers]
        val_losses = [t.history['val_loss'][:min_len] for t in self.trainers]

        for t in self.trainers:
            ax1.plot(t.history['train_loss'], color='blue', alpha=0.15, linewidth=1)
            ax1.plot(t.history['val_loss'], color='red', alpha=0.15, linewidth=1)

        avg_train = np.mean(train_losses, axis=0)
        avg_val = np.mean(val_losses, axis=0)

        ax1.plot(avg_train, color='blue', linewidth=2.5, label='Avg Train Loss (Raw)')
        ax1.plot(avg_val, color='red', linewidth=2.5, label='Avg Val Loss (Raw)')
        ax1.set_title(f'Ensemble Loss - Raw Only ({len(self.trainers)} Models)')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Raw Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2 = axes[1]
        val_precisions = [t.history['val_precision'][:min_len] for t in self.trainers]
        val_recalls = [t.history['val_recall'][:min_len] for t in self.trainers]
        avg_precision = np.mean(val_precisions, axis=0)
        avg_recall = np.mean(val_recalls, axis=0)

        for t in self.trainers:
            ax2.plot(t.history['val_precision'], color='green', alpha=0.15)
            ax2.plot(t.history['val_recall'], color='orange', alpha=0.15)

        ax2.plot(avg_precision, color='green', linewidth=2.5, label='Avg Val Precision')
        ax2.plot(avg_recall, color='orange', linewidth=2.5, label='Avg Val Recall')
        ax2.axhline(CONFIG.MIN_PRECISION, color='black', linestyle='--', label='Target Precision')

        ax2.set_title('Validation Precision & Recall Evolution')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Score')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = f"{save_dir}/ensemble_learning_curves.png"
        plt.savefig(save_path, dpi=150)
        plt.close()
        print(f"  📊 Learning curves saved to {save_path}")


# --------------------------------------------------------------------
# Optuna XGBoost (Auto-Balanced)
# --------------------------------------------------------------------
class OptunaXGBOptimizer:
    def __init__(self, config: Config, class_weights: Dict = None):
        self.config = config
        self.class_weights = class_weights or {'scale_pos_weight': 1.0}
        self.best_model = None
        self.best_params = None
        self.study = None

    def objective(self, trial, X_train, y_train, X_val, y_val):
        base_spw = self.class_weights['scale_pos_weight']
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'max_bin': 256,
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_child_weight': trial.suggest_int('min_child_weight', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 0.8),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 50.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 100.0, log=True),
            'gamma': trial.suggest_float('gamma', 0.1, 5.0),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', base_spw * 0.5, base_spw * 2.0),
            'random_state': self.config.RANDOM_STATE,
            'early_stopping_rounds': self.config.EARLY_STOPPING,
        }

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

        valid_mask = precisions[:-1] >= self.config.MIN_PRECISION
        if valid_mask.any():
            valid_indices = np.where(valid_mask)[0]
            best_idx = valid_indices[np.argmax(recalls[:-1][valid_indices])]
            achieved_recall = recalls[best_idx]
            return float(achieved_recall)
        return -1.0

    def optimize(self, X_train, y_train, X_val, y_val, n_trials=50):
        print(f"\n{'='*60}")
        print(f"OPTUNA XGBOOST OPTIMIZATION ({n_trials} trials)")
        print(f"Target: Maximize recall while precision >= {CONFIG.MIN_PRECISION:.0%}")
        print(f"Base scale_pos_weight: {self.class_weights['scale_pos_weight']:.3f} (auto-balanced)")
        print(f"{'='*60}")

        self.study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=self.config.RANDOM_STATE))
        self.study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
                            n_trials=n_trials, show_progress_bar=True, gc_after_trial=True)

        self.best_params = self.study.best_params
        print(f"\n  ✓ Best Recall (at precision >= {CONFIG.MIN_PRECISION:.0%}): {self.study.best_value:.4f}")
        print(f"  ✓ Tuned scale_pos_weight: {self.best_params.get('scale_pos_weight', 'N/A'):.3f}")

        final_params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'max_bin': 256,
            'random_state': self.config.RANDOM_STATE,
            'early_stopping_rounds': self.config.EARLY_STOPPING,
            **self.best_params
        }

        self.best_model = xgb.XGBClassifier(**final_params)
        self.best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        return self.best_model, self.best_params


# --------------------------------------------------------------------
# NEW: Optuna AdaBoost (Precision-constrained Recall Maximization)
# --------------------------------------------------------------------
class OptunaAdaBoostOptimizer:
    def __init__(self, config: Config, class_weights: Dict):
        self.config = config
        self.class_weights = class_weights
        self.best_model = None
        self.best_params = None
        self.study = None

    def objective(self, trial, X_train, y_train, X_val, y_val, sw_train):
        # Decision stump / shallow tree is typical for AdaBoost
        max_depth = trial.suggest_int("max_depth", 1, 3)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 50)
        n_estimators = trial.suggest_int("n_estimators", 50, 600)
        learning_rate = trial.suggest_float("learning_rate", 0.01, 1.0, log=True)

        base = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=self.config.RANDOM_STATE
        )

        model = AdaBoostClassifier(
            estimator=base,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=self.config.RANDOM_STATE
        )

        model.fit(X_train, y_train, sample_weight=sw_train)

        y_proba = model.predict_proba(X_val)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

        valid_mask = precisions[:-1] >= self.config.MIN_PRECISION
        if valid_mask.any():
            valid_indices = np.where(valid_mask)[0]
            best_idx = valid_indices[np.argmax(recalls[:-1][valid_indices])]
            return float(recalls[best_idx])  # maximize recall under precision constraint

        return -1.0

    def optimize(self, X_train, y_train, X_val, y_val, n_trials=30):
        print(f"\n{'='*60}")
        print(f"OPTUNA ADABOOST OPTIMIZATION ({n_trials} trials)")
        print(f"Target: Maximize recall while precision >= {CONFIG.MIN_PRECISION:.0%}")
        print(f"Using sample_weight with pos_weight={self.class_weights['pos_weight']:.3f}")
        print(f"{'='*60}")

        sw_train = make_sample_weight(y_train, self.class_weights['pos_weight'])

        self.study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=self.config.RANDOM_STATE))
        self.study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val, sw_train),
                            n_trials=n_trials, show_progress_bar=True, gc_after_trial=True)

        self.best_params = self.study.best_params
        print(f"\n  ✓ Best Recall (at precision >= {CONFIG.MIN_PRECISION:.0%}): {self.study.best_value:.4f}")
        print(f"  ✓ Best Params: {self.best_params}")

        # Train final model with best params
        base = DecisionTreeClassifier(
            max_depth=self.best_params["max_depth"],
            min_samples_leaf=self.best_params["min_samples_leaf"],
            random_state=self.config.RANDOM_STATE
        )
        self.best_model = AdaBoostClassifier(
            estimator=base,
            n_estimators=self.best_params["n_estimators"],
            learning_rate=self.best_params["learning_rate"],
            random_state=self.config.RANDOM_STATE
        )
        self.best_model.fit(X_train, y_train, sample_weight=sw_train)

        return self.best_model, self.best_params


# --------------------------------------------------------------------
# Data Loading
# --------------------------------------------------------------------
def load_data(config: Config):
    print(f"\n{'='*60}")
    print("LOADING DATA")
    print(f"{'='*60}")

    if not os.path.exists(config.INPUT_FILE):
        raise FileNotFoundError(f"File not found: {config.INPUT_FILE}")

    df = pd.read_csv(config.INPUT_FILE)

    for c in df.columns:
        if c.lower() in ['date', 'datetime', 'timestamp', 'time']:
            df[c] = pd.to_datetime(df[c])
            df.set_index(c, inplace=True)
            df.sort_index(inplace=True)
            break

    prices = df[['open', 'close']].copy()

    future_returns = df['close'].pct_change(periods=config.TARGET_HORIZON).shift(-config.TARGET_HORIZON)
    y = (future_returns > config.THRESHOLD).astype(int)

    valid = ~(df.isna().any(axis=1) | y.isna())
    X = df.loc[valid].copy()
    y = y.loc[valid].copy()
    prices = prices.loc[valid].copy()

    drop_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'log_returns', 'gap_flag', 'target']
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')
    X = X.astype(np.float32)

    print(f"  Samples: {len(X):,}, Features: {X.shape[1]}")
    print(f"  Class Balance: {y.mean():.2%} positive")

    return X, y, prices


def time_series_split(X, y, prices, test_size=0.2, val_size=0.1):
    n = len(X)
    test_idx = int(n * (1 - test_size))
    val_idx = int(test_idx * (1 - val_size))

    return (
        X.iloc[:val_idx], X.iloc[val_idx:test_idx], X.iloc[test_idx:],
        y.iloc[:val_idx], y.iloc[val_idx:test_idx], y.iloc[test_idx:],
        prices.iloc[test_idx:]
    )


# --------------------------------------------------------------------
# Full Ensemble with Agreement (XGB + NN + AdaBoost)
# --------------------------------------------------------------------
class FullGPUEnsemble:
    def __init__(self, xgb_model, nn_ensemble, preprocessor,
                 ada_model: Optional[AdaBoostClassifier] = None,
                 weights: List[float] = [0.34, 0.33, 0.33]):
        self.xgb_model = xgb_model
        self.nn_ensemble = nn_ensemble
        self.preprocessor = preprocessor
        self.ada_model = ada_model

        # Normalize weights robustly
        w = np.array(weights, dtype=np.float32)
        w = w / (w.sum() + 1e-12)
        self.weights = w.tolist()

        # Agreement settings
        self.use_agreement = CONFIG.USE_AGREEMENT
        self.agreement_threshold = CONFIG.DEFAULT_AGREEMENT_THRESHOLD
        self.prob_threshold = 0.5

        self.optimal_config = None

    def predict_proba(self, X):
        """Weighted average probability across XGB + NN + AdaBoost (if enabled)."""
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X

        # XGB proba
        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]

        # NN proba (uses preprocessor)
        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)
        nn_proba = self.nn_ensemble.predict_proba(X_norm)[:, 1]

        # Ada proba (optional)
        if self.ada_model is not None:
            ada_proba = self.ada_model.predict_proba(X_np)[:, 1]
        else:
            ada_proba = 0.0

        if self.ada_model is None:
            # fall back to 2-model weights if ada not provided
            ensemble_proba = 0.5 * xgb_proba + 0.5 * nn_proba
        else:
            ensemble_proba = self.weights[0] * xgb_proba + self.weights[1] * nn_proba + self.weights[2] * ada_proba

        return np.column_stack([1 - ensemble_proba, ensemble_proba])

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)[:, 1]
        return (probs > threshold).astype(int)

    def predict_with_agreement(
        self,
        X,
        xgb_threshold: float = 0.5,
        ada_threshold: float = 0.5,
        nn_prob_threshold: float = 0.5,
        nn_agreement_threshold: float = 0.6,
        require_xgb_agree: bool = True,
        require_ada_agree: bool = True
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Combined prediction with agreement filter:

        - NN ensemble: requires >= nn_agreement_threshold models predict positive at nn_prob_threshold
        - XGB: thresholded at xgb_threshold
        - AdaBoost: thresholded at ada_threshold (if enabled)
        - Final_pred:
            if require_xgb_agree: must include XGB positive
            if require_ada_agree: must include Ada positive (only if ada_model exists)
            always includes NN agreement positive
        """
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X

        # XGB prediction
        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]
        xgb_pred = (xgb_proba > xgb_threshold).astype(int)

        # Ada prediction (if present)
        if self.ada_model is not None:
            ada_proba = self.ada_model.predict_proba(X_np)[:, 1]
            ada_pred = (ada_proba > ada_threshold).astype(int)
        else:
            ada_proba = np.zeros_like(xgb_proba)
            ada_pred = np.ones_like(xgb_pred)  # neutral if not used

        # NN with agreement
        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)
        nn_pred, nn_agreement = self.nn_ensemble.predict_with_agreement(
            X_norm,
            prob_threshold=nn_prob_threshold,
            agreement_threshold=nn_agreement_threshold
        )

        # Combine decisions
        final_pred = (nn_pred == 1)

        if require_xgb_agree:
            final_pred = final_pred & (xgb_pred == 1)

        if self.ada_model is not None and require_ada_agree:
            final_pred = final_pred & (ada_pred == 1)

        final_pred = final_pred.astype(int)

        # Confidence: weighted combination (probabilities + agreement)
        if self.ada_model is None:
            confidence = 0.5 * xgb_proba + 0.5 * nn_agreement
        else:
            confidence = (
                self.weights[0] * xgb_proba +
                self.weights[1] * nn_agreement +
                self.weights[2] * ada_proba
            )

        return final_pred, confidence

    def find_optimal_thresholds(self, X, y_true, min_precision: float = 0.54) -> Dict:
        """
        Grid search optimal thresholds for combined ensemble agreement.
        Now includes AdaBoost threshold if enabled.
        """
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X
        y_true = to_numpy(y_true) if not isinstance(y_true, np.ndarray) else y_true

        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]

        if self.ada_model is not None:
            ada_proba = self.ada_model.predict_proba(X_np)[:, 1]
            ada_threshold_grid = [0.4, 0.45, 0.5, 0.55, 0.6]
            require_ada_grid = [True, False]
        else:
            ada_proba = None
            ada_threshold_grid = [0.5]
            require_ada_grid = [False]

        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)

        best_result = {
            'precision': 0.0,
            'recall': 0.0,
            'f1_score': 0.0,
            'n_trades': 0,
            'xgb_threshold': 0.5,
            'ada_threshold': 0.5,
            'nn_agreement': 0.6,
            'require_xgb': True,
            'require_ada': False
        }

        print(f"\n  🔍 Grid searching optimal thresholds (XGB + NN + Ada)...")

        for xgb_thresh in [0.4, 0.45, 0.5, 0.55, 0.6]:
            xgb_pred = (xgb_proba > xgb_thresh).astype(int)

            for ada_thresh in ada_threshold_grid:
                if self.ada_model is not None:
                    ada_pred = (ada_proba > ada_thresh).astype(int)
                else:
                    ada_pred = np.ones_like(xgb_pred)

                for nn_agree in [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
                    nn_pred, _ = self.nn_ensemble.predict_with_agreement(
                        X_norm, prob_threshold=0.5, agreement_threshold=nn_agree
                    )

                    for require_xgb in [True, False]:
                        for require_ada in require_ada_grid:
                            final_pred = (nn_pred == 1)

                            if require_xgb:
                                final_pred = final_pred & (xgb_pred == 1)

                            if self.ada_model is not None and require_ada:
                                final_pred = final_pred & (ada_pred == 1)

                            final_pred = final_pred.astype(int)

                            tp = ((final_pred == 1) & (y_true == 1)).sum()
                            fp = ((final_pred == 1) & (y_true == 0)).sum()
                            fn = ((final_pred == 0) & (y_true == 1)).sum()

                            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                            n_trades = int(final_pred.sum())

                            if precision >= min_precision and recall > best_result['recall']:
                                best_result = {
                                    'precision': float(precision),
                                    'recall': float(recall),
                                    'f1_score': float(f1),
                                    'n_trades': int(n_trades),
                                    'xgb_threshold': float(xgb_thresh),
                                    'ada_threshold': float(ada_thresh),
                                    'nn_agreement': float(nn_agree),
                                    'require_xgb': bool(require_xgb),
                                    'require_ada': bool(require_ada)
                                }

        self.optimal_config = best_result
        return best_result

    def predict_optimal(self, X) -> Tuple[np.ndarray, np.ndarray]:
        if self.optimal_config is None:
            raise ValueError("Must call find_optimal_thresholds() first!")

        return self.predict_with_agreement(
            X,
            xgb_threshold=self.optimal_config['xgb_threshold'],
            ada_threshold=self.optimal_config.get('ada_threshold', 0.5),
            nn_prob_threshold=0.5,
            nn_agreement_threshold=self.optimal_config['nn_agreement'],
            require_xgb_agree=self.optimal_config['require_xgb'],
            require_ada_agree=self.optimal_config.get('require_ada', False)
        )


# --------------------------------------------------------------------
# Evaluation with Agreement
# --------------------------------------------------------------------
def evaluate_with_agreement(model: FullGPUEnsemble, X_test, y_test, prices_test, config: Config):
    print(f"\n{'='*60}")
    print("ENSEMBLE AGREEMENT EVALUATION")
    print(f"{'='*60}")

    y_true = y_test.values if hasattr(y_test, 'values') else y_test

    # Analyze agreement on NN ensemble
    print("\n  📊 Analyzing NN Ensemble Agreement...")
    X_t = to_torch(X_test)
    X_norm = model.preprocessor.transform(X_t)

    model.nn_ensemble.plot_agreement_analysis(
        X_norm, y_true,
        save_path=f"{config.OUTPUT_DIR}/agreement_analysis.png"
    )

    opt_agree, opt_metrics = model.nn_ensemble.find_optimal_agreement_threshold(
        X_norm, y_true, min_precision=config.MIN_PRECISION
    )

    print(f"\n  🎯 Optimal NN-Only Agreement Threshold: {opt_agree:.0%}")
    print(f"     Precision: {opt_metrics.get('precision', 0):.2%}")
    print(f"     Recall: {opt_metrics.get('recall', 0):.2%}")
    print(f"     Trades: {opt_metrics.get('n_trades', 0)}")

    # Find optimal combined thresholds (XGB + NN + Ada)
    best_config = model.find_optimal_thresholds(X_test, y_true, min_precision=config.MIN_PRECISION)

    print(f"\n  ✓ Best Combined Configuration:")
    print(f"     XGBoost Threshold: {best_config['xgb_threshold']}")
    if model.ada_model is not None:
        print(f"     AdaBoost Threshold: {best_config['ada_threshold']}")
    print(f"     NN Agreement: {best_config['nn_agreement']:.0%}")
    print(f"     Require XGBoost Agree: {best_config['require_xgb']}")
    print(f"     Require AdaBoost Agree: {best_config.get('require_ada', False)}")
    print(f"     Precision: {best_config['precision']:.2%}")
    print(f"     Recall: {best_config['recall']:.2%}")
    print(f"     F1 Score: {best_config['f1_score']:.4f}")
    print(f"     Trades: {best_config['n_trades']}")

    # Final predictions using optimal config
    y_pred, confidence = model.predict_with_agreement(
        X_test,
        xgb_threshold=best_config['xgb_threshold'],
        ada_threshold=best_config.get('ada_threshold', 0.5),
        nn_prob_threshold=0.5,
        nn_agreement_threshold=best_config['nn_agreement'],
        require_xgb_agree=best_config['require_xgb'],
        require_ada_agree=best_config.get('require_ada', False)
    )

    # Confusion Matrix
    cm_analyzer = ConfusionMatrixAnalyzer()
    cm_metrics = cm_analyzer.compute(y_true, y_pred)
    cm_analyzer.print_report()
    cm_analyzer.plot(
        save_path=f"{config.OUTPUT_DIR}/confusion_matrix_agreement.png",
        title=f"Confusion Matrix (NN Agree={best_config['nn_agreement']:.0%})"
    )

    # Standard PR curve for reference (using standard averaged proba)
    probs = model.predict_proba(X_test)[:, 1]
    pr_analyzer = PrecisionRecallAnalyzer(min_precision=config.MIN_PRECISION)
    pr_analyzer.analyze(y_true, probs)
    pr_analyzer.plot_precision_recall_curve(save_path=f"{config.OUTPUT_DIR}/precision_recall_curve.png")

    # Trading Backtest
    print(f"\n  {'='*50}")
    print(f"  TRADING BACKTEST (With Agreement Filter)")
    print(f"  {'='*50}")

    signals = y_pred
    entry_price = prices_test['open'].shift(-1).values
    exit_price = prices_test['close'].shift(-config.TARGET_HORIZON).values
    raw_returns = (exit_price - entry_price) / entry_price
    raw_returns = np.nan_to_num(raw_returns, 0)

    cost = config.FIXED_COMMISSION / config.AVG_PRICE_EST + config.SLIPPAGE_BPS
    net_returns = raw_returns * signals - signals * cost

    valid_len = len(signals)
    net_returns = net_returns[:valid_len]
    market_returns = prices_test['close'].pct_change().fillna(0).values[:valid_len]

    cum_market = np.cumprod(1 + market_returns)
    cum_net = np.cumprod(1 + net_returns)

    total_trades = int(signals.sum())
    win_rate = float((net_returns[signals == 1] > 0).mean()) if total_trades > 0 else 0.0
    final_return = float(cum_net[-1] - 1)
    max_dd = float(((cum_net - np.maximum.accumulate(cum_net)) / (np.maximum.accumulate(cum_net) + 1e-12)).min())
    sharpe = float(np.mean(net_returns) / np.std(net_returns) * np.sqrt(252 * 390)) if np.std(net_returns) > 0 else 0.0

    print(f"\n  Trading Results:")
    print(f"     Total Trades: {total_trades}")
    print(f"     Trade Win Rate: {win_rate:.2%}")
    print(f"     Net Return: {final_return:.2%}")
    print(f"     Max Drawdown: {max_dd:.2%}")
    print(f"     Sharpe Ratio: {sharpe:.2f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    ax1.plot(cum_market, label='Buy & Hold', alpha=0.5, linewidth=1.5)
    ax1.plot(cum_net, label='Strategy (Agreement Filter)', linewidth=2, color='green')
    ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    ax1.set_title(f'Cumulative Returns (Precision={cm_metrics["precision"]:.1%})')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.hist(confidence, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    ax2.axvline(x=np.mean(confidence), color='red', linestyle='--', label=f'Mean: {np.mean(confidence):.3f}')
    ax2.set_title('Confidence Distribution')
    ax2.set_xlabel('Confidence Score')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    if total_trades > 0:
        trade_returns = net_returns[signals == 1] * 100
        ax3.hist(trade_returns, bins=30, alpha=0.7, color='orange', edgecolor='black')
        ax3.axvline(0, color='black', linewidth=2)
        ax3.axvline(np.mean(trade_returns), color='blue', linestyle='--', label=f'Mean: {np.mean(trade_returns):.3f}%')
        ax3.legend()
    ax3.set_title('Per-Trade Returns Distribution (%)')
    ax3.set_xlabel('Return (%)')
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    drawdown = (cum_net - np.maximum.accumulate(cum_net)) / (np.maximum.accumulate(cum_net) + 1e-12) * 100
    ax4.fill_between(range(len(drawdown)), drawdown, 0, alpha=0.7, color='red')
    ax4.set_title(f'Drawdown (Max: {max_dd:.2%})')
    ax4.set_xlabel('Time')
    ax4.set_ylabel('Drawdown (%)')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_DIR}/trading_results_agreement.png", dpi=150, bbox_inches='tight')
    plt.close()

    results = {
        'optimal_nn_agreement': float(opt_agree),
        'combined_nn_agreement': float(best_config['nn_agreement']),
        'xgb_threshold': float(best_config['xgb_threshold']),
        'ada_threshold': float(best_config.get('ada_threshold', 0.5)),
        'require_xgb_agree': bool(best_config['require_xgb']),
        'require_ada_agree': bool(best_config.get('require_ada', False)),
        'precision': float(cm_metrics['precision']),
        'recall': float(cm_metrics['recall']),
        'f1_score': float(cm_metrics['f1_score']),
        'specificity': float(cm_metrics['specificity']),
        'total_trades': int(total_trades),
        'win_rate': float(win_rate),
        'final_return': float(final_return),
        'max_drawdown': float(max_dd),
        'sharpe': float(sharpe),
        'true_positives': int(cm_metrics['true_positives']),
        'false_positives': int(cm_metrics['false_positives']),
        'false_negatives': int(cm_metrics['false_negatives']),
        'true_negatives': int(cm_metrics['true_negatives'])
    }

    print(f"\n  {'='*50}")
    if cm_metrics['precision'] >= config.MIN_PRECISION:
        print(f"  ✅ PRECISION TARGET MET: {cm_metrics['precision']:.2%} >= {config.MIN_PRECISION:.0%}")
    else:
        print(f"  ❌ PRECISION TARGET NOT MET: {cm_metrics['precision']:.2%} < {config.MIN_PRECISION:.0%}")
    print(f"  {'='*50}")

    return results


# --------------------------------------------------------------------
# Main Pipeline
# --------------------------------------------------------------------
def main():
    print("\n" + "=" * 60)
    print("GPU-NATIVE MODEL v10 - PRECISION-FOCUSED + ENSEMBLE AGREEMENT + ADABOOST")
    print(f"Target Precision: >= {CONFIG.MIN_PRECISION:.0%}")
    print(f"Agreement Filter: {CONFIG.USE_AGREEMENT}")
    print(f"AdaBoost Enabled: {CONFIG.USE_ADABOOST}")
    print("=" * 60)

    start_time = time.perf_counter()

    # 1) Load data
    X, y, prices = load_data(CONFIG)

    # 2) Split
    X_train, X_val, X_test, y_train, y_val, y_test, prices_test = time_series_split(
        X, y, prices, test_size=CONFIG.TEST_SIZE_PCT
    )
    print(f"\n  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

    # 3) Auto-balance weights (from training set)
    class_weights = calculate_class_weights(y_train.values)

    # 4) Preprocessing (GPU)
    preprocessor = GPUPreprocessor(rolling_window=100)

    X_train_t = to_torch(X_train.values)
    X_val_t = to_torch(X_val.values)
    X_test_t = to_torch(X_test.values)

    X_train_norm = preprocessor.fit_transform(X_train_t)
    X_val_norm = preprocessor.transform(X_val_t)
    X_test_norm = preprocessor.transform(X_test_t)

    # Convert normalized tensors -> numpy for tree models
    X_train_np = to_numpy(X_train_norm)
    X_val_np = to_numpy(X_val_norm)
    X_test_np = to_numpy(X_test_norm)

    y_train_np = y_train.values.astype(int)
    y_val_np = y_val.values.astype(int)

    # 5) XGBoost Optimization
    optimizer = OptunaXGBOptimizer(CONFIG, class_weights=class_weights)
    best_xgb, best_params = optimizer.optimize(
        X_train_np, y_train_np, X_val_np, y_val_np, n_trials=CONFIG.OPTUNA_TRIALS
    )

    # 6) AdaBoost Optimization (NEW)
    if CONFIG.USE_ADABOOST:
        ada_optimizer = OptunaAdaBoostOptimizer(CONFIG, class_weights=class_weights)
        best_ada, best_ada_params = ada_optimizer.optimize(
            X_train_np, y_train_np, X_val_np, y_val_np, n_trials=CONFIG.ADABOOST_TRIALS
        )
    else:
        best_ada, best_ada_params = None, None

    # 7) NN Ensemble
    nn_ensemble = DiverseNNEnsemble(
        input_dim=X_train.shape[1],
        n_models=CONFIG.N_ENSEMBLE_MODELS,
        use_diversity=CONFIG.ENSEMBLE_DIVERSITY,
        class_weights=class_weights,
        base_config={
            'hidden_dims': [256, 128, 64],
            'dropout_rates': CONFIG.NN_DROPOUT_RATES[:3],
            'l1_lambda': CONFIG.NN_L1_LAMBDA,
            'l2_lambda': CONFIG.NN_L2_LAMBDA,
            'lr': CONFIG.NN_LR,
            'epochs': CONFIG.NN_EPOCHS,
            'patience': CONFIG.NN_PATIENCE,
            'batch_size': CONFIG.BATCH_SIZE,
            'pos_weight': class_weights['pos_weight'],
            'fp_weight': 2.5
        }
    )
    nn_ensemble.fit(X_train_norm, to_torch(y_train_np), X_val_norm, to_torch(y_val_np), use_bagging=True)

    # Plot Training History
    nn_ensemble.plot_history(CONFIG.OUTPUT_DIR)

    # 8) Full Ensemble (XGB + NN + Ada)
    # If AdaBoost is enabled -> 3-model weights, else it will fall back to 2-model inside predict_proba
    full_ensemble = FullGPUEnsemble(
        best_xgb,
        nn_ensemble,
        preprocessor,
        ada_model=best_ada,
        weights=[0.34, 0.33, 0.33] if best_ada is not None else [0.5, 0.5, 0.0]
    )

    # 9) Evaluation with Agreement Filter
    results = evaluate_with_agreement(full_ensemble, X_test_np, y_test, prices_test, CONFIG)

    # 10) Save Summary
    elapsed = time.perf_counter() - start_time

    with open(f"{CONFIG.OUTPUT_DIR}/summary.txt", 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("PRECISION-FOCUSED MODEL SUMMARY (ENSEMBLE AGREEMENT + ADABOOST)\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Target Precision: >= {CONFIG.MIN_PRECISION:.0%}\n")
        f.write(f"Training Time: {elapsed:.1f}s\n\n")

        f.write("Class Weights (Auto-Calculated):\n")
        f.write(f"  pos_weight (NN / Ada sample_weight): {class_weights['pos_weight']:.3f}\n")
        f.write(f"  scale_pos_weight (XGB): {class_weights['scale_pos_weight']:.3f}\n\n")

        f.write("Models:\n")
        f.write("  - XGBoost: enabled\n")
        f.write("  - NN Ensemble: enabled\n")
        f.write(f"  - AdaBoost: {'enabled' if best_ada is not None else 'disabled'}\n\n")

        if best_ada_params is not None:
            f.write("AdaBoost best params:\n")
            for k, v in best_ada_params.items():
                f.write(f"  {k}: {v}\n")
            f.write("\n")

        f.write("Optimal Configuration:\n")
        f.write(f"  NN Agreement Threshold: {results['combined_nn_agreement']:.0%}\n")
        f.write(f"  XGBoost Threshold: {results['xgb_threshold']}\n")
        f.write(f"  AdaBoost Threshold: {results['ada_threshold']}\n")
        f.write(f"  Require XGBoost Agree: {results['require_xgb_agree']}\n")
        f.write(f"  Require AdaBoost Agree: {results['require_ada_agree']}\n\n")

        f.write("Results:\n")
        for k, v in results.items():
            if isinstance(v, float):
                f.write(f"  {k}: {v:.6f}\n")
            else:
                f.write(f"  {k}: {v}\n")

    print(f"\n{'='*60}")
    print("COMPLETE")
    print(f"{'='*60}")
    print(f"  ⏱ Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    print(f"  📁 Output: {CONFIG.OUTPUT_DIR}")
    print_gpu_memory()


if __name__ == "__main__":
    main()


✓ Device: cuda
✓ Target Precision: >= 51%
✓ Ensemble Agreement: True (threshold: 60%)
✓ AdaBoost Enabled: True
  GPU: Tesla T4

GPU-NATIVE MODEL v10 - PRECISION-FOCUSED + ENSEMBLE AGREEMENT + ADABOOST
Target Precision: >= 51%
Agreement Filter: True
AdaBoost Enabled: True

LOADING DATA


[I 2026-02-02 07:24:03,410] A new study created in memory with name: no-name-ab6f0ec0-cb08-4a1f-b2df-ae76e4cab562


  Samples: 179,291, Features: 54
  Class Balance: 9.58% positive

  Train: 129,088 | Val: 14,344 | Test: 35,859

  ⚖️ Auto-Balanced Class Weights:
     Negative samples: 117,943 (91.4%)
     Positive samples: 11,145 (8.6%)
     pos_weight (NN): 10.583
     scale_pos_weight (XGB): 10.583

OPTUNA XGBOOST OPTIMIZATION (50 trials)
Target: Maximize recall while precision >= 51%
Base scale_pos_weight: 10.583 (auto-balanced)


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-02 07:24:04,945] Trial 0 finished with value: -1.0 and parameters: {'n_estimators': 437, 'learning_rate': 0.0862735828664018, 'max_depth': 5, 'min_child_weight': 64, 'subsample': 0.5468055921327309, 'colsample_bytree': 0.5467983561008608, 'reg_alpha': 0.1434715951720141, 'reg_lambda': 53.99484409787433, 'gamma': 3.0454635575417233, 'scale_pos_weight': 16.531162500179317}. Best is trial 0 with value: -1.0.
[I 2026-02-02 07:24:06,663] Trial 1 finished with value: -1.0 and parameters: {'n_estimators': 118, 'learning_rate': 0.09138013915892866, 'max_depth': 6, 'min_child_weight': 29, 'subsample': 0.5545474901621302, 'colsample_bytree': 0.5550213529560302, 'reg_alpha': 0.6624310605949987, 'reg_lambda': 11.207606211860567, 'gamma': 2.2165305913463675, 'scale_pos_weight': 9.91423577600417}. Best is trial 0 with value: -1.0.
[I 2026-02-02 07:24:08,288] Trial 2 finished with value: -1.0 and parameters: {'n_estimators': 651, 'learning_rate': 0.007593739613361234, 'max_depth': 4, 'min_

[I 2026-02-02 07:25:37,837] A new study created in memory with name: no-name-507824f4-790e-451b-8501-e3c41f072901



OPTUNA ADABOOST OPTIMIZATION (30 trials)
Target: Maximize recall while precision >= 51%
Using sample_weight with pos_weight=10.583


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-02-02 07:42:54,597] Trial 0 finished with value: 0.004410143329658214 and parameters: {'max_depth': 2, 'min_samples_leaf': 48, 'n_estimators': 453, 'learning_rate': 0.15751320499779725}. Best is trial 0 with value: 0.004410143329658214.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 16.2 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python
# minute_model_gpu_native_v10.py

"""
GPU-Native Model Training Pipeline (v10) - Precision-Focused Edition with Ensemble Agreement
---------------------------------------------------------------------------------------------
Key Changes:
1. PRECISION THRESHOLD: Find threshold for precision >= 0.54
2. PRECISION-RECALL CURVE: Visualize precision/recall tradeoff
3. CONFUSION MATRIX: Detailed classification analysis
4. PRECISION-OPTIMIZED TRAINING: Loss functions and metrics tuned for precision
5. TRAINING VISUALIZATION: Added Loss vs Val Loss curves
6. BALANCED LOSS: Auto-calculated pos_weight to prevent lazy models
7. FIXED LOSS CHART: Separates raw loss from regularization for apples-to-apples comparison
8. ENSEMBLE AGREEMENT: Only trade when multiple models agree (variance reduction)

Optimized for T4 GPU (16GB VRAM)
"""

import os
import gc
import time
import warnings
from typing import Tuple, Dict, List, Optional
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
#F.relu, F.softmax and other functions

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

# Sklearn metrics (just for evaluation - not training)
from sklearn.metrics import (
    precision_recall_curve, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, average_precision_score,
    roc_auc_score
)

# Optuna
import optuna
from optuna.samplers import TPESampler

# XGBoost
import xgboost as xgb

# CuPy
try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False
    cp = np

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')


# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
@dataclass
class Config:
    TICKER: str = "GLD"
    DATA_DIR: str = "./data"
    OUTPUT_DIR: str = "./data/model_gpu_native_v12"

    # Trading
    TARGET_HORIZON: int = 5
    THRESHOLD: float = 0.0010

    # PRECISION TARGET
    MIN_PRECISION: float = 0.51      # Minimum precision threshold

    # Costs
    AVG_PRICE_EST: float = 75.0
    FIXED_COMMISSION: float = 0.005
    SLIPPAGE_BPS: float = 0.0001

    # Training
    TEST_SIZE_PCT: float = 0.20
    OPTUNA_TRIALS: int = 50
    RANDOM_STATE: int = 42
    EARLY_STOPPING: int = 50

    # GPU
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Mini-Batch Configuration
    BATCH_SIZE: int = 2048
    SHUFFLE_TRAIN: bool = True
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True

    # Neural Network Regularization
    NN_EPOCHS: int = 150
    NN_LR: float = 0.001
    NN_PATIENCE: int = 20
    NN_L1_LAMBDA: float = 0.001
    NN_L2_LAMBDA: float = 0.01
    NN_DROPOUT_RATES: List[float] = field(default_factory=lambda: [0.5, 0.4, 0.3, 0.2])
    NN_LABEL_SMOOTHING: float = 0.1
    NN_GRADIENT_CLIP: float = 1.0

    # Ensemble Configuration
    N_ENSEMBLE_MODELS: int = 5
    ENSEMBLE_DIVERSITY: bool = True

    # Ensemble Agreement Configuration
    USE_AGREEMENT: bool = True
    DEFAULT_AGREEMENT_THRESHOLD: float = 0.6  # 60% of models must agree
    REQUIRE_XGB_AGREE: bool = True  # XGBoost must also agree

    def __post_init__(self):
        self.INPUT_FILE = f"{self.DATA_DIR}/{self.TICKER}.US_minute_engineered_v4.csv"
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        print(f"✓ Device: {self.DEVICE}")
        print(f"✓ Target Precision: >= {self.MIN_PRECISION:.0%}")
        print(f"✓ Ensemble Agreement: {self.USE_AGREEMENT} (threshold: {self.DEFAULT_AGREEMENT_THRESHOLD:.0%})")
        if self.DEVICE == "cuda":
            print(f"  GPU: {torch.cuda.get_device_name(0)}")


CONFIG = Config()


# --------------------------------------------------------------------
# GPU Utilities
# --------------------------------------------------------------------
def to_torch(x, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(CONFIG.DEVICE, dtype=dtype)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.values
    return torch.tensor(x, dtype=dtype, device=CONFIG.DEVICE)


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    if CUPY_AVAILABLE and isinstance(x, cp.ndarray):
        return cp.asnumpy(x)
    return np.asarray(x)


def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        print(f"  📊 GPU Memory: {allocated:.2f}GB allocated")


# --------------------------------------------------------------------
# Auto-Balance Calculator
# --------------------------------------------------------------------
def calculate_class_weights(y: np.ndarray) -> Dict:
    """
    Auto-calculate class weights based on class imbalance.
    Returns weights for both XGBoost and Neural Network.
    """
    n_neg = (y == 0).sum()
    n_pos = (y == 1).sum()

    # pos_weight = n_negative / n_positive (how much more to weight positives)
    pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

    # scale_pos_weight for XGBoost (same concept)
    scale_pos_weight = pos_weight

    print(f"\n  ⚖️ Auto-Balanced Class Weights:")
    print(f"     Negative samples: {n_neg:,} ({n_neg/(n_neg+n_pos):.1%})")
    print(f"     Positive samples: {n_pos:,} ({n_pos/(n_neg+n_pos):.1%})")
    print(f"     pos_weight (NN): {pos_weight:.3f}")
    print(f"     scale_pos_weight (XGB): {scale_pos_weight:.3f}")

    return {
        'pos_weight': pos_weight,
        'scale_pos_weight': scale_pos_weight,
        'n_neg': n_neg,
        'n_pos': n_pos
    }


# --------------------------------------------------------------------
# Precision-Recall Analysis
# --------------------------------------------------------------------
class PrecisionRecallAnalyzer:
    """
    Analyze precision-recall tradeoff and find optimal threshold.
    """

    def __init__(self, min_precision: float = 0.54):
        self.min_precision = min_precision
        self.precisions = None
        self.recalls = None
        self.thresholds = None
        self.optimal_threshold = None
        self.optimal_precision = None
        self.optimal_recall = None
        self.average_precision = None

    def analyze(self, y_true: np.ndarray, y_proba: np.ndarray) -> Dict:
        """
        Compute precision-recall curve and find threshold for target precision.
        """
        # Compute precision-recall curve
        self.precisions, self.recalls, self.thresholds = precision_recall_curve(y_true, y_proba)

        # Average precision (area under PR curve)
        self.average_precision = average_precision_score(y_true, y_proba)

        # Find threshold that achieves minimum precision with maximum recall
        valid_mask = self.precisions[:-1] >= self.min_precision

        if valid_mask.any():
            # Among thresholds meeting precision target, pick one with best recall
            valid_indices = np.where(valid_mask)[0]
            best_idx = valid_indices[np.argmax(self.recalls[:-1][valid_indices])]

            self.optimal_threshold = self.thresholds[best_idx]
            self.optimal_precision = self.precisions[best_idx]
            self.optimal_recall = self.recalls[best_idx]
        else:
            # If no threshold meets target, use highest precision available
            best_idx = np.argmax(self.precisions[:-1])
            self.optimal_threshold = self.thresholds[best_idx]
            self.optimal_precision = self.precisions[best_idx]
            self.optimal_recall = self.recalls[best_idx]
            print(f"  ⚠️ Warning: Cannot achieve {self.min_precision:.0%} precision")
            print(f"     Best achievable: {self.optimal_precision:.2%}")

        return {
            'optimal_threshold': self.optimal_threshold,
            'optimal_precision': self.optimal_precision,
            'optimal_recall': self.optimal_recall,
            'average_precision': self.average_precision,
            'f1_at_threshold': 2 * (self.optimal_precision * self.optimal_recall) /
                               (self.optimal_precision + self.optimal_recall + 1e-8)
        }

    def plot_precision_recall_curve(self, save_path: str = None):
        """
        Plot precision-recall curve with optimal threshold marked.
        """
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # === Plot 1: Precision-Recall Curve ===
        ax1 = axes[0]
        ax1.plot(self.recalls, self.precisions, 'b-', linewidth=2, label='PR Curve')
        ax1.fill_between(self.recalls, self.precisions, alpha=0.2)

        # Mark optimal point
        ax1.scatter([self.optimal_recall], [self.optimal_precision],
                   color='red', s=150, zorder=5, marker='*',
                   label=f'Optimal (P={self.optimal_precision:.2%}, R={self.optimal_recall:.2%})')

        # Mark precision threshold line
        ax1.axhline(y=self.min_precision, color='green', linestyle='--',
                   label=f'Target Precision = {self.min_precision:.0%}')

        ax1.set_xlabel('Recall', fontsize=12)
        ax1.set_ylabel('Precision', fontsize=12)
        ax1.set_title(f'Precision-Recall Curve (AP={self.average_precision:.3f})', fontsize=14)
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim([0, 1])
        ax1.set_ylim([0, 1])

        # === Plot 2: Precision & Recall vs Threshold ===
        ax2 = axes[1]
        ax2.plot(self.thresholds, self.precisions[:-1], 'b-', linewidth=2, label='Precision')
        ax2.plot(self.thresholds, self.recalls[:-1], 'r-', linewidth=2, label='Recall')

        # Mark optimal threshold
        ax2.axvline(x=self.optimal_threshold, color='green', linestyle='--',
                   label=f'Optimal Threshold = {self.optimal_threshold:.3f}')
        ax2.axhline(y=self.min_precision, color='purple', linestyle=':', alpha=0.7,
                   label=f'Min Precision = {self.min_precision:.0%}')

        ax2.set_xlabel('Threshold', fontsize=12)
        ax2.set_ylabel('Score', fontsize=12)
        ax2.set_title('Precision & Recall vs Threshold', fontsize=14)
        ax2.legend(loc='best')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim([0, 1])
        ax2.set_ylim([0, 1])

        # === Plot 3: F1 Score vs Threshold ===
        ax3 = axes[2]
        f1_scores = 2 * (self.precisions[:-1] * self.recalls[:-1]) / \
                    (self.precisions[:-1] + self.recalls[:-1] + 1e-8)

        ax3.plot(self.thresholds, f1_scores, 'g-', linewidth=2, label='F1 Score')
        ax3.axvline(x=self.optimal_threshold, color='red', linestyle='--',
                   label=f'Selected Threshold')

        # Find max F1 threshold for reference
        max_f1_idx = np.argmax(f1_scores)
        max_f1_threshold = self.thresholds[max_f1_idx]
        ax3.axvline(x=max_f1_threshold, color='blue', linestyle=':', alpha=0.7,
                   label=f'Max F1 Threshold = {max_f1_threshold:.3f}')

        ax3.set_xlabel('Threshold', fontsize=12)
        ax3.set_ylabel('F1 Score', fontsize=12)
        ax3.set_title('F1 Score vs Threshold', fontsize=14)
        ax3.legend(loc='best')
        ax3.grid(True, alpha=0.3)
        ax3.set_xlim([0, 1])

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Precision-Recall curve saved to {save_path}")

        plt.close()

        return fig


# --------------------------------------------------------------------
# Confusion Matrix Visualization
# --------------------------------------------------------------------
class ConfusionMatrixAnalyzer:
    """
    Detailed confusion matrix analysis and visualization.
    """

    def __init__(self):
        self.cm = None
        self.tn = None
        self.fp = None
        self.fn = None
        self.tp = None
        self.metrics = None

    def compute(self, y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
        """
        Compute confusion matrix and derived metrics.
        """
        self.cm = confusion_matrix(y_true, y_pred)
        self.tn, self.fp, self.fn, self.tp = self.cm.ravel()

        # Compute metrics
        precision = self.tp / (self.tp + self.fp) if (self.tp + self.fp) > 0 else 0
        recall = self.tp / (self.tp + self.fn) if (self.tp + self.fn) > 0 else 0
        specificity = self.tn / (self.tn + self.fp) if (self.tn + self.fp) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn)

        # False positive rate (important for trading - avoid bad trades)
        fpr = self.fp / (self.fp + self.tn) if (self.fp + self.tn) > 0 else 0

        # Positive predictive value (same as precision)
        ppv = precision

        # Negative predictive value
        npv = self.tn / (self.tn + self.fn) if (self.tn + self.fn) > 0 else 0

        self.metrics = {
            'precision': precision,
            'recall': recall,
            'specificity': specificity,
            'f1_score': f1,
            'accuracy': accuracy,
            'false_positive_rate': fpr,
            'ppv': ppv,
            'npv': npv,
            'true_positives': self.tp,
            'true_negatives': self.tn,
            'false_positives': self.fp,
            'false_negatives': self.fn,
            'total_predictions': len(y_pred),
            'total_positive_predictions': int(y_pred.sum()),
            'actual_positives': int(y_true.sum())
        }

        return self.metrics

    def plot(self, save_path: str = None, title: str = "Confusion Matrix"):
        """
        Plot detailed confusion matrix with metrics.
        """
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # === Plot 1: Confusion Matrix Heatmap ===
        ax1 = axes[0]

        # Normalized confusion matrix
        cm_normalized = self.cm.astype('float') / self.cm.sum(axis=1)[:, np.newaxis]

        # Plot
        sns.heatmap(
            self.cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Predict Negative\n(No Trade)', 'Predict Positive\n(Trade)'],
            yticklabels=['Actual Negative\n(Price Down/Flat)', 'Actual Positive\n(Price Up)'],
            annot_kws={'size': 14}
        )

        # Add percentages
        for i in range(2):
            for j in range(2):
                pct = cm_normalized[i, j] * 100
                ax1.text(j + 0.5, i + 0.7, f'({pct:.1f}%)',
                        ha='center', va='center', fontsize=10, color='gray')

        ax1.set_title(f'{title}\nThreshold for Precision ≥ {CONFIG.MIN_PRECISION:.0%}', fontsize=14)
        ax1.set_ylabel('Actual', fontsize=12)
        ax1.set_xlabel('Predicted', fontsize=12)

        # === Plot 2: Metrics Bar Chart ===
        ax2 = axes[1]

        metrics_to_plot = {
            'Precision': self.metrics['precision'],
            'Recall': self.metrics['recall'],
            'Specificity': self.metrics['specificity'],
            'F1 Score': self.metrics['f1_score'],
            'Accuracy': self.metrics['accuracy'],
            'NPV': self.metrics['npv']
        }

        colors = ['green' if v >= CONFIG.MIN_PRECISION else 'orange'
                  for v in metrics_to_plot.values()]
        colors[0] = 'green' if self.metrics['precision'] >= CONFIG.MIN_PRECISION else 'red'

        bars = ax2.barh(list(metrics_to_plot.keys()), list(metrics_to_plot.values()),
                       color=colors, alpha=0.7, edgecolor='black')

        # Add value labels
        for bar, (name, value) in zip(bars, metrics_to_plot.items()):
            ax2.text(value + 0.02, bar.get_y() + bar.get_height()/2,
                    f'{value:.2%}', va='center', fontsize=11, fontweight='bold')

        # Add precision threshold line
        ax2.axvline(x=CONFIG.MIN_PRECISION, color='red', linestyle='--', linewidth=2,
                   label=f'Min Precision Target ({CONFIG.MIN_PRECISION:.0%})')

        ax2.set_xlim([0, 1.1])
        ax2.set_xlabel('Score', fontsize=12)
        ax2.set_title('Classification Metrics', fontsize=14)
        ax2.legend(loc='lower right')
        ax2.grid(True, alpha=0.3, axis='x')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Confusion matrix saved to {save_path}")

        plt.close()

        return fig

    def print_report(self):
        """Print detailed classification report."""
        print(f"\n  {'='*50}")
        print(f"  CONFUSION MATRIX ANALYSIS")
        print(f"  {'='*50}")
        print(f"\n  Matrix:")
        print(f"                    Predicted")
        print(f"                 Neg      Pos")
        print(f"  Actual Neg   {self.tn:6d}   {self.fp:6d}")
        print(f"  Actual Pos   {self.fn:6d}   {self.tp:6d}")

        print(f"\n  Key Metrics:")
        print(f"  {'─'*40}")
        print(f"  Precision:          {self.metrics['precision']:.4f} ({self.metrics['precision']:.2%})")
        print(f"  Recall:             {self.metrics['recall']:.4f} ({self.metrics['recall']:.2%})")
        print(f"  F1 Score:           {self.metrics['f1_score']:.4f}")
        print(f"  Specificity:        {self.metrics['specificity']:.4f}")
        print(f"  False Positive Rate:{self.metrics['false_positive_rate']:.4f}")

        print(f"\n  Trading Impact:")
        print(f"  {'─'*40}")
        print(f"  Total Trades Taken: {self.metrics['total_positive_predictions']}")
        print(f"  Winning Trades (TP):{self.tp}")
        print(f"  Losing Trades (FP): {self.fp}")
        print(f"  Missed Opportunities (FN): {self.fn}")

        # Precision status
        if self.metrics['precision'] >= CONFIG.MIN_PRECISION:
            print(f"\n  ✅ PRECISION TARGET MET: {self.metrics['precision']:.2%} >= {CONFIG.MIN_PRECISION:.0%}")
        else:
            print(f"\n  ❌ PRECISION TARGET NOT MET: {self.metrics['precision']:.2%} < {CONFIG.MIN_PRECISION:.0%}")


# --------------------------------------------------------------------
# GPU-Native Preprocessing
# --------------------------------------------------------------------
class GPUPreprocessor:
    """GPU-accelerated rolling z-score normalization."""

    def __init__(self, rolling_window: int = 100):
        self.rolling_window = rolling_window

    def fit_transform(self, X: torch.Tensor) -> torch.Tensor:
        X = to_torch(X)
        n_samples, n_features = X.shape
        X_normalized = torch.zeros_like(X)

        for i in range(n_features):
            col = X[:, i]
            cumsum = torch.cumsum(col, dim=0)
            cumsum_sq = torch.cumsum(col ** 2, dim=0)

            rolling_sum = cumsum.clone()
            rolling_sum[self.rolling_window:] = cumsum[self.rolling_window:] - cumsum[:-self.rolling_window]
            counts = torch.clamp(torch.arange(1, n_samples + 1, device=CONFIG.DEVICE), max=self.rolling_window).float()
            rolling_mean = rolling_sum / counts

            rolling_sum_sq = cumsum_sq.clone()
            rolling_sum_sq[self.rolling_window:] = cumsum_sq[self.rolling_window:] - cumsum_sq[:-self.rolling_window]
            rolling_var = (rolling_sum_sq / counts) - (rolling_mean ** 2)
            rolling_std = torch.sqrt(torch.clamp(rolling_var, min=1e-8))

            X_normalized[:, i] = (col - rolling_mean) / rolling_std

        X_normalized = torch.clamp(X_normalized, -3, 3)
        X_normalized = torch.nan_to_num(X_normalized, nan=0.0, posinf=3.0, neginf=-3.0)

        return X_normalized

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        return self.fit_transform(X)


# --------------------------------------------------------------------
# Elastic Net Regularization
# --------------------------------------------------------------------
class ElasticNetRegularizer:
    def __init__(self, l1_lambda: float = 0.001, l2_lambda: float = 0.01):
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda

    def __call__(self, model: nn.Module) -> torch.Tensor:
        l1_penalty = torch.tensor(0.0, device=CONFIG.DEVICE)
        l2_penalty = torch.tensor(0.0, device=CONFIG.DEVICE)

        for name, param in model.named_parameters():
            if 'weight' in name:
                l1_penalty += torch.sum(torch.abs(param))
                l2_penalty += torch.sum(param ** 2)

        return self.l1_lambda * l1_penalty + self.l2_lambda * l2_penalty


# --------------------------------------------------------------------
# Neural Network with Dropout
# --------------------------------------------------------------------
class SpatialDropout1D(nn.Module):
    def __init__(self, p: float = 0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = torch.bernoulli(torch.ones(x.shape[1], device=x.device) * (1 - self.p))
        mask = mask.unsqueeze(0).expand_as(x)
        return x * mask / (1 - self.p)


class RegularizedMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int] = [256, 128, 64],
        dropout_rates: List[float] = [0.5, 0.4, 0.3],
        use_batch_norm: bool = True
    ):
        super().__init__()

        layers = []
        prev_dim = input_dim

        if len(dropout_rates) < len(hidden_dims):
            dropout_rates = dropout_rates + [dropout_rates[-1]] * (len(hidden_dims) - len(dropout_rates))

        for i, (hidden_dim, drop_rate) in enumerate(zip(hidden_dims, dropout_rates)):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(SpatialDropout1D(drop_rate))
            prev_dim = hidden_dim

        self.hidden_layers = nn.Sequential(*layers)
        self.output_layer = nn.Linear(prev_dim, 1)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.hidden_layers(x)
        out = self.output_layer(out)
        return out.squeeze(-1)


# --------------------------------------------------------------------
# Balanced Loss Function
# --------------------------------------------------------------------
class BalancedLoss(nn.Module):
    """
    Balanced BCE Loss with pos_weight to prevent lazy all-negative predictions.
    """

    def __init__(self, pos_weight: float = 1.0, fp_weight: float = 1.0, label_smoothing: float = 0.1):
        super().__init__()
        self.pos_weight = pos_weight
        self.fp_weight = fp_weight
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Apply label smoothing
        if self.label_smoothing > 0:
            smoothed_targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            smoothed_targets = targets

        pos_weight_tensor = torch.tensor([self.pos_weight], device=logits.device)

        bce_loss = F.binary_cross_entropy_with_logits(
            logits,
            smoothed_targets,
            pos_weight=pos_weight_tensor,
            reduction='none'
        )

        # Additional FP penalty (optional, for precision focus)
        if self.fp_weight > 1.0:
            probs = torch.sigmoid(logits)
            fp_mask = (probs > 0.5) & (targets < 0.5)
            weights = torch.ones_like(bce_loss)
            weights[fp_mask] = self.fp_weight
            bce_loss = bce_loss * weights

        return bce_loss.mean()


# --------------------------------------------------------------------
# Mini-Batch Trainer with Balanced Loss
# --------------------------------------------------------------------
class MiniBatchTrainer:
    def __init__(
        self,
        model: nn.Module,
        l1_lambda: float = 0.001,
        l2_lambda: float = 0.01,
        lr: float = 0.001,
        batch_size: int = 2048,
        epochs: int = 100,
        patience: int = 15,
        pos_weight: float = 1.0,
        fp_weight: float = 2.0,
        label_smoothing: float = 0.1,
        gradient_clip: float = 1.0,
        device: str = "cuda"
    ):
        self.model = model.to(device)
        self.device = device
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.gradient_clip = gradient_clip

        self.elastic_net = ElasticNetRegularizer(l1_lambda, l2_lambda)
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0)

        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=5
        )

        self.criterion = BalancedLoss(
            pos_weight=pos_weight,
            fp_weight=fp_weight,
            label_smoothing=label_smoothing
        )

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_total_loss': [],
            'val_precision': [],
            'val_recall': []
        }
        self.best_val_precision = 0
        self.best_model_state = None
        self.epochs_without_improvement = 0

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        X_train = to_torch(X_train)
        y_train = to_torch(y_train)

        if X_val is not None:
            X_val = to_torch(X_val)
            y_val = to_torch(y_val)

        train_dataset = TensorDataset(X_train, y_train)

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False,
            drop_last=True
        )

        for epoch in range(self.epochs):
            self.model.train()
            epoch_raw_loss = 0.0
            epoch_total_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(batch_X)

                raw_loss = self.criterion(outputs, batch_y)
                reg_loss = self.elastic_net(self.model)
                total_loss = raw_loss + reg_loss

                total_loss.backward()

                if self.gradient_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)

                self.optimizer.step()

                epoch_raw_loss += raw_loss.item()
                epoch_total_loss += total_loss.item()

            epoch_raw_loss /= len(train_loader)
            epoch_total_loss /= len(train_loader)

            self.history['train_loss'].append(epoch_raw_loss)
            self.history['train_total_loss'].append(epoch_total_loss)

            if X_val is not None:
                self.model.eval()
                with torch.no_grad():
                    val_outputs = self.model(X_val)
                    val_probs = torch.sigmoid(val_outputs)

                    val_raw_loss = self.criterion(val_outputs, y_val).item()

                    threshold = 0.5
                    val_preds = (val_probs > threshold).float()

                    tp = ((val_preds == 1) & (y_val == 1)).sum().item()
                    fp = ((val_preds == 1) & (y_val == 0)).sum().item()
                    fn = ((val_preds == 0) & (y_val == 1)).sum().item()

                    val_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    val_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

                self.history['val_loss'].append(val_raw_loss)
                self.history['val_precision'].append(val_precision)
                self.history['val_recall'].append(val_recall)

                self.scheduler.step(val_precision)

                if val_precision > self.best_val_precision:
                    self.best_val_precision = val_precision
                    self.best_model_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                    self.epochs_without_improvement = 0
                else:
                    self.epochs_without_improvement += 1

                if self.epochs_without_improvement >= self.patience:
                    print(f"    Early stopping at epoch {epoch + 1}")
                    break

                if (epoch + 1) % 10 == 0:
                    print(f"    Epoch {epoch + 1}/{self.epochs} | "
                          f"Raw Loss: {epoch_raw_loss:.4f} | Val Loss: {val_raw_loss:.4f} | "
                          f"Precision: {val_precision:.4f} | Recall: {val_recall:.4f}")

        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            self.model.to(self.device)

        return self


# --------------------------------------------------------------------
# Diverse NN Ensemble with Agreement Filter
# --------------------------------------------------------------------
class DiverseNNEnsemble:
    def __init__(self, input_dim: int, n_models: int = 5, use_diversity: bool = True,
                 base_config: Dict = None, class_weights: Dict = None):
        self.input_dim = input_dim
        self.n_models = n_models
        self.use_diversity = use_diversity
        self.models = []
        self.trainers = []

        self.class_weights = class_weights or {'pos_weight': 1.0}

        self.base_config = base_config or {
            'hidden_dims': [256, 128, 64],
            'dropout_rates': [0.5, 0.4, 0.3],
            'l1_lambda': 0.001,
            'l2_lambda': 0.01,
            'lr': 0.001,
            'epochs': 100,
            'patience': 15,
            'batch_size': 2048,
            'pos_weight': self.class_weights['pos_weight'],
            'fp_weight': 2.0
        }

        self.configs = self._generate_diverse_configs()

    def _generate_diverse_configs(self) -> List[Dict]:
        configs = []
        architectures = [[256, 128, 64], [512, 256, 128], [128, 64, 32], [256, 128, 64, 32], [384, 192, 96]]
        dropout_configs = [[0.5, 0.4, 0.3], [0.6, 0.5, 0.4], [0.4, 0.3, 0.2], [0.5, 0.5, 0.5], [0.3, 0.3, 0.3]]
        elastic_configs = [(0.001, 0.01), (0.005, 0.005), (0.01, 0.001), (0.0001, 0.05), (0.002, 0.008)]
        fp_weights = [2.0, 2.5, 3.0, 1.5, 2.0]

        for i in range(self.n_models):
            config = self.base_config.copy()
            if self.use_diversity:
                config['hidden_dims'] = architectures[i % len(architectures)]
                config['dropout_rates'] = dropout_configs[i % len(dropout_configs)]
                config['l1_lambda'], config['l2_lambda'] = elastic_configs[i % len(elastic_configs)]
                config['fp_weight'] = fp_weights[i % len(fp_weights)]
            config['pos_weight'] = self.class_weights['pos_weight']
            config['model_id'] = i
            configs.append(config)

        return configs

    def fit(self, X_train, y_train, X_val=None, y_val=None, use_bagging: bool = True):
        print(f"\n{'='*60}")
        print(f"TRAINING DIVERSE NN ENSEMBLE ({self.n_models} models)")
        print(f"Using pos_weight={self.class_weights['pos_weight']:.3f} (auto-balanced)")
        print(f"{'='*60}")

        X_train = to_torch(X_train)
        y_train = to_torch(y_train)
        if X_val is not None:
            X_val = to_torch(X_val)
            y_val = to_torch(y_val)

        for i, config in enumerate(self.configs):
            print(f"\n  Model {i + 1}/{self.n_models}:")
            print(f"    Architecture: {config['hidden_dims']}, pos_weight: {config['pos_weight']:.2f}, FP Weight: {config['fp_weight']}")

            model = RegularizedMLP(
                input_dim=self.input_dim,
                hidden_dims=config['hidden_dims'],
                dropout_rates=config['dropout_rates'],
                use_batch_norm=True
            )

            if use_bagging:
                n_samples = len(X_train)
                indices = torch.randint(0, n_samples, (int(n_samples * 0.8),), device=CONFIG.DEVICE)
                X_bag, y_bag = X_train[indices], y_train[indices]
            else:
                X_bag, y_bag = X_train, y_train

            trainer = MiniBatchTrainer(
                model=model,
                l1_lambda=config['l1_lambda'],
                l2_lambda=config['l2_lambda'],
                lr=config['lr'],
                batch_size=config['batch_size'],
                epochs=config['epochs'],
                patience=config['patience'],
                pos_weight=config['pos_weight'],
                fp_weight=config['fp_weight'],
                label_smoothing=CONFIG.NN_LABEL_SMOOTHING,
                gradient_clip=CONFIG.NN_GRADIENT_CLIP,
                device=CONFIG.DEVICE
            )

            trainer.fit(X_bag, y_bag, X_val, y_val)
            self.models.append(model)
            self.trainers.append(trainer)

        return self

    def predict_proba(self, X):
        """Standard ensemble average probability."""
        X = to_torch(X)
        all_probs = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
            all_probs.append(probs)

        ensemble_probs = torch.stack(all_probs).mean(dim=0)
        ensemble_probs = to_numpy(ensemble_probs)

        return np.column_stack([1 - ensemble_probs, ensemble_probs])

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)[:, 1]
        return (probs > threshold).astype(int)

    # ================================================================
    # ENSEMBLE AGREEMENT METHODS
    # ================================================================

    def predict_with_agreement(
        self,
        X,
        prob_threshold: float = 0.5,
        agreement_threshold: float = 0.6,
        return_details: bool = False
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Predict with ensemble agreement filter.

        Only predicts positive (trade) when >= agreement_threshold fraction
        of models predict positive.

        Args:
            X: Input features
            prob_threshold: Threshold for individual model predictions
            agreement_threshold: Fraction of models that must agree (e.g., 0.6 = 3/5 models)
            return_details: If True, return per-model predictions

        Returns:
            predictions: Final binary predictions (1 = trade, 0 = no trade)
            confidence: Agreement ratio (0 to 1) - can be used for position sizing
            (optional) model_preds: Individual model predictions if return_details=True
        """
        X = to_torch(X)
        all_probs = []
        all_preds = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
                preds = (probs > prob_threshold).float()
            all_probs.append(probs)
            all_preds.append(preds)

        # Stack predictions: (n_models, n_samples)
        stacked_preds = torch.stack(all_preds)
        stacked_probs = torch.stack(all_probs)

        # Agreement ratio: fraction of models predicting 1
        agreement_ratio = stacked_preds.mean(dim=0)

        # Final prediction: only trade if enough models agree
        final_pred = (agreement_ratio >= agreement_threshold).int()

        # Confidence is the agreement ratio itself
        confidence = agreement_ratio

        # Convert to numpy
        final_pred_np = to_numpy(final_pred)
        confidence_np = to_numpy(confidence)

        if return_details:
            model_preds_np = to_numpy(stacked_preds)
            model_probs_np = to_numpy(stacked_probs)
            return final_pred_np, confidence_np, model_preds_np, model_probs_np

        return final_pred_np, confidence_np

    def predict_proba_with_uncertainty(self, X) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Get ensemble probabilities with uncertainty estimates.

        Returns:
            mean_proba: Average probability across models
            std_proba: Standard deviation (uncertainty)
            agreement_at_05: Agreement ratio at threshold 0.5
        """
        X = to_torch(X)
        all_probs = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
            all_probs.append(probs)

        stacked = torch.stack(all_probs)  # (n_models, n_samples)

        mean_proba = stacked.mean(dim=0)
        std_proba = stacked.std(dim=0)

        # Agreement at default threshold
        preds = (stacked > 0.5).float()
        agreement = preds.mean(dim=0)

        return to_numpy(mean_proba), to_numpy(std_proba), to_numpy(agreement)

    def analyze_agreement(self, X, y_true, prob_threshold: float = 0.5) -> Dict:
        """
        Analyze how agreement levels correlate with prediction accuracy.

        Useful for finding the optimal agreement_threshold.
        """
        X = to_torch(X)
        y_true = to_numpy(y_true) if not isinstance(y_true, np.ndarray) else y_true

        all_preds = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                outputs = model(X)
                probs = torch.sigmoid(outputs)
                preds = (probs > prob_threshold).float()
            all_preds.append(preds)

        stacked = torch.stack(all_preds)
        agreement_ratio = to_numpy(stacked.mean(dim=0))

        # Analyze precision at different agreement levels
        results = {
            'agreement_levels': [],
            'precision': [],
            'recall': [],
            'f1_score': [],
            'n_trades': [],
            'n_samples': len(y_true)
        }

        for agreement_thresh in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
            pred = (agreement_ratio >= agreement_thresh).astype(int)

            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            fn = ((pred == 0) & (y_true == 1)).sum()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            n_trades = pred.sum()

            results['agreement_levels'].append(agreement_thresh)
            results['precision'].append(precision)
            results['recall'].append(recall)
            results['f1_score'].append(f1)
            results['n_trades'].append(n_trades)

        return results

    def find_optimal_agreement_threshold(
        self,
        X,
        y_true,
        min_precision: float = 0.54,
        prob_threshold: float = 0.5
    ) -> Tuple[float, Dict]:
        """
        Find the minimum agreement threshold that achieves target precision.

        Returns:
            optimal_threshold: The agreement threshold to use
            metrics: Dict with precision, recall, n_trades at that threshold
        """
        analysis = self.analyze_agreement(X, y_true, prob_threshold)

        optimal_thresh = 1.0
        optimal_metrics = {'precision': 0, 'recall': 0, 'n_trades': 0}

        for i, (thresh, prec, rec, f1, n_trades) in enumerate(zip(
            analysis['agreement_levels'],
            analysis['precision'],
            analysis['recall'],
            analysis['f1_score'],
            analysis['n_trades']
        )):
            if prec >= min_precision and n_trades > 0:
                # Find lowest threshold that meets precision target (maximizes recall)
                if thresh < optimal_thresh:
                    optimal_thresh = thresh
                    optimal_metrics = {
                        'precision': prec,
                        'recall': rec,
                        'f1_score': f1,
                        'n_trades': n_trades,
                        'trade_rate': n_trades / analysis['n_samples']
                    }

        return optimal_thresh, optimal_metrics

    def plot_agreement_analysis(self, X, y_true, save_path: str = None):
        """Plot precision/recall vs agreement threshold."""
        analysis = self.analyze_agreement(X, y_true)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Plot 1: Precision & Recall vs Agreement
        ax1 = axes[0]
        ax1.plot(analysis['agreement_levels'], analysis['precision'],
                'b-o', linewidth=2, markersize=8, label='Precision')
        ax1.plot(analysis['agreement_levels'], analysis['recall'],
                'r-o', linewidth=2, markersize=8, label='Recall')
        ax1.axhline(CONFIG.MIN_PRECISION, color='green', linestyle='--',
                   label=f'Target Precision ({CONFIG.MIN_PRECISION:.0%})')
        ax1.set_xlabel('Agreement Threshold', fontsize=12)
        ax1.set_ylabel('Score', fontsize=12)
        ax1.set_title('Precision & Recall vs Agreement Threshold', fontsize=14)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim([0.35, 1.05])
        ax1.set_ylim([0, 1])

        # Plot 2: Number of Trades vs Agreement
        ax2 = axes[1]
        ax2.bar(analysis['agreement_levels'], analysis['n_trades'],
               width=0.08, color='steelblue', edgecolor='black', alpha=0.7)
        ax2.set_xlabel('Agreement Threshold', fontsize=12)
        ax2.set_ylabel('Number of Trades', fontsize=12)
        ax2.set_title('Trade Count vs Agreement Threshold', fontsize=14)
        ax2.grid(True, alpha=0.3, axis='y')

        # Plot 3: Precision-Recall Tradeoff at different agreements
        ax3 = axes[2]
        colors = plt.cm.viridis(np.linspace(0, 1, len(analysis['agreement_levels'])))
        for i, (thresh, prec, rec) in enumerate(zip(
            analysis['agreement_levels'], analysis['precision'], analysis['recall']
        )):
            ax3.scatter(rec, prec, c=[colors[i]], s=150,
                       label=f'Agree={thresh:.0%}', edgecolors='black', zorder=5)

        ax3.axhline(CONFIG.MIN_PRECISION, color='green', linestyle='--', alpha=0.7)
        ax3.set_xlabel('Recall', fontsize=12)
        ax3.set_ylabel('Precision', fontsize=12)
        ax3.set_title('Precision-Recall at Different Agreement Levels', fontsize=14)
        ax3.legend(loc='best', fontsize=9)
        ax3.grid(True, alpha=0.3)
        ax3.set_xlim([0, 1])
        ax3.set_ylim([0, 1])

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  📊 Agreement analysis saved to {save_path}")

        plt.close()
        return fig

    def plot_history(self, save_dir: str):
        """Generates Training vs Val Loss curves."""
        if not self.trainers:
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        ax1 = axes[0]
        min_len = min([len(t.history['train_loss']) for t in self.trainers])

        train_losses = [t.history['train_loss'][:min_len] for t in self.trainers]
        val_losses = [t.history['val_loss'][:min_len] for t in self.trainers]

        for i, t in enumerate(self.trainers):
            ax1.plot(t.history['train_loss'], color='blue', alpha=0.15, linewidth=1)
            ax1.plot(t.history['val_loss'], color='red', alpha=0.15, linewidth=1)

        avg_train = np.mean(train_losses, axis=0)
        avg_val = np.mean(val_losses, axis=0)

        ax1.plot(avg_train, color='blue', linewidth=2.5, label='Avg Train Loss (Raw)')
        ax1.plot(avg_val, color='red', linewidth=2.5, label='Avg Val Loss (Raw)')

        ax1.set_title(f'Ensemble Loss - Raw Only ({len(self.trainers)} Models)')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Raw Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2 = axes[1]
        val_precisions = [t.history['val_precision'][:min_len] for t in self.trainers]
        val_recalls = [t.history['val_recall'][:min_len] for t in self.trainers]
        avg_precision = np.mean(val_precisions, axis=0)
        avg_recall = np.mean(val_recalls, axis=0)

        for i, t in enumerate(self.trainers):
            ax2.plot(t.history['val_precision'], color='green', alpha=0.15)
            ax2.plot(t.history['val_recall'], color='orange', alpha=0.15)

        ax2.plot(avg_precision, color='green', linewidth=2.5, label='Avg Val Precision')
        ax2.plot(avg_recall, color='orange', linewidth=2.5, label='Avg Val Recall')
        ax2.axhline(CONFIG.MIN_PRECISION, color='black', linestyle='--', label='Target Precision')

        ax2.set_title('Validation Precision & Recall Evolution')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Score')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = f"{save_dir}/ensemble_learning_curves.png"
        plt.savefig(save_path, dpi=150)
        plt.close()
        print(f"  📊 Learning curves saved to {save_path}")


# --------------------------------------------------------------------
# Full Ensemble with Agreement
# --------------------------------------------------------------------
class FullGPUEnsemble:
    def __init__(self, xgb_model, nn_ensemble, preprocessor, weights=[0.5, 0.5]):
        self.xgb_model = xgb_model
        self.nn_ensemble = nn_ensemble
        self.preprocessor = preprocessor
        self.weights = weights

        # Agreement settings (can be tuned)
        self.use_agreement = CONFIG.USE_AGREEMENT
        self.agreement_threshold = CONFIG.DEFAULT_AGREEMENT_THRESHOLD
        self.prob_threshold = 0.5

        # Optimal thresholds (set after calibration)
        self.optimal_config = None

    def predict_proba(self, X):
        """Standard ensemble average probability."""
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X
        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]

        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)
        nn_proba = self.nn_ensemble.predict_proba(X_norm)[:, 1]

        ensemble_proba = self.weights[0] * xgb_proba + self.weights[1] * nn_proba
        return np.column_stack([1 - ensemble_proba, ensemble_proba])

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)[:, 1]
        return (probs > threshold).astype(int)

    def predict_with_agreement(
        self,
        X,
        xgb_threshold: float = 0.5,
        nn_prob_threshold: float = 0.5,
        nn_agreement_threshold: float = 0.6,
        require_xgb_agree: bool = True
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Combined XGBoost + NN ensemble prediction with agreement filter.

        Args:
            X: Input features (raw, will be preprocessed for NN)
            xgb_threshold: Probability threshold for XGBoost
            nn_prob_threshold: Probability threshold for individual NN models
            nn_agreement_threshold: Fraction of NN models that must agree
            require_xgb_agree: If True, XGBoost must also predict positive

        Returns:
            predictions: Final binary predictions
            confidence: Combined confidence score
        """
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X

        # XGBoost prediction
        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]
        xgb_pred = (xgb_proba > xgb_threshold).astype(int)

        # NN ensemble with agreement
        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)
        nn_pred, nn_agreement = self.nn_ensemble.predict_with_agreement(
            X_norm,
            prob_threshold=nn_prob_threshold,
            agreement_threshold=nn_agreement_threshold
        )

        # Combine predictions
        if require_xgb_agree:
            # Both XGBoost and NN ensemble must agree
            final_pred = (xgb_pred == 1) & (nn_pred == 1)
        else:
            # Just use NN agreement
            final_pred = nn_pred

        final_pred = final_pred.astype(int)

        # Confidence: weighted combination of XGBoost proba and NN agreement
        confidence = self.weights[0] * xgb_proba + self.weights[1] * nn_agreement

        return final_pred, confidence

    def find_optimal_thresholds(
        self,
        X,
        y_true,
        min_precision: float = 0.54
    ) -> Dict:
        """
        Grid search to find optimal thresholds for the combined ensemble.
        """
        X_np = to_numpy(X) if isinstance(X, torch.Tensor) else X
        y_true = to_numpy(y_true) if not isinstance(y_true, np.ndarray) else y_true

        # Precompute XGBoost probabilities
        xgb_proba = self.xgb_model.predict_proba(X_np)[:, 1]

        # Precompute NN predictions at different agreement levels
        X_t = to_torch(X)
        X_norm = self.preprocessor.transform(X_t)

        best_result = {
            'precision': 0,
            'recall': 0,
            'f1_score': 0,
            'n_trades': 0,
            'xgb_threshold': 0.5,
            'nn_agreement': 0.6,
            'require_xgb': True
        }

        print(f"\n  🔍 Grid searching optimal thresholds...")

        # Grid search
        for xgb_thresh in [0.4, 0.45, 0.5, 0.55, 0.6]:
            for nn_agree in [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
                for require_xgb in [True, False]:

                    xgb_pred = (xgb_proba > xgb_thresh).astype(int)
                    nn_pred, _ = self.nn_ensemble.predict_with_agreement(
                        X_norm, prob_threshold=0.5, agreement_threshold=nn_agree
                    )

                    if require_xgb:
                        final_pred = (xgb_pred == 1) & (nn_pred == 1)
                    else:
                        final_pred = nn_pred

                    final_pred = final_pred.astype(int)

                    tp = ((final_pred == 1) & (y_true == 1)).sum()
                    fp = ((final_pred == 1) & (y_true == 0)).sum()
                    fn = ((final_pred == 0) & (y_true == 1)).sum()

                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
                    n_trades = final_pred.sum()

                    # Update if meets precision and has better recall
                    if precision >= min_precision and recall > best_result['recall']:
                        best_result = {
                            'precision': precision,
                            'recall': recall,
                            'f1_score': f1,
                            'n_trades': int(n_trades),
                            'xgb_threshold': xgb_thresh,
                            'nn_agreement': nn_agree,
                            'require_xgb': require_xgb
                        }

        self.optimal_config = best_result
        return best_result

    def predict_optimal(self, X) -> Tuple[np.ndarray, np.ndarray]:
        """
        Predict using the optimal thresholds found during calibration.
        Must call find_optimal_thresholds() first.
        """
        if self.optimal_config is None:
            raise ValueError("Must call find_optimal_thresholds() first!")

        return self.predict_with_agreement(
            X,
            xgb_threshold=self.optimal_config['xgb_threshold'],
            nn_prob_threshold=0.5,
            nn_agreement_threshold=self.optimal_config['nn_agreement'],
            require_xgb_agree=self.optimal_config['require_xgb']
        )


# --------------------------------------------------------------------
# Optuna XGBoost (Auto-Balanced)
# --------------------------------------------------------------------
class OptunaXGBOptimizer:
    def __init__(self, config: Config, class_weights: Dict = None):
        self.config = config
        self.class_weights = class_weights or {'scale_pos_weight': 1.0}
        self.best_model = None
        self.best_params = None
        self.study = None

    def objective(self, trial, X_train, y_train, X_val, y_val):
        base_spw = self.class_weights['scale_pos_weight']

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'max_bin': 256,
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 6),
            'min_child_weight': trial.suggest_int('min_child_weight', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 0.8),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 50.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 100.0, log=True),
            'gamma': trial.suggest_float('gamma', 0.1, 5.0),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', base_spw * 0.5, base_spw * 2.0),
            'random_state': self.config.RANDOM_STATE,
            'early_stopping_rounds': self.config.EARLY_STOPPING,
        }

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        y_pred_proba = model.predict_proba(X_val)[:, 1]

        precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_proba)

        valid_mask = precisions[:-1] >= self.config.MIN_PRECISION
        if valid_mask.any():
            valid_indices = np.where(valid_mask)[0]
            best_idx = valid_indices[np.argmax(recalls[:-1][valid_indices])]
            achieved_recall = recalls[best_idx]
            return achieved_recall
        else:
            return -1.0

    def optimize(self, X_train, y_train, X_val, y_val, n_trials=50):
        print(f"\n{'='*60}")
        print(f"OPTUNA XGBOOST OPTIMIZATION ({n_trials} trials)")
        print(f"Target: Maximize recall while precision >= {CONFIG.MIN_PRECISION:.0%}")
        print(f"Base scale_pos_weight: {self.class_weights['scale_pos_weight']:.3f} (auto-balanced)")
        print(f"{'='*60}")

        self.study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=self.config.RANDOM_STATE))
        self.study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials, show_progress_bar=True, gc_after_trial=True
        )

        self.best_params = self.study.best_params
        print(f"\n  ✓ Best Recall (at precision >= {CONFIG.MIN_PRECISION:.0%}): {self.study.best_value:.4f}")
        print(f"  ✓ Tuned scale_pos_weight: {self.best_params.get('scale_pos_weight', 'N/A'):.3f}")

        final_params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda',
            'max_bin': 256,
            'random_state': self.config.RANDOM_STATE,
            'early_stopping_rounds': self.config.EARLY_STOPPING,
            **self.best_params
        }

        self.best_model = xgb.XGBClassifier(**final_params)
        self.best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        return self.best_model, self.best_params


# --------------------------------------------------------------------
# Data Loading
# --------------------------------------------------------------------
def load_data(config: Config):
    print(f"\n{'='*60}")
    print("LOADING DATA")
    print(f"{'='*60}")

    if not os.path.exists(config.INPUT_FILE):
        raise FileNotFoundError(f"File not found: {config.INPUT_FILE}")

    df = pd.read_csv(config.INPUT_FILE)

    for c in df.columns:
        if c.lower() in ['date', 'datetime', 'timestamp', 'time']:
            df[c] = pd.to_datetime(df[c])
            df.set_index(c, inplace=True)
            df.sort_index(inplace=True)
            break

    prices = df[['open', 'close']].copy()

    future_returns = df['close'].pct_change(periods=config.TARGET_HORIZON).shift(-config.TARGET_HORIZON)
    y = (future_returns > config.THRESHOLD).astype(int)

    valid = ~(df.isna().any(axis=1) | y.isna())
    X = df.loc[valid].copy()
    y = y.loc[valid].copy()
    prices = prices.loc[valid].copy()

    drop_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'log_returns', 'gap_flag', 'target']
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')
    X = X.astype(np.float32)

    print(f"  Samples: {len(X):,}, Features: {X.shape[1]}")
    print(f"  Class Balance: {y.mean():.2%} positive")

    return X, y, prices


def time_series_split(X, y, prices, test_size=0.2, val_size=0.1):
    n = len(X)
    test_idx = int(n * (1 - test_size))
    val_idx = int(test_idx * (1 - val_size))

    return (
        X.iloc[:val_idx], X.iloc[val_idx:test_idx], X.iloc[test_idx:],
        y.iloc[:val_idx], y.iloc[val_idx:test_idx], y.iloc[test_idx:],
        prices.iloc[test_idx:]
    )


# --------------------------------------------------------------------
# Evaluation with Agreement
# --------------------------------------------------------------------
def evaluate_with_agreement(model, X_test, y_test, prices_test, config: Config):
    """
    Comprehensive evaluation using ensemble agreement for predictions.
    """
    print(f"\n{'='*60}")
    print("ENSEMBLE AGREEMENT EVALUATION")
    print(f"{'='*60}")

    y_true = y_test.values if hasattr(y_test, 'values') else y_test

    # Analyze agreement on NN ensemble
    print("\n  📊 Analyzing NN Ensemble Agreement...")
    X_t = to_torch(X_test)
    X_norm = model.preprocessor.transform(X_t)

    # Plot agreement analysis
    model.nn_ensemble.plot_agreement_analysis(
        X_norm, y_true,
        save_path=f"{config.OUTPUT_DIR}/agreement_analysis.png"
    )

    # Find optimal agreement threshold for NN only
    opt_agree, opt_metrics = model.nn_ensemble.find_optimal_agreement_threshold(
        X_norm, y_true, min_precision=config.MIN_PRECISION
    )

    print(f"\n  🎯 Optimal NN-Only Agreement Threshold: {opt_agree:.0%}")
    print(f"     Precision: {opt_metrics.get('precision', 0):.2%}")
    print(f"     Recall: {opt_metrics.get('recall', 0):.2%}")
    print(f"     Trades: {opt_metrics.get('n_trades', 0)}")

    # Find optimal combined thresholds (XGB + NN)
    best_config = model.find_optimal_thresholds(X_test, y_true, min_precision=config.MIN_PRECISION)

    print(f"\n  ✓ Best Combined Configuration:")
    print(f"     XGBoost Threshold: {best_config['xgb_threshold']}")
    print(f"     NN Agreement: {best_config['nn_agreement']:.0%}")
    print(f"     Require XGBoost Agree: {best_config['require_xgb']}")
    print(f"     Precision: {best_config['precision']:.2%}")
    print(f"     Recall: {best_config['recall']:.2%}")
    print(f"     F1 Score: {best_config['f1_score']:.4f}")
    print(f"     Trades: {best_config['n_trades']}")

    # Get final predictions using optimal config
    y_pred, confidence = model.predict_with_agreement(
        X_test,
        xgb_threshold=best_config['xgb_threshold'],
        nn_prob_threshold=0.5,
        nn_agreement_threshold=best_config['nn_agreement'],
        require_xgb_agree=best_config['require_xgb']
    )

    # Confusion Matrix Analysis
    cm_analyzer = ConfusionMatrixAnalyzer()
    cm_metrics = cm_analyzer.compute(y_true, y_pred)
    cm_analyzer.print_report()
    cm_analyzer.plot(
        save_path=f"{config.OUTPUT_DIR}/confusion_matrix_agreement.png",
        title=f"Confusion Matrix (Agreement={best_config['nn_agreement']:.0%})"
    )

    # Also generate standard PR curve for reference
    probs = model.predict_proba(X_test)[:, 1]
    pr_analyzer = PrecisionRecallAnalyzer(min_precision=config.MIN_PRECISION)
    pr_analyzer.analyze(y_true, probs)
    pr_analyzer.plot_precision_recall_curve(save_path=f"{config.OUTPUT_DIR}/precision_recall_curve.png")

    # Trading Backtest
    print(f"\n  {'='*50}")
    print(f"  TRADING BACKTEST (With Agreement Filter)")
    print(f"  {'='*50}")

    signals = y_pred
    entry_price = prices_test['open'].shift(-1).values
    exit_price = prices_test['close'].shift(-config.TARGET_HORIZON).values
    raw_returns = (exit_price - entry_price) / entry_price
    raw_returns = np.nan_to_num(raw_returns, 0)

    cost = config.FIXED_COMMISSION / config.AVG_PRICE_EST + config.SLIPPAGE_BPS
    net_returns = raw_returns * signals - signals * cost

    valid_len = len(signals)
    net_returns = net_returns[:valid_len]
    market_returns = prices_test['close'].pct_change().fillna(0).values[:valid_len]

    cum_market = np.cumprod(1 + market_returns)
    cum_net = np.cumprod(1 + net_returns)

    total_trades = signals.sum()
    win_rate = (net_returns[signals == 1] > 0).mean() if total_trades > 0 else 0
    final_return = cum_net[-1] - 1
    max_dd = ((cum_net - np.maximum.accumulate(cum_net)) / np.maximum.accumulate(cum_net)).min()
    sharpe = np.mean(net_returns) / np.std(net_returns) * np.sqrt(252 * 390) if np.std(net_returns) > 0 else 0

    print(f"\n  Trading Results:")
    print(f"     Total Trades: {total_trades}")
    print(f"     Trade Win Rate: {win_rate:.2%}")
    print(f"     Net Return: {final_return:.2%}")
    print(f"     Max Drawdown: {max_dd:.2%}")
    print(f"     Sharpe Ratio: {sharpe:.2f}")

    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    ax1.plot(cum_market, label='Buy & Hold', alpha=0.5, linewidth=1.5)
    ax1.plot(cum_net, label='Strategy (Agreement Filter)', linewidth=2, color='green')
    ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    ax1.set_title(f'Cumulative Returns (Precision={cm_metrics["precision"]:.1%})')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.hist(confidence, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    ax2.axvline(x=np.mean(confidence), color='red', linestyle='--',
               label=f'Mean: {np.mean(confidence):.3f}')
    ax2.set_title('Confidence (Agreement) Distribution')
    ax2.set_xlabel('Confidence Score')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    if total_trades > 0:
        trade_returns = net_returns[signals == 1] * 100
        ax3.hist(trade_returns, bins=30, alpha=0.7, color='orange', edgecolor='black')
        ax3.axvline(0, color='black', linewidth=2)
        ax3.axvline(np.mean(trade_returns), color='blue', linestyle='--',
                   label=f'Mean: {np.mean(trade_returns):.3f}%')
        ax3.legend()
    ax3.set_title('Per-Trade Returns Distribution (%)')
    ax3.set_xlabel('Return (%)')
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    drawdown = (cum_net - np.maximum.accumulate(cum_net)) / np.maximum.accumulate(cum_net) * 100
    ax4.fill_between(range(len(drawdown)), drawdown, 0, alpha=0.7, color='red')
    ax4.set_title(f'Drawdown (Max: {max_dd:.2%})')
    ax4.set_xlabel('Time')
    ax4.set_ylabel('Drawdown (%)')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_DIR}/trading_results_agreement.png", dpi=150, bbox_inches='tight')
    plt.close()

    results = {
        'optimal_nn_agreement': opt_agree,
        'combined_nn_agreement': best_config['nn_agreement'],
        'xgb_threshold': best_config['xgb_threshold'],
        'require_xgb_agree': best_config['require_xgb'],
        'precision': cm_metrics['precision'],
        'recall': cm_metrics['recall'],
        'f1_score': cm_metrics['f1_score'],
        'specificity': cm_metrics['specificity'],
        'total_trades': int(total_trades),
        'win_rate': win_rate,
        'final_return': final_return,
        'max_drawdown': max_dd,
        'sharpe': sharpe,
        'true_positives': cm_metrics['true_positives'],
        'false_positives': cm_metrics['false_positives'],
        'false_negatives': cm_metrics['false_negatives'],
        'true_negatives': cm_metrics['true_negatives']
    }

    print(f"\n  {'='*50}")
    if cm_metrics['precision'] >= config.MIN_PRECISION:
        print(f"  ✅ PRECISION TARGET MET: {cm_metrics['precision']:.2%} >= {config.MIN_PRECISION:.0%}")
    else:
        print(f"  ❌ PRECISION TARGET NOT MET: {cm_metrics['precision']:.2%} < {config.MIN_PRECISION:.0%}")
    print(f"  {'='*50}")

    return results


# --------------------------------------------------------------------
# Main Pipeline
# --------------------------------------------------------------------
def main():
    print("\n" + "=" * 60)
    print("GPU-NATIVE MODEL v10 - PRECISION-FOCUSED + ENSEMBLE AGREEMENT")
    print(f"Target Precision: >= {CONFIG.MIN_PRECISION:.0%}")
    print(f"Agreement Filter: {CONFIG.USE_AGREEMENT}")
    print("=" * 60)

    start_time = time.perf_counter()

    # 1. Load data
    X, y, prices = load_data(CONFIG)

    # 2. Split
    X_train, X_val, X_test, y_train, y_val, y_test, prices_test = time_series_split(
        X, y, prices, test_size=CONFIG.TEST_SIZE_PCT
    )
    print(f"\n  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

    # 3. AUTO-BALANCE: Calculate class weights from training data
    class_weights = calculate_class_weights(y_train.values)

    # 4. Preprocessing
    preprocessor = GPUPreprocessor(rolling_window=100)

    X_train_t = to_torch(X_train.values)
    X_val_t = to_torch(X_val.values)
    X_test_t = to_torch(X_test.values)

    X_train_norm = preprocessor.fit_transform(X_train_t)
    X_val_norm = preprocessor.transform(X_val_t)
    X_test_norm = preprocessor.transform(X_test_t)

    X_train_np = to_numpy(X_train_norm)
    X_val_np = to_numpy(X_val_norm)
    X_test_np = to_numpy(X_test_norm)

    y_train_np = y_train.values
    y_val_np = y_val.values

    # 5. XGBoost Optimization (with auto-balanced scale_pos_weight)
    optimizer = OptunaXGBOptimizer(CONFIG, class_weights=class_weights)
    best_xgb, best_params = optimizer.optimize(X_train_np, y_train_np, X_val_np, y_val_np, n_trials=CONFIG.OPTUNA_TRIALS)

    # 6. NN Ensemble (with auto-balanced pos_weight)
    nn_ensemble = DiverseNNEnsemble(
        input_dim=X_train.shape[1],
        n_models=CONFIG.N_ENSEMBLE_MODELS,
        use_diversity=CONFIG.ENSEMBLE_DIVERSITY,
        class_weights=class_weights,
        base_config={
            'hidden_dims': [256, 128, 64],
            'dropout_rates': CONFIG.NN_DROPOUT_RATES[:3],
            'l1_lambda': CONFIG.NN_L1_LAMBDA,
            'l2_lambda': CONFIG.NN_L2_LAMBDA,
            'lr': CONFIG.NN_LR,
            'epochs': CONFIG.NN_EPOCHS,
            'patience': CONFIG.NN_PATIENCE,
            'batch_size': CONFIG.BATCH_SIZE,
            'pos_weight': class_weights['pos_weight'],
            'fp_weight': 2.5
        }
    )

    nn_ensemble.fit(X_train_norm, to_torch(y_train_np), X_val_norm, to_torch(y_val_np), use_bagging=True)

    # Plot Training History
    nn_ensemble.plot_history(CONFIG.OUTPUT_DIR)

    # 7. Full Ensemble
    full_ensemble = FullGPUEnsemble(best_xgb, nn_ensemble, preprocessor, weights=[0.5, 0.5])

    # 8. Evaluation with Agreement Filter
    results = evaluate_with_agreement(full_ensemble, X_test_np, y_test, prices_test, CONFIG)

    # 9. Save Summary
    elapsed = time.perf_counter() - start_time

    with open(f"{CONFIG.OUTPUT_DIR}/summary.txt", 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("PRECISION-FOCUSED MODEL SUMMARY (ENSEMBLE AGREEMENT)\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Target Precision: >= {CONFIG.MIN_PRECISION:.0%}\n")
        f.write(f"Training Time: {elapsed:.1f}s\n\n")
        f.write(f"Class Weights (Auto-Calculated):\n")
        f.write(f"  pos_weight (NN): {class_weights['pos_weight']:.3f}\n")
        f.write(f"  scale_pos_weight (XGB): {class_weights['scale_pos_weight']:.3f}\n\n")
        f.write(f"Optimal Configuration:\n")
        f.write(f"  NN Agreement Threshold: {results['combined_nn_agreement']:.0%}\n")
        f.write(f"  XGBoost Threshold: {results['xgb_threshold']}\n")
        f.write(f"  Require XGBoost Agree: {results['require_xgb_agree']}\n\n")
        f.write("Results:\n")
        for k, v in results.items():
            if isinstance(v, float):
                f.write(f"  {k}: {v:.4f}\n")
            else:
                f.write(f"  {k}: {v}\n")

    print(f"\n{'='*60}")
    print("COMPLETE")
    print(f"{'='*60}")
    print(f"  ⏱ Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    print(f"  📁 Output: {CONFIG.OUTPUT_DIR}")
    print_gpu_memory()


if __name__ == "__main__":
    main()


✓ Device: cuda
✓ Target Precision: >= 51%
✓ Ensemble Agreement: True (threshold: 60%)
  GPU: Tesla T4

GPU-NATIVE MODEL v10 - PRECISION-FOCUSED + ENSEMBLE AGREEMENT
Target Precision: >= 51%
Agreement Filter: True

LOADING DATA


[I 2026-02-02 07:18:07,845] A new study created in memory with name: no-name-335216a1-8f5f-4a8f-949f-d720cd01e09d


  Samples: 179,291, Features: 54
  Class Balance: 9.58% positive

  Train: 129,088 | Val: 14,344 | Test: 35,859

  ⚖️ Auto-Balanced Class Weights:
     Negative samples: 117,943 (91.4%)
     Positive samples: 11,145 (8.6%)
     pos_weight (NN): 10.583
     scale_pos_weight (XGB): 10.583

OPTUNA XGBOOST OPTIMIZATION (50 trials)
Target: Maximize recall while precision >= 51%
Base scale_pos_weight: 10.583 (auto-balanced)


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-02 07:18:09,392] Trial 0 finished with value: -1.0 and parameters: {'n_estimators': 437, 'learning_rate': 0.0862735828664018, 'max_depth': 5, 'min_child_weight': 64, 'subsample': 0.5468055921327309, 'colsample_bytree': 0.5467983561008608, 'reg_alpha': 0.1434715951720141, 'reg_lambda': 53.99484409787433, 'gamma': 3.0454635575417233, 'scale_pos_weight': 16.531162500179317}. Best is trial 0 with value: -1.0.
[I 2026-02-02 07:18:11,071] Trial 1 finished with value: -1.0 and parameters: {'n_estimators': 118, 'learning_rate': 0.09138013915892866, 'max_depth': 6, 'min_child_weight': 29, 'subsample': 0.5545474901621302, 'colsample_bytree': 0.5550213529560302, 'reg_alpha': 0.6624310605949987, 'reg_lambda': 11.207606211860567, 'gamma': 2.2165305913463675, 'scale_pos_weight': 9.91423577600417}. Best is trial 0 with value: -1.0.
[I 2026-02-02 07:18:12,650] Trial 2 finished with value: -1.0 and parameters: {'n_estimators': 651, 'learning_rate': 0.007593739613361234, 'max_depth': 4, 'min_

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 18.1 MB/s eta 0:00:00
